# 07 — G3 Haugland (2131v/12530e): k5 (esperado 0) + k4 (PREGUNTA ABIERTA)

Control local ya hecho: SIREN k4 en 874 (5-cromático probado) se planta en 12+.
k5 en G3 se sabe SAT (0.39 s kissat) → SIREN debe dar 0 (chequeo de escala).
k4 en G3 es DESCONOCIDO (solo el CaDiCaL del autor decidió el G1): 0 = G3 4-coloreable
(noticia), piso alto = consistente con 5-cromático. TPU ~10 min.


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "--upgrade", "jax[tpu]",
                           "-f", "https://storage.googleapis.com/jax-releases/libtpu_releases.html"])
subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "python-sat"])
import jax
print('jax', jax.__version__, jax.devices())
assert any('TPU' in str(d) for d in jax.devices()), 'SIN TPU: Entorno->Cambiar tipo (TPU v5e-1)'


## Archivos


In [ ]:
NNJAX = "#!/usr/bin/env python3\n\"\"\"nn_jax.py \u2014 coloreo probabil\u00edstico Hadwiger-Nelson en JAX (CPU ac\u00e1, TPU en Colab).\n\nModelo (igual idea que lab/colG_pack.py, pero vectorizado + pmap/vmap):\n  logits (R, N, k) -> softmax -> p[r,i,c]. Loss[r] = media sobre aristas de\n  sum_c p[r,i,c]*p[r,j,c]  -  entropy_coef * entrop\u00eda_media (para no colapsar).\n  Adam a mano (sin optax) + jit + vmap sobre restarts. En TPU vmap corre en los\n  8 cores del host.\n\nValor honesto: BUSCA coloreos r\u00e1pido (heur\u00edstica, NO prueba). Si llega a\n0 violaciones con argmax, el grafo es k-coloreable (testigo de descarte para\ncandidatos grandes donde kissat tarda horas). Si queda en piso >0, NO prueba\nnada: el candidato va a kissat.\n\nFormatos .edge: 'e a b' o 'a b', 1-based por defecto (--zero-based para 0-based).\n\nEjemplos:\n  python nn_jax.py --demo\n  python nn_jax.py --edges /tmp/opencode/874.edge --k 5 --restarts 8 --steps 3000\n\"\"\"\n\nimport argparse\nimport json\nimport sys\nimport time\n\nimport jax\nimport jax.numpy as jnp\n\n\ndef load_edges(path, zero_based=False):\n    edges = []\n    seen = set()\n    with open(path) as f:\n        for line in f:\n            p = line.split()\n            if not p or p[0].startswith((\"c\", \"p\", \"#\")):\n                continue\n            if p[0] == \"e\" and len(p) >= 3:\n                a, b = int(p[1]), int(p[2])\n            elif len(p) >= 2:\n                try:\n                    a, b = int(p[0]), int(p[1])\n                except ValueError:\n                    continue\n            else:\n                continue\n            if not zero_based:\n                a, b = a - 1, b - 1\n            if a == b:\n                continue\n            if a > b:\n                a, b = b, a\n            if (a, b) not in seen:\n                seen.add((a, b))\n                edges.append((a, b))\n    return edges\n\n\ndef train(key, ei, ej, n, k, steps, lr, entropy_coef, temp_init):\n    rkey, skey = jax.random.split(key)\n    logits = jax.random.normal(rkey, (n, k)) * temp_init\n\n    m = jnp.zeros_like(logits)\n    v = jnp.zeros_like(logits)\n    b1, b2, eps = 0.9, 0.999, 1e-8\n\n    def loss_fn(lg, beta):\n        p = jax.nn.softmax(lg, axis=-1)\n        same = jnp.sum(p[ei] * p[ej], axis=-1).mean()\n        ent = -(p * jnp.log(p + 1e-12)).sum(-1).mean()\n        return same - beta * ent\n\n    # schedule coseno = lab/colG_pack.py::entropy_beta (el que plant\u00f3 874 en 0 en torch)\n    t_all = jnp.arange(steps)\n    betas = entropy_coef * 0.5 * (1.0 + jnp.cos(jnp.pi * t_all / steps))\n\n    @jax.jit\n    def step(carry, tb):\n        t, beta = tb\n        lg, m, v = carry\n        loss, g = jax.value_and_grad(loss_fn)(lg, beta)\n        m = b1 * m + (1 - b1) * g\n        v = b2 * v + (1 - b2) * g * g\n        mh = m / (1 - b1 ** (t + 1))\n        vh = v / (1 - b2 ** (t + 1))\n        lg = lg - lr * mh / (jnp.sqrt(vh) + eps)\n        return (lg, m, v), loss\n\n    (lg, _, _), losses = jax.lax.scan(step, (logits, m, v), (t_all, betas))\n    p = jax.nn.softmax(lg, axis=-1)\n    hard = jnp.argmax(p, axis=-1)\n    viol = jnp.sum(hard[ei] == hard[ej])\n    return losses[-1], viol, hard\n\n\ndef tabusearch(col, ei, ej, k, max_iters, seed=0, tenure_base=10):\n    \"\"\"TabuCol cl\u00e1sico (Hertz & de Werra 1987, coraz\u00f3n del SOTA 2025/TabuEdges):\n    mueve un v\u00e9rtice en conflicto al color menos conflictivo no-tab\u00fa (o por\n    aspiraci\u00f3n si mejora el r\u00e9cord). Supera al greedy: acepta empeorar para\n    salir de pozos. Numpy/CPU, ms por miles de iters.\"\"\"\n    import numpy as np\n\n    rng = np.random.default_rng(seed)\n    col = list(col)\n    ei_l = list(ei)\n    ej_l = list(ej)\n    n = max(len(col), max(ei_l) + 1, max(ej_l) + 1)\n    adj = [[] for _ in range(n)]\n    for a, b in zip(ei_l, ej_l):\n        adj[a].append(b)\n        adj[b].append(a)\n    cnt = [[0] * k for _ in range(n)]\n    for v in range(n):\n        for u in adj[v]:\n            cnt[v][col[u]] += 1\n    viol = sum(1 for a, b in zip(ei_l, ej_l) if col[a] == col[b])\n    tabu = [[-1] * k for _ in range(n)]\n    best_col, best_v = list(col), viol\n    it = 0\n    while it < max_iters and best_v > 0:\n        # v\u00e9rtices en conflicto\n        conf = [v for v in range(n) if cnt[v][col[v]] > 0]\n        if not conf:\n            best_col, best_v = list(col), 0\n            break\n        v = conf[int(rng.integers(0, len(conf)))]\n        cur = col[v]\n        # mejor movimiento (delta m\u00ednimo), respetando tab\u00fa salvo aspiraci\u00f3n\n        best_d, best_c = None, cur\n        for c in range(k):\n            if c == cur:\n                continue\n            d = cnt[v][c] - cnt[v][cur]\n            if tabu[v][c] > it and viol + d >= best_v:\n                continue\n            if best_d is None or d < best_d:\n                best_d, best_c = d, c\n        if best_d is None:\n            best_c = min((c for c in range(k) if c != cur), key=lambda c: cnt[v][c])\n            best_d = cnt[v][best_c] - cnt[v][cur]\n        col[v] = best_c\n        viol += best_d\n        for u in adj[v]:\n            cnt[u][cur] -= 1\n            cnt[u][best_c] += 1\n        tabu[v][cur] = it + tenure_base + int(rng.integers(0, 10)) + viol\n        if viol < best_v:\n            best_v, best_col = viol, list(col)\n        it += 1\n    return best_v, best_col, it\n\n\ndef slim_rounds(col, ei, ej, k, rounds, radius, cap, seed=0):\n    \"\"\"GC-SLIM 2023 (Schidler & Szeider) simplificado: por ronda, para v\u00e9rtices\n    en conflicto, toma bola radio `radius` (tope `cap` nodos), fija el borde y\n    re-colorea la bola \u00d3PTIMO con SAT (Glucose) minimizando conflictos v\u00eda\n    CardEnc por decisi\u00f3n. Exacto en chico, h\u00edbrido en grande.\"\"\"\n    from pysat.solvers import Glucose3\n    from pysat.card import CardEnc, EncType\n    import numpy as np\n\n    rng = np.random.default_rng(seed)\n    col = list(col)\n    ei_l = [int(x) for x in list(ei)]\n    ej_l = [int(x) for x in list(ej)]\n    n = max(len(col), max(ei_l) + 1, max(ej_l) + 1)\n    adj = [[] for _ in range(n)]\n    for a, b in zip(ei_l, ej_l):\n        adj[a].append(b)\n        adj[b].append(a)\n\n    def viol_of(c):\n        return sum(1 for a, b in zip(ei_l, ej_l) if c[a] == c[b])\n\n    for r in range(rounds):\n        cur = viol_of(col)\n        if cur == 0:\n            return 0, col, r\n        conf = [v for v in range(n) if any(col[v] == col[u] for u in adj[v])]\n        order = rng.permutation(conf).tolist()[:32]\n        improved = False\n        for v in order:\n            seen, frontier = {v}, [v]\n            for _ in range(radius):\n                nxt = []\n                for x in frontier:\n                    for u in adj[x]:\n                        if u not in seen:\n                            seen.add(u)\n                            nxt.append(u)\n                frontier = nxt\n            ball = sorted(seen)\n            if len(ball) > cap:\n                ball = sorted(rng.choice(ball, size=cap, replace=False).tolist())\n            bset = set(ball)\n            idx = {x: i for i, x in enumerate(ball)}\n            m = len(ball)\n\n            def V(x, c):\n                return idx[x] * k + c + 1\n\n            nvars = m * k\n            clauses = []\n            for x in ball:\n                clauses.append([V(x, c) for c in range(k)])\n                for a in range(k):\n                    for b in range(a + 1, k):\n                        clauses.append([-V(x, a), -V(x, b)])\n            for x in ball:  # borde fijo: prohibir colores de vecinos externos\n                for u in adj[x]:\n                    if u not in bset:\n                        clauses.append([-V(x, col[u])])\n            bedges = [(a, b) for a, b in zip(ei_l, ej_l) if a in bset and b in bset]\n            cur_in = sum(\n                1\n                for a, b in zip(ei_l, ej_l)\n                if (a in bset or b in bset) and col[a] == col[b]\n            )\n            if cur_in == 0:\n                continue\n            ind = []\n            for a, b in bedges:\n                e = nvars + len(ind) + 1\n                ind.append(e)\n                for c in range(k):\n                    clauses.append([-V(a, c), -V(b, c), e])\n            for target in range(cur_in - 1, -1, -1):\n                card = CardEnc.equals(\n                    lits=ind, bound=target, encoding=EncType.seqcounter\n                )\n                with Glucose3() as s:\n                    for cl in clauses:\n                        s.add_clause(cl)\n                    for cl in card.clauses:\n                        s.add_clause(cl)\n                    if not s.solve():\n                        continue\n                    model = set(x for x in s.get_model() or [] if x > 0)\n                    for x in ball:\n                        for c in range(k):\n                            if V(x, c) in model:\n                                col[x] = c\n                                break\n                    improved = True\n                    break\n            if improved:\n                break\n        if not improved:\n            return viol_of(col), col, r + 1\n    return viol_of(col), col, rounds\n\n\ndef refine_numpy(\n    hard,\n    ei,\n    ej,\n    k,\n    max_passes=50,\n    kicks=0,\n    kick_size=5,\n    seed=0,\n    tabu_iters=0,\n    slim_rounds_n=0,\n    slim_radius=2,\n):\n    \"\"\"Descenso greedy discreto (numpy, CPU) + basin hopping opcional.\n    Greedy solo remata (~40\u2192~35); con kicks (perturbaci\u00f3n aleatoria +\n    greedy, qued\u00e1ndose con lo mejor) escapa de \u00f3ptimos locales: 12\u21920.\"\"\"\n    import numpy as np\n\n    col0 = np.array(hard, dtype=np.int64).tolist()\n    ei_l = np.array(ei, dtype=np.int64).tolist()\n    ej_l = np.array(ej, dtype=np.int64).tolist()\n    n = max(len(col0), max(ei_l) + 1, max(ej_l) + 1)\n    adj = [[] for _ in range(n)]\n    for a, b in zip(ei_l, ej_l):\n        adj[a].append(b)\n        adj[b].append(a)\n\n    def viol_of(col):\n        return sum(1 for a, b in zip(ei_l, ej_l) if col[a] == col[b])\n\n    def greedy(col):\n        viol = viol_of(col)\n        for _ in range(max_passes):\n            improved = False\n            for v in range(n):\n                cur = col[v]\n                cnt = [0] * k\n                for u in adj[v]:\n                    cnt[col[u]] += 1\n                best_c = min(range(k), key=lambda c: cnt[c])\n                if cnt[best_c] < cnt[cur]:\n                    col[v] = best_c\n                    viol += cnt[best_c] - cnt[cur]\n                    improved = True\n            if not improved:\n                break\n        return col, viol\n\n    rng = np.random.default_rng(seed)\n    best_col, best_v = greedy(list(col0))\n    for _ in range(kicks):\n        cand = list(best_col)\n        for v in rng.integers(0, n, size=kick_size).tolist():\n            cand[v] = int(rng.integers(0, k))\n        cand, v = greedy(cand)\n        if v < best_v:\n            best_col, best_v = cand, v\n            if best_v == 0:\n                break\n    tabu_it = 0\n    if tabu_iters > 0 and best_v > 0:\n        best_v, best_col, tabu_it = tabusearch(\n            best_col, ei_l, ej_l, k, tabu_iters, seed=seed + 999\n        )\n    slim_it = 0\n    if slim_rounds_n > 0 and best_v > 0:\n        best_v, best_col, slim_it = slim_rounds(\n            best_col,\n            ei,\n            ej,\n            k,\n            slim_rounds_n,\n            slim_radius,\n            cap=120,\n            seed=seed + 7777,\n        )\n    return best_v, best_col, tabu_it, slim_it\n\n\ndef run(\n    edges,\n    k,\n    restarts,\n    steps,\n    lr,\n    entropy_coef,\n    temp_init,\n    seed,\n    kicks=0,\n    kick_size=5,\n    tabu_iters=0,\n    slim_rounds_n=0,\n    slim_radius=2,\n):\n    nodes = set()\n    for a, b in edges:\n        nodes.add(a)\n        nodes.add(b)\n    n = max(nodes) + 1\n    ei = jnp.array([a for a, _ in edges], dtype=jnp.int32)\n    ej = jnp.array([b for _, b in edges], dtype=jnp.int32)\n    keys = jax.random.split(jax.random.PRNGKey(seed), restarts)\n    vtrain = jax.vmap(\n        lambda key: train(key, ei, ej, n, k, steps, lr, entropy_coef, temp_init)\n    )\n    t0 = time.time()\n    losses, viols, hards = vtrain(keys)\n    dt = time.time() - t0\n    losses = [float(x) for x in losses]\n    viols = [int(x) for x in viols]\n    best = int(jnp.argmin(jnp.array(viols)))\n    t1 = time.time()\n    refined, refined_col, tabu_it, slim_it = refine_numpy(\n        hards[best],\n        ei,\n        ej,\n        k,\n        kicks=kicks,\n        kick_size=kick_size,\n        seed=seed,\n        tabu_iters=tabu_iters,\n        slim_rounds_n=slim_rounds_n,\n        slim_radius=slim_radius,\n    )\n    refine_s = round(time.time() - t1, 3)\n    return {\n        \"n\": n,\n        \"edges\": len(edges),\n        \"k\": k,\n        \"restarts\": restarts,\n        \"steps\": steps,\n        \"time_s\": round(dt, 3),\n        \"devices\": [str(d) for d in jax.devices()],\n        \"best_restart\": best,\n        \"best_violations\": viols[best],\n        \"refined_violations\": refined,\n        \"refined_coloring\": refined_col,\n        \"refine_s\": refine_s,\n        \"tabu_iters_done\": tabu_it,\n        \"slim_rounds_done\": slim_it,\n        \"best_loss\": losses[best],\n        \"all_violations\": viols,\n    }\n\n\ndef demo():\n    # tri\u00e1ngulo: k2 piso 1/3 (espejo UNSAT), k3 -> 0 (espejo SAT)\n    tri = [(0, 1), (1, 2), (0, 2)]\n    k2 = run(tri, 2, 4, 800, 0.05, 0.02, 1.0, 0)\n    k3 = run(tri, 3, 4, 800, 0.05, 0.02, 1.0, 1)\n    ok = k2[\"best_violations\"] >= 1 and k3[\"best_violations\"] == 0\n    result = {\"triangle_k2\": k2, \"triangle_k3\": k3, \"demo_ok\": bool(ok)}\n    print(json.dumps(result))\n    return 0 if ok else 1\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--demo\", action=\"store_true\")\n    ap.add_argument(\"--edges\", default=None)\n    ap.add_argument(\"--zero-based\", action=\"store_true\")\n    ap.add_argument(\"--k\", type=int, default=5)\n    ap.add_argument(\"--restarts\", type=int, default=8)\n    ap.add_argument(\"--steps\", type=int, default=3000)\n    ap.add_argument(\"--lr\", type=float, default=0.15)\n    ap.add_argument(\"--entropy\", type=float, default=0.05)\n    ap.add_argument(\"--temp\", type=float, default=1.0)\n    ap.add_argument(\"--seed\", type=int, default=0)\n    ap.add_argument(\"--kicks\", type=int, default=0)\n    ap.add_argument(\"--kick-size\", type=int, default=5)\n    ap.add_argument(\"--tabu-iters\", type=int, default=0)\n    ap.add_argument(\"--slim-rounds\", type=int, default=0)\n    ap.add_argument(\"--slim-radius\", type=int, default=2)\n    ap.add_argument(\n        \"--x64\",\n        action=\"store_true\",\n        help=\"float64 (igual que el torch ganador; en TPU emula lento: usar T4)\",\n    )\n    a = ap.parse_args()\n    if a.x64:\n        jax.config.update(\"jax_enable_x64\", True)\n    if a.demo:\n        return demo()\n    if not a.edges:\n        ap.error(\"--edges requerido sin --demo\")\n    edges = load_edges(a.edges, a.zero_based)\n    print(\n        json.dumps(\n            run(\n                edges,\n                a.k,\n                a.restarts,\n                a.steps,\n                a.lr,\n                a.entropy,\n                a.temp,\n                a.seed,\n                kicks=a.kicks,\n                kick_size=a.kick_size,\n                tabu_iters=a.tabu_iters,\n                slim_rounds_n=a.slim_rounds,\n                slim_radius=a.slim_radius,\n            )\n        )\n    )\n    return 0\n\n"
NNSIREN = "#!/usr/bin/env python3\n\"\"\"nn_siren.py \u2014 SIREN por coordenadas para coloreo HN (JAX, CPU ac\u00e1 / TPU all\u00e1).\n\nLiteratura (Mundinger et al. 2024/25): MLPs con activaci\u00f3n seno tienen bias\nespectral a soluciones estructuradas/espacialmente coherentes \u2014 justo lo que\nquiere un grafo unit-distance. Diferencia con nn_jax.py: ah\u00ed cada v\u00e9rtice tiene\nlogits LIBRES (N\u00d7k params, sin geometr\u00eda); ac\u00e1 UNA red coords\u2192colores (params\ncompartidos, la geometr\u00eda manda).\n\nRed: x(2) -> Linear(256, w0=30 seno) -> 2\u00d7 Linear(256, seno) -> Linear(k).\nLoss: media sobre aristas de sum_c p_i p_j - beta*entrop\u00eda (coseno, = colG).\nRefine: greedy + kicks + TabuCol + SLIM reusados de nn_jax (import).\n\nEjemplos:\n  python nn_siren.py --demo\n  python nn_siren.py --vtx 874.vtx.json --edges 874.edge --k 5 --restarts 8 --steps 8000\n\"\"\"\n\nimport argparse\nimport json\nimport sys\nimport time\n\nimport jax\nimport jax.numpy as jnp\n\nfrom nn_jax import load_edges, refine_numpy\n\n\ndef init_siren(key, widths, w0_first=5.0, w0=1.0):\n    params = []\n    keys = jax.random.split(key, len(widths) - 1)\n    for i, (din, dout) in enumerate(zip(widths[:-1], widths[1:])):\n        k1, k2 = jax.random.split(keys[i])\n        bound = jnp.sqrt(6.0 / din) / (w0_first if i == 0 else w0)\n        W = jax.random.uniform(k1, (din, dout), minval=-bound, maxval=bound)\n        b = jnp.zeros((dout,))\n        params.append((W, b))\n    return params\n\n\nW0F = 5.0\n\n\ndef forward(params, x, w0=1.0):\n    h = x\n    for i, (W, b) in enumerate(params):\n        h = h @ W + b\n        if i < len(params) - 1:\n            h = jnp.sin((W0F if i == 0 else w0) * h)\n    return h\n\n\ndef train(key, xs, ei, ej, k, steps, lr, entropy_coef, hidden=256, depth=3):\n    n = xs.shape[0]\n    widths = [2] + [hidden] * depth + [k]\n    params = init_siren(key, widths)\n    m = jax.tree.map(jnp.zeros_like, params)\n    v = jax.tree.map(jnp.zeros_like, params)\n    b1, b2, eps = 0.9, 0.999, 1e-8\n    t_all = jnp.arange(steps)\n    betas = entropy_coef * 0.5 * (1.0 + jnp.cos(jnp.pi * t_all / steps))\n\n    def loss_fn(pm, beta):\n        lg = forward(pm, xs)\n        p = jax.nn.softmax(lg, axis=-1)\n        same = jnp.sum(p[ei] * p[ej], axis=-1).mean()\n        ent = -(p * jnp.log(p + 1e-12)).sum(-1).mean()\n        return same - beta * ent\n\n    @jax.jit\n    def step(carry, tb):\n        t, beta = tb\n        pm, m, v = carry\n        loss, g = jax.value_and_grad(loss_fn)(pm, beta)\n        m = jax.tree.map(lambda a, b: b1 * a + (1 - b1) * b, m, g)\n        v = jax.tree.map(lambda a, b: b2 * a + (1 - b2) * b * b, v, g)\n        upd = jax.tree.map(\n            lambda mi, vi, gi: (\n                lr\n                * (mi / (1 - b1 ** (t + 1)))\n                / (jnp.sqrt(vi / (1 - b2 ** (t + 1))) + eps)\n            ),\n            m,\n            v,\n            g,\n        )\n        pm = jax.tree.map(lambda p, u: p - u, pm, upd)\n        return (pm, m, v), loss\n\n    (pm, _, _), losses = jax.lax.scan(step, (params, m, v), (t_all, betas))\n    lg = forward(pm, xs)\n    hard = jnp.argmax(jax.nn.softmax(lg, axis=-1), axis=-1)\n    viol = jnp.sum(hard[ei] == hard[ej])\n    return losses[-1], viol, hard\n\n\ndef run(\n    vtx,\n    edges,\n    k,\n    restarts,\n    steps,\n    lr,\n    entropy_coef,\n    seed,\n    kicks=0,\n    kick_size=5,\n    tabu_iters=0,\n    slim_rounds_n=0,\n    slim_radius=2,\n    hidden=256,\n    depth=3,\n):\n    import numpy as np\n\n    xs = jnp.array(np.array(vtx, dtype=np.float32))\n    # normalizar coords a [-1,1] para el seno\n    lo, hi = xs.min(), xs.max()\n    xs = 2 * (xs - lo) / (hi - lo + 1e-12) - 1\n    ei = jnp.array([a for a, _ in edges], dtype=jnp.int32)\n    ej = jnp.array([b for _, b in edges], dtype=jnp.int32)\n    n = xs.shape[0]\n    keys = jax.random.split(jax.random.PRNGKey(seed), restarts)\n    results = []\n    t0 = time.time()\n    for r in range(restarts):\n        loss, viol, hard = train(\n            keys[r], xs, ei, ej, k, steps, lr, entropy_coef, hidden, depth\n        )\n        results.append((float(loss), int(viol), np.array(hard)))\n    dt = time.time() - t0\n    results.sort(key=lambda t: t[1])\n    loss, viol, hard = results[0]\n    viols = [v for _, v, _ in results]\n    t1 = time.time()\n    # hard ya es np.array concreto (fuera del jit): refine corre en numpy\n    refined, refined_col, tabu_it, slim_it = refine_numpy(\n        hard,\n        ei,\n        ej,\n        k,\n        kicks=kicks,\n        kick_size=kick_size,\n        seed=seed,\n        tabu_iters=tabu_iters,\n        slim_rounds_n=slim_rounds_n,\n        slim_radius=slim_radius,\n    )\n    refine_s = round(time.time() - t1, 3)\n    return {\n        \"n\": n,\n        \"edges\": len(edges),\n        \"k\": k,\n        \"arch\": f\"siren-{hidden}x{depth}\",\n        \"restarts\": restarts,\n        \"steps\": steps,\n        \"time_s\": round(dt, 3),\n        \"devices\": [str(d) for d in jax.devices()],\n        \"best_violations\": viol,\n        \"refined_violations\": refined,\n        \"refined_coloring\": refined_col,\n        \"refine_s\": refine_s,\n        \"tabu_iters_done\": tabu_it,\n        \"slim_rounds_done\": slim_it,\n        \"best_loss\": loss,\n        \"all_violations\": viols,\n    }\n\n\ndef demo():\n    tri_v = [[0.0, 0.0], [1.0, 0.0], [0.5, 0.8660254]]\n    tri_e = [(0, 1), (1, 2), (0, 2)]\n    k2 = run(tri_v, tri_e, 2, 2, 2000, 0.001, 0.05, 0, hidden=64, depth=2)\n    k3 = run(tri_v, tri_e, 3, 2, 2000, 0.001, 0.05, 1, hidden=64, depth=2)\n    ok = k2[\"best_violations\"] >= 1 and k3[\"best_violations\"] == 0\n    result = {\"triangle_k2\": k2, \"triangle_k3\": k3, \"demo_ok\": bool(ok)}\n    print(json.dumps(result))\n    return 0 if ok else 1\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--demo\", action=\"store_true\")\n    ap.add_argument(\"--vtx\", default=None, help=\"JSON [[x,y],...]\")\n    ap.add_argument(\"--edges\", default=None)\n    ap.add_argument(\"--zero-based\", action=\"store_true\")\n    ap.add_argument(\"--k\", type=int, default=5)\n    ap.add_argument(\"--restarts\", type=int, default=8)\n    ap.add_argument(\"--steps\", type=int, default=8000)\n    ap.add_argument(\"--lr\", type=float, default=0.001)\n    ap.add_argument(\"--entropy\", type=float, default=0.05)\n    ap.add_argument(\"--seed\", type=int, default=0)\n    ap.add_argument(\"--kicks\", type=int, default=0)\n    ap.add_argument(\"--kick-size\", type=int, default=5)\n    ap.add_argument(\"--tabu-iters\", type=int, default=0)\n    ap.add_argument(\"--slim-rounds\", type=int, default=0)\n    ap.add_argument(\"--slim-radius\", type=int, default=2)\n    ap.add_argument(\"--hidden\", type=int, default=256)\n    ap.add_argument(\"--depth\", type=int, default=3)\n    a = ap.parse_args()\n    if a.demo:\n        return demo()\n    if not a.vtx or not a.edges:\n        ap.error(\"--vtx y --edges requeridos sin --demo\")\n    vtx = json.load(open(a.vtx))\n    edges = load_edges(a.edges, a.zero_based)\n    print(\n        json.dumps(\n            run(\n                vtx,\n                edges,\n                a.k,\n                a.restarts,\n                a.steps,\n                a.lr,\n                a.entropy,\n                a.seed,\n                kicks=a.kicks,\n                kick_size=a.kick_size,\n                tabu_iters=a.tabu_iters,\n                slim_rounds_n=a.slim_rounds,\n                slim_radius=a.slim_radius,\n                hidden=a.hidden,\n                depth=a.depth,\n            )\n        )\n    )\n    return 0\n\n"
open('/tmp/nn_jax.py','w').write(NNJAX)
open('/tmp/nn_siren.py','w').write(NNSIREN)
EDGEG3 = "e 1 2\ne 1 7\ne 1 11\ne 1 14\ne 1 15\ne 1 19\ne 1 22\ne 1 25\ne 1 28\ne 1 32\ne 1 36\ne 1 40\ne 1 43\ne 1 44\ne 1 50\ne 1 57\ne 1 61\ne 1 64\ne 1 67\ne 1 70\ne 1 80\ne 1 84\ne 1 92\ne 1 95\ne 1 98\ne 1 102\ne 1 106\ne 1 122\ne 1 143\ne 1 157\ne 1 185\ne 1 211\ne 1 216\ne 1 238\ne 1 242\ne 1 247\ne 1 252\ne 1 256\ne 1 265\ne 1 270\ne 1 274\ne 1 278\ne 1 283\ne 1 290\ne 1 318\ne 1 322\ne 1 330\ne 1 368\ne 1 380\ne 1 409\ne 1 417\ne 1 521\ne 1 541\ne 1 576\ne 1 595\ne 1 619\ne 1 656\ne 1 665\ne 1 1067\ne 1 1072\ne 1 1076\ne 1 1079\ne 1 1080\ne 1 1084\ne 1 1087\ne 1 1090\ne 1 1093\ne 1 1097\ne 1 1101\ne 1 1105\ne 1 1108\ne 1 1109\ne 1 1115\ne 1 1122\ne 1 1126\ne 1 1129\ne 1 1132\ne 1 1135\ne 1 1145\ne 1 1149\ne 1 1157\ne 1 1160\ne 1 1163\ne 1 1167\ne 1 1171\ne 1 1187\ne 1 1208\ne 1 1222\ne 1 1250\ne 1 1276\ne 1 1281\ne 1 1303\ne 1 1307\ne 1 1312\ne 1 1317\ne 1 1321\ne 1 1330\ne 1 1335\ne 1 1339\ne 1 1343\ne 1 1348\ne 1 1355\ne 1 1383\ne 1 1387\ne 1 1395\ne 1 1433\ne 1 1445\ne 1 1474\ne 1 1482\ne 1 1586\ne 1 1606\ne 1 1641\ne 1 1660\ne 1 1684\ne 1 1721\ne 1 1730\ne 2 3\ne 2 80\ne 2 111\ne 2 114\ne 2 118\ne 2 238\ne 2 483\ne 3 4\ne 3 69\ne 3 261\ne 3 274\ne 3 316\ne 3 410\ne 3 758\ne 4 5\ne 4 27\ne 4 172\ne 4 281\ne 4 282\ne 4 306\ne 4 369\ne 4 416\ne 4 483\ne 4 758\ne 5 6\ne 5 49\ne 5 69\ne 5 77\ne 5 82\ne 5 170\ne 5 180\ne 5 293\ne 5 307\ne 5 310\ne 5 313\ne 5 416\ne 5 420\ne 5 491\ne 5 520\ne 5 594\ne 5 609\ne 5 640\ne 5 698\ne 5 724\ne 5 740\ne 5 757\ne 5 858\ne 5 951\ne 6 10\ne 6 14\ne 6 18\ne 6 31\ne 6 35\ne 6 39\ne 6 43\ne 6 49\ne 6 53\ne 6 56\ne 6 60\ne 6 73\ne 6 83\ne 6 87\ne 6 110\ne 6 117\ne 6 121\ne 6 132\ne 6 142\ne 6 150\ne 6 164\ne 6 171\ne 6 184\ne 6 189\ne 6 196\ne 6 215\ne 6 220\ne 6 227\ne 6 231\ne 6 237\ne 6 246\ne 6 251\ne 6 260\ne 6 264\ne 6 269\ne 6 273\ne 6 282\ne 6 297\ne 6 304\ne 6 326\ne 6 337\ne 6 353\ne 6 364\ne 6 372\ne 6 379\ne 6 390\ne 6 408\ne 6 413\ne 6 430\ne 6 436\ne 6 468\ne 6 514\ne 6 525\ne 6 529\ne 6 551\ne 6 580\ne 6 698\ne 6 774\ne 6 780\ne 6 792\ne 6 830\ne 6 845\ne 6 847\ne 6 871\ne 7 8\ne 7 29\ne 7 33\ne 7 65\ne 7 84\ne 7 93\ne 7 152\ne 7 168\ne 7 172\ne 7 175\ne 7 178\ne 7 181\ne 7 265\ne 7 279\ne 7 410\ne 7 418\ne 7 548\ne 7 577\ne 7 596\ne 7 632\ne 7 718\ne 8 9\ne 8 116\ne 8 181\ne 8 252\ne 8 316\ne 8 357\ne 8 618\ne 8 678\ne 8 695\ne 8 1008\ne 9 10\ne 9 163\ne 9 175\ne 9 223\ne 9 282\ne 9 728\ne 9 949\ne 9 1008\ne 10 72\ne 10 96\ne 10 110\ne 10 529\ne 10 695\ne 10 737\ne 10 784\ne 10 891\ne 10 974\ne 10 1020\ne 11 12\ne 11 85\ne 11 95\ne 11 109\ne 11 200\ne 11 203\ne 11 206\ne 11 209\ne 11 278\ne 12 13\ne 12 161\ne 12 162\ne 12 200\ne 12 322\ne 12 370\ne 12 371\ne 12 454\ne 12 481\ne 12 726\ne 13 14\ne 13 37\ne 13 87\ne 13 119\ne 13 235\ne 13 240\ne 13 279\ne 13 294\ne 13 424\ne 13 460\ne 13 596\ne 13 602\ne 13 603\ne 13 629\ne 13 721\ne 14 16\ne 14 20\ne 14 24\ne 14 25\ne 14 27\ne 14 38\ne 14 41\ne 14 43\ne 14 47\ne 14 52\ne 14 55\ne 14 66\ne 14 88\ne 14 96\ne 14 100\ne 14 101\ne 14 120\ne 14 165\ne 14 175\ne 14 193\ne 14 206\ne 14 223\ne 14 245\ne 14 268\ne 14 294\ne 14 298\ne 14 301\ne 14 304\ne 14 305\ne 14 308\ne 14 311\ne 14 314\ne 14 349\ne 14 371\ne 14 373\ne 14 423\ne 14 447\ne 14 448\ne 14 450\ne 14 464\ne 14 475\ne 14 483\ne 14 484\ne 14 496\ne 14 499\ne 14 505\ne 14 508\ne 14 513\ne 14 520\ne 14 540\ne 14 562\ne 14 573\ne 14 597\ne 14 598\ne 14 613\ne 14 641\ne 14 652\ne 14 664\ne 14 674\ne 14 704\ne 14 729\ne 14 731\ne 15 16\ne 15 36\ne 15 74\ne 15 114\ne 15 181\ne 15 252\ne 15 338\ne 15 341\ne 15 344\ne 15 347\ne 15 350\ne 15 354\ne 15 361\ne 15 365\ne 15 376\ne 15 444\ne 15 542\ne 15 616\ne 15 682\ne 15 713\ne 15 735\ne 16 17\ne 16 75\ne 16 223\ne 16 341\ne 16 359\ne 16 543\ne 16 643\ne 16 653\ne 16 654\ne 16 876\ne 17 18\ne 17 259\ne 17 263\ne 17 295\ne 17 347\ne 17 363\ne 17 379\ne 17 513\ne 17 825\ne 17 1053\ne 18 21\ne 18 56\ne 18 199\ne 18 230\ne 18 296\ne 18 389\ne 18 404\ne 18 439\ne 18 565\ne 18 591\ne 18 643\ne 18 661\ne 18 691\ne 18 988\ne 19 20\ne 19 136\ne 19 209\ne 19 254\ne 19 274\ne 19 365\ne 19 384\ne 19 387\ne 19 391\ne 19 394\ne 19 397\ne 19 400\ne 19 405\ne 19 440\ne 19 479\ne 19 500\ne 19 521\ne 19 648\ne 19 660\ne 19 669\ne 19 715\ne 20 21\ne 20 27\ne 20 207\ne 20 391\ne 20 403\ne 20 404\ne 20 480\ne 20 615\ne 20 670\ne 20 923\ne 21 269\ne 21 311\ne 21 400\ne 21 458\ne 21 694\ne 21 814\ne 21 835\ne 22 23\ne 22 54\ne 22 57\ne 22 152\ne 22 290\ne 22 443\ne 22 448\ne 22 534\ne 23 24\ne 23 58\ne 23 197\ne 23 242\ne 23 302\ne 23 443\ne 23 447\ne 23 568\ne 24 54\ne 24 73\ne 24 79\ne 24 210\ne 24 314\ne 24 361\ne 24 362\ne 24 397\ne 24 444\ne 24 465\ne 24 481\ne 24 519\ne 24 555\ne 24 648\ne 25 26\ne 25 451\ne 25 454\ne 25 459\ne 25 462\ne 25 644\ne 26 27\ne 26 274\ne 26 457\ne 26 466\ne 26 497\ne 26 549\ne 26 593\ne 27 104\ne 27 173\ne 27 274\ne 27 294\ne 27 458\ne 27 467\ne 27 550\ne 27 571\ne 27 594\ne 27 599\ne 27 600\ne 27 717\ne 27 748\ne 27 847\ne 27 874\ne 27 876\ne 27 920\ne 27 927\ne 27 1016\ne 27 1032\ne 28 29\ne 28 318\ne 28 465\ne 28 469\ne 28 472\ne 28 475\ne 28 478\ne 28 576\ne 29 30\ne 29 94\ne 29 173\ne 29 349\ne 29 357\ne 29 402\ne 29 471\ne 29 479\ne 29 503\ne 29 561\ne 29 562\ne 29 577\ne 29 623\ne 29 702\ne 30 31\ne 30 175\ne 30 364\ne 30 471\ne 30 475\ne 30 480\ne 30 486\ne 30 550\ne 30 827\ne 30 983\ne 31 63\ne 31 94\ne 31 104\ne 31 117\ne 31 177\ne 31 366\ne 31 367\ne 31 393\ne 31 404\ne 31 456\ne 31 557\ne 31 625\ne 31 717\ne 31 923\ne 32 33\ne 32 90\ne 32 276\ne 32 330\ne 32 481\ne 32 484\ne 32 489\ne 32 531\ne 32 595\ne 33 34\ne 33 90\ne 33 213\ne 33 416\ne 33 460\ne 33 483\ne 33 485\ne 33 491\ne 33 538\ne 33 562\ne 33 575\ne 33 586\ne 33 596\ne 34 35\ne 34 141\ne 34 164\ne 34 168\ne 34 307\ne 34 461\ne 34 485\ne 34 487\ne 34 531\ne 34 677\ne 34 895\ne 34 997\ne 34 1047\ne 34 1048\ne 35 121\ne 35 241\ne 35 441\ne 35 442\ne 35 471\ne 35 486\ne 35 562\ne 35 671\ne 35 742\ne 35 744\ne 35 795\ne 35 804\ne 35 871\ne 35 895\ne 35 916\ne 35 924\ne 36 37\ne 36 118\ne 36 119\ne 36 386\ne 36 492\ne 36 497\ne 36 500\ne 36 503\ne 37 38\ne 37 102\ne 37 288\ne 37 685\ne 37 735\ne 38 39\ne 38 366\ne 38 405\ne 38 475\ne 38 500\ne 38 576\ne 38 581\ne 38 589\ne 38 590\ne 38 652\ne 39 406\ne 39 408\ne 39 501\ne 39 502\ne 39 580\ne 39 581\ne 39 685\ne 39 879\ne 39 1024\ne 40 41\ne 40 42\ne 40 61\ne 40 368\ne 40 506\ne 40 509\ne 40 647\ne 41 42\ne 41 196\ne 41 373\ne 41 510\ne 41 540\ne 41 603\ne 41 642\ne 41 791\ne 42 43\ne 42 196\ne 42 219\ne 42 328\ne 42 357\ne 42 377\ne 42 479\ne 42 604\ne 42 626\ne 42 628\ne 42 680\ne 42 716\ne 42 857\ne 42 934\ne 43 46\ne 43 69\ne 43 79\ne 43 94\ne 43 106\ne 43 109\ne 43 111\ne 43 135\ne 43 144\ne 43 158\ne 43 168\ne 43 174\ne 43 186\ne 43 192\ne 43 214\ne 43 219\ne 43 221\ne 43 224\ne 43 228\ne 43 232\ne 43 235\ne 43 236\ne 43 257\ne 43 266\ne 43 286\ne 43 291\ne 43 316\ne 43 327\ne 43 333\ne 43 334\ne 43 341\ne 43 378\ne 43 386\ne 43 391\ne 43 396\ne 43 402\ne 43 427\ne 43 436\ne 43 469\ne 43 511\ne 43 526\ne 43 531\ne 43 534\ne 43 547\ne 43 552\ne 43 566\ne 43 579\ne 43 581\ne 43 604\ne 43 606\ne 43 610\ne 43 620\ne 43 631\ne 43 634\ne 43 637\ne 43 657\ne 43 668\ne 43 691\ne 43 695\ne 43 701\ne 43 708\ne 43 748\ne 43 752\ne 43 753\ne 43 765\ne 43 798\ne 43 962\ne 43 1036\ne 44 45\ne 44 47\ne 44 64\ne 44 380\ne 44 511\ne 44 515\ne 44 518\ne 45 46\ne 45 232\ne 45 252\ne 45 399\ne 45 564\ne 45 705\ne 45 823\ne 46 79\ne 46 139\ne 46 203\ne 46 243\ne 46 244\ne 46 266\ne 46 276\ne 46 353\ne 46 515\ne 46 564\ne 46 703\ne 46 706\ne 46 732\ne 46 734\ne 46 745\ne 47 48\ne 47 52\ne 47 142\ne 47 399\ne 47 496\ne 47 511\ne 47 554\ne 47 585\ne 47 586\ne 47 592\ne 48 49\ne 48 167\ne 48 301\ne 48 353\ne 48 382\ne 48 515\ne 48 516\ne 49 89\ne 49 160\ne 49 167\ne 49 281\ne 49 480\ne 49 554\ne 49 589\ne 49 615\ne 49 634\ne 49 711\ne 49 712\ne 49 725\ne 49 891\ne 50 51\ne 50 54\ne 50 417\ne 50 545\ne 50 548\ne 50 552\ne 50 555\ne 50 560\ne 50 563\ne 51 52\ne 51 120\ne 51 330\ne 51 452\ne 51 478\ne 51 479\ne 51 506\ne 51 515\ne 51 517\ne 51 546\ne 52 53\ne 52 64\ne 52 176\ne 52 452\ne 52 508\ne 52 547\ne 52 553\ne 52 706\ne 53 138\ne 53 142\ne 53 336\ne 53 390\ne 53 453\ne 53 517\ne 53 547\ne 53 591\ne 53 906\ne 53 914\ne 53 926\ne 53 942\ne 53 995\ne 54 55\ne 54 444\ne 54 477\ne 54 563\ne 55 56\ne 55 66\ne 55 105\ne 55 152\ne 55 154\ne 55 232\ne 55 497\ne 55 503\ne 55 509\ne 55 535\ne 55 548\ne 55 549\ne 55 663\ne 55 727\ne 56 153\ne 56 171\ne 56 232\ne 56 234\ne 56 399\ne 56 429\ne 56 474\ne 56 477\ne 56 494\ne 56 527\ne 56 537\ne 56 550\ne 56 554\ne 56 571\ne 56 618\ne 56 659\ne 56 680\ne 56 728\ne 56 740\ne 56 881\ne 56 882\ne 56 1053\ne 57 58\ne 57 566\ne 57 570\ne 57 573\ne 57 619\ne 58 59\ne 58 90\ne 58 112\ne 58 242\ne 58 333\ne 58 395\ne 58 396\ne 58 460\ne 58 567\ne 58 681\ne 58 692\ne 59 60\ne 59 91\ne 59 132\ne 59 166\ne 59 301\ne 59 302\ne 59 567\ne 59 573\ne 59 684\ne 59 693\ne 59 837\ne 60 63\ne 60 75\ne 60 87\ne 60 91\ne 60 97\ne 60 333\ne 60 426\ne 60 488\ne 60 491\ne 60 575\ne 60 654\ne 60 719\ne 60 720\ne 60 1040\ne 61 62\ne 61 332\ne 61 540\ne 61 601\ne 61 604\ne 61 665\ne 62 63\ne 62 94\ne 62 102\ne 62 333\ne 62 503\ne 62 509\ne 62 601\ne 62 605\ne 62 624\ne 62 645\ne 62 662\ne 62 749\ne 63 510\ne 63 540\ne 63 605\ne 63 625\ne 63 729\ne 64 65\ne 64 92\ne 64 444\ne 64 519\ne 64 547\ne 64 607\ne 65 66\ne 65 93\ne 65 170\ne 65 176\ne 65 520\ne 65 607\ne 65 705\ne 66 171\ne 66 181\ne 66 194\ne 66 208\ne 66 209\ne 66 317\ne 66 444\ne 66 446\ne 66 450\ne 66 479\ne 66 498\ne 66 633\ne 66 739\ne 66 749\ne 66 833\ne 66 927\ne 67 68\ne 67 95\ne 67 541\ne 67 610\ne 67 613\ne 67 616\ne 68 69\ne 68 93\ne 68 122\ne 68 248\ne 68 307\ne 68 522\ne 68 538\ne 68 604\ne 68 953\ne 69 108\ne 69 123\ne 69 520\ne 69 533\ne 69 634\ne 69 638\ne 69 639\ne 69 757\ne 70 71\ne 70 74\ne 70 76\ne 70 78\ne 70 102\ne 70 239\ne 70 248\ne 70 271\ne 70 632\ne 70 635\ne 70 638\ne 70 641\ne 70 644\ne 70 647\ne 70 798\ne 71 72\ne 71 74\ne 71 79\ne 71 210\ne 71 252\ne 71 611\ne 71 695\ne 71 918\ne 72 73\ne 72 75\ne 72 205\ne 72 210\ne 72 223\ne 72 407\ne 72 612\ne 72 641\ne 72 733\ne 72 737\ne 72 743\ne 72 913\ne 72 914\ne 72 918\ne 72 1057\ne 73 77\ne 73 79\ne 73 125\ne 73 146\ne 73 167\ne 73 189\ne 73 277\ne 73 325\ne 73 329\ne 73 353\ne 73 363\ne 73 445\ne 73 477\ne 73 556\ne 73 559\ne 73 568\ne 73 612\ne 73 651\ne 73 693\ne 73 793\ne 73 860\ne 73 941\ne 74 75\ne 74 333\ne 74 469\ne 74 473\ne 74 477\ne 74 494\ne 74 735\ne 74 856\ne 75 250\ne 75 408\ne 75 641\ne 75 856\ne 75 1058\ne 76 77\ne 76 368\ne 76 477\ne 76 533\ne 76 547\ne 76 591\ne 76 592\ne 76 636\ne 76 647\ne 76 691\ne 76 740\ne 76 907\ne 77 103\ne 77 124\ne 77 167\ne 77 276\ne 77 327\ne 77 369\ne 77 489\ne 77 507\ne 77 512\ne 77 519\ne 77 547\ne 77 609\ne 77 806\ne 78 79\ne 78 167\ne 78 409\ne 78 478\ne 78 515\ne 78 634\ne 78 894\ne 79 145\ne 79 324\ne 79 472\ne 79 611\ne 79 692\ne 79 860\ne 80 81\ne 80 576\ne 80 579\ne 80 649\ne 80 652\ne 80 682\ne 81 82\ne 81 105\ne 81 166\ne 81 417\ne 81 482\ne 81 520\ne 81 532\ne 81 573\ne 81 592\ne 81 596\ne 81 655\ne 82 83\ne 82 293\ne 82 401\ne 82 427\ne 82 474\ne 82 553\ne 82 579\ne 82 583\ne 82 621\ne 82 677\ne 82 901\ne 82 908\ne 82 1030\ne 83 227\ne 83 260\ne 83 367\ne 83 566\ne 83 573\ne 83 683\ne 83 819\ne 83 875\ne 83 908\ne 84 85\ne 84 88\ne 84 90\ne 84 198\ne 84 319\ne 84 462\ne 84 607\ne 84 672\ne 84 675\ne 84 678\ne 84 681\ne 84 686\ne 84 689\ne 84 752\ne 85 86\ne 85 144\ne 85 333\ne 85 446\ne 85 493\ne 85 618\ne 85 659\ne 85 681\ne 85 719\ne 85 726\ne 85 1028\ne 86 87\ne 86 117\ne 86 134\ne 86 135\ne 86 194\ne 86 200\ne 86 202\ne 86 221\ne 86 602\ne 86 625\ne 86 633\ne 86 675\ne 86 676\ne 86 726\ne 86 1027\ne 87 113\ne 87 128\ne 87 162\ne 87 195\ne 87 235\ne 87 241\ne 87 289\ne 87 300\ne 87 412\ne 87 420\ne 87 425\ne 87 430\ne 87 439\ne 87 461\ne 87 625\ne 87 659\ne 87 677\ne 87 685\ne 87 963\ne 87 964\ne 87 1013\ne 88 89\ne 88 91\ne 88 175\ne 88 199\ne 88 314\ne 88 320\ne 88 462\ne 88 553\ne 88 554\ne 88 684\ne 88 688\ne 88 714\ne 88 719\ne 88 728\ne 88 752\ne 88 780\ne 88 821\ne 88 865\ne 88 884\ne 88 983\ne 88 988\ne 88 1030\ne 89 184\ne 89 301\ne 89 321\ne 89 675\ne 89 714\ne 89 905\ne 89 929\ne 90 91\ne 90 271\ne 90 333\ne 90 463\ne 90 482\ne 90 487\ne 90 490\ne 90 587\ne 90 630\ne 90 631\ne 90 686\ne 91 230\ne 91 412\ne 91 413\ne 91 463\ne 91 484\ne 91 485\ne 91 487\ne 91 536\ne 91 589\ne 91 621\ne 91 688\ne 91 801\ne 91 888\ne 92 93\ne 92 185\ne 92 508\ne 92 637\ne 92 692\ne 93 94\ne 93 99\ne 93 111\ne 93 169\ne 93 177\ne 93 194\ne 93 419\ne 93 649\ne 93 673\ne 94 99\ne 94 103\ne 94 136\ne 94 192\ne 94 317\ne 94 393\ne 94 507\ne 94 552\ne 94 624\ne 94 650\ne 94 660\ne 94 811\ne 95 96\ne 95 503\ne 95 695\ne 95 699\ne 95 702\ne 95 705\ne 96 97\ne 96 206\ne 96 210\ne 96 550\ne 96 613\ne 96 617\ne 96 658\ne 96 695\ne 96 971\ne 97 301\ne 97 375\ne 97 551\ne 97 646\ne 97 697\ne 97 699\ne 97 707\ne 97 971\ne 97 1013\ne 98 99\ne 98 100\ne 98 211\ne 98 213\ne 98 538\ne 98 619\ne 98 708\ne 98 713\ne 98 715\ne 99 129\ne 99 191\ne 99 203\ne 99 214\ne 99 252\ne 99 455\ne 99 456\ne 99 539\ne 99 626\ne 99 709\ne 99 713\ne 99 716\ne 100 101\ne 100 172\ne 100 297\ne 100 359\ne 100 456\ne 100 548\ne 100 555\ne 100 674\ne 100 708\ne 100 711\ne 101 174\ne 101 211\ne 101 312\ne 101 371\ne 101 422\ne 101 549\ne 101 551\ne 101 811\ne 102 103\ne 102 105\ne 102 216\ne 102 287\ne 102 405\ne 102 489\ne 102 509\ne 102 606\ne 102 608\ne 102 706\ne 102 718\ne 102 721\ne 102 723\ne 102 726\ne 102 729\ne 102 732\ne 102 735\ne 102 738\ne 103 104\ne 103 124\ne 103 146\ne 103 257\ne 103 274\ne 103 405\ne 103 638\ne 103 850\ne 104 514\ne 104 572\ne 104 594\ne 104 729\ne 104 850\ne 104 926\ne 105 165\ne 105 380\ne 105 474\ne 105 503\ne 105 586\ne 105 596\ne 105 614\ne 105 633\ne 105 726\ne 106 107\ne 106 217\ne 106 323\ne 106 492\ne 106 522\ne 106 545\ne 106 753\ne 107 108\ne 107 133\ne 107 191\ne 107 212\ne 107 239\ne 107 248\ne 107 265\ne 107 266\ne 107 937\ne 108 109\ne 108 123\ne 108 633\ne 108 723\ne 108 724\ne 109 110\ne 109 162\ne 109 204\ne 109 205\ne 109 206\ne 109 378\ne 109 633\ne 109 695\ne 109 772\ne 109 824\ne 109 970\ne 109 1010\ne 109 1028\ne 110 206\ne 110 351\ne 110 379\ne 110 724\ne 110 725\ne 110 736\ne 110 772\ne 110 1009\ne 111 112\ne 111 115\ne 111 164\ne 111 214\ne 111 419\ne 111 483\ne 111 579\ne 111 692\ne 111 758\ne 111 955\ne 111 993\ne 112 113\ne 112 115\ne 112 118\ne 112 172\ne 112 226\ne 112 386\ne 112 411\ne 112 447\ne 112 460\ne 112 673\ne 112 674\ne 112 688\ne 112 901\ne 113 119\ne 113 121\ne 113 134\ne 113 139\ne 113 213\ne 113 214\ne 113 241\ne 113 244\ne 113 386\ne 113 437\ne 113 575\ne 113 627\ne 113 713\ne 113 1029\ne 114 115\ne 114 118\ne 114 190\ne 114 316\ne 114 332\ne 114 333\ne 114 601\ne 114 603\ne 114 654\ne 114 682\ne 114 791\ne 115 116\ne 115 194\ne 115 195\ne 115 341\ne 115 435\ne 115 488\ne 115 654\ne 115 683\ne 115 790\ne 115 809\ne 115 833\ne 115 909\ne 115 1010\ne 116 117\ne 116 172\ne 116 186\ne 116 190\ne 116 191\ne 116 215\ne 116 243\ne 116 266\ne 116 335\ne 116 358\ne 116 360\ne 116 456\ne 116 697\ne 116 980\ne 116 1007\ne 117 138\ne 117 192\ne 117 199\ne 117 202\ne 117 249\ne 117 337\ne 117 340\ne 117 346\ne 117 358\ne 117 388\ne 117 524\ne 117 618\ne 117 646\ne 117 651\ne 117 664\ne 117 671\ne 117 683\ne 117 714\ne 117 762\ne 117 827\ne 117 976\ne 118 119\ne 118 410\ne 118 687\ne 119 120\ne 119 238\ne 119 240\ne 120 121\ne 120 254\ne 120 286\ne 120 350\ne 120 381\ne 120 415\ne 120 457\ne 120 500\ne 120 506\ne 120 542\ne 120 560\ne 120 562\ne 120 687\ne 121 255\ne 121 277\ne 121 282\ne 121 286\ne 121 320\ne 121 343\ne 121 346\ne 121 351\ne 121 356\ne 121 383\ne 121 416\ne 121 458\ne 121 501\ne 121 517\ne 121 575\ne 121 628\ne 121 688\ne 121 717\ne 121 841\ne 121 842\ne 121 888\ne 122 123\ne 122 126\ne 122 129\ne 122 133\ne 122 134\ne 122 136\ne 122 139\ne 122 242\ne 122 443\ne 122 450\ne 122 451\ne 122 624\ne 122 672\ne 122 765\ne 123 124\ne 123 129\ne 123 148\ne 123 256\ne 123 309\ne 123 310\ne 123 378\ne 123 706\ne 123 956\ne 124 125\ne 124 136\ne 124 174\ne 124 706\ne 124 1014\ne 124 1016\ne 125 146\ne 125 205\ne 125 237\ne 125 335\ne 125 358\ne 125 378\ne 125 391\ne 125 397\ne 125 449\ne 125 628\ne 125 1032\ne 125 1060\ne 125 1064\ne 126 127\ne 126 139\ne 126 235\ne 126 283\ne 126 334\ne 126 460\ne 126 461\ne 126 1035\ne 127 128\ne 127 148\ne 127 151\ne 127 235\ne 127 236\ne 127 279\ne 127 446\ne 127 1034\ne 128 149\ne 128 168\ne 128 237\ne 128 279\ne 128 280\ne 128 348\ne 128 378\ne 128 411\ne 128 461\ne 128 720\ne 128 838\ne 128 1011\ne 128 1034\ne 128 1063\ne 129 130\ne 129 203\ne 129 316\ne 129 368\ne 129 369\ne 129 377\ne 129 434\ne 129 452\ne 129 453\ne 129 498\ne 129 546\ne 129 547\ne 130 131\ne 130 204\ne 130 219\ne 130 428\ne 130 435\ne 130 453\ne 130 709\ne 130 765\ne 130 906\ne 130 956\ne 130 999\ne 130 1010\ne 131 132\ne 131 219\ne 131 224\ne 131 369\ne 131 374\ne 131 458\ne 131 628\ne 131 693\ne 131 694\ne 131 698\ne 131 794\ne 131 806\ne 131 867\ne 131 1021\ne 132 321\ne 132 379\ne 132 396\ne 132 413\ne 132 499\ne 132 571\ne 132 746\ne 132 794\ne 132 837\ne 133 134\ne 133 240\ne 133 279\ne 133 380\ne 133 381\ne 133 434\ne 133 492\ne 133 495\ne 133 623\ne 133 633\ne 133 1063\ne 134 135\ne 134 148\ne 134 191\ne 134 270\ne 134 316\ne 134 346\ne 134 360\ne 134 713\ne 134 958\ne 135 158\ne 135 231\ne 135 241\ne 135 271\ne 135 493\ne 135 505\ne 135 604\ne 135 665\ne 135 671\ne 135 759\ne 135 1006\ne 136 137\ne 136 138\ne 136 334\ne 136 346\ne 136 394\ne 136 507\ne 136 517\ne 136 817\ne 136 923\ne 137 138\ne 137 305\ne 137 335\ne 137 387\ne 137 453\ne 137 508\ne 138 249\ne 138 336\ne 138 405\ne 138 919\ne 138 980\ne 139 140\ne 139 417\ne 139 431\ne 139 517\ne 139 620\ne 139 627\ne 139 816\ne 140 141\ne 140 227\ne 140 353\ne 140 423\ne 140 450\ne 140 461\ne 140 816\ne 140 995\ne 141 142\ne 141 234\ne 141 307\ne 141 527\ne 141 528\ne 141 996\ne 142 156\ne 142 249\ne 142 364\ne 142 511\ne 142 583\ne 142 844\ne 142 861\ne 142 883\ne 143 144\ne 143 147\ne 143 151\ne 143 154\ne 143 247\ne 143 349\ne 143 656\ne 143 663\ne 144 145\ne 144 202\ne 144 203\ne 144 236\ne 144 349\ne 144 429\ne 144 468\ne 144 618\ne 144 657\ne 144 1003\ne 144 1026\ne 145 146\ne 145 203\ne 145 217\ne 145 465\ne 145 650\ne 146 187\ne 146 221\ne 146 259\ne 146 354\ne 146 366\ne 146 392\ne 146 465\ne 146 467\ne 146 468\ne 146 469\ne 146 735\ne 146 849\ne 146 950\ne 147 148\ne 147 197\ne 147 202\ne 147 242\ne 147 294\ne 147 602\ne 147 664\ne 147 666\ne 147 681\ne 148 149\ne 148 151\ne 148 158\ne 148 285\ne 148 294\ne 148 316\ne 148 447\ne 148 769\ne 148 1016\ne 149 150\ne 149 163\ne 149 282\ne 149 295\ne 149 303\ne 149 310\ne 149 313\ne 149 348\ne 149 360\ne 149 383\ne 149 769\ne 150 158\ne 150 165\ne 150 231\ne 150 383\ne 150 474\ne 150 514\ne 150 768\ne 150 773\ne 150 1011\ne 151 152\ne 151 154\ne 151 217\ne 151 283\ne 151 348\ne 151 1026\ne 152 153\ne 152 446\ne 152 447\ne 152 649\ne 152 663\ne 152 705\ne 153 168\ne 153 337\ne 153 534\ne 153 578\ne 153 618\ne 153 889\ne 153 910\ne 153 1002\ne 153 1026\ne 153 1033\ne 154 155\ne 154 294\ne 154 417\ne 154 429\ne 154 434\ne 154 623\ne 155 156\ne 155 295\ne 155 348\ne 155 349\ne 155 423\ne 155 429\ne 156 282\ne 156 305\ne 156 398\ne 156 423\ne 156 428\ne 156 434\ne 156 453\ne 156 495\ne 156 571\ne 156 844\ne 157 158\ne 157 161\ne 157 165\ne 157 256\ne 157 384\ne 157 665\ne 158 159\ne 158 165\ne 158 257\ne 158 285\ne 158 768\ne 158 972\ne 158 973\ne 158 992\ne 158 1059\ne 159 160\ne 159 384\ne 159 391\ne 159 442\ne 159 582\ne 159 594\ne 159 615\ne 159 671\ne 159 788\ne 159 789\ne 159 954\ne 159 984\ne 159 1016\ne 159 1042\ne 159 1059\ne 160 411\ne 160 436\ne 160 516\ne 160 582\ne 160 622\ne 160 634\ne 160 757\ne 160 777\ne 160 795\ne 160 854\ne 160 892\ne 160 894\ne 160 895\ne 160 899\ne 160 901\ne 160 904\ne 160 929\ne 160 975\ne 160 1011\ne 160 1048\ne 161 162\ne 161 235\ne 161 252\ne 161 291\ne 161 323\ne 161 692\ne 161 732\ne 161 972\ne 162 163\ne 162 165\ne 162 223\ne 162 244\ne 162 272\ne 162 289\ne 162 327\ne 162 339\ne 162 372\ne 162 693\ne 162 696\ne 162 722\ne 162 773\ne 162 825\ne 162 877\ne 162 972\ne 162 1013\ne 163 164\ne 163 272\ne 163 313\ne 163 724\ne 163 947\ne 164 177\ne 164 215\ne 164 483\ne 164 580\ne 164 693\ne 164 993\ne 164 996\ne 164 997\ne 165 166\ne 165 179\ne 165 288\ne 165 381\ne 165 505\ne 165 513\ne 165 615\ne 165 633\ne 166 167\ne 166 431\ne 166 491\ne 166 693\ne 166 778\ne 167 431\ne 167 499\ne 167 592\ne 167 641\ne 167 894\ne 168 169\ne 168 170\ne 168 175\ne 168 182\ne 168 225\ne 168 234\ne 168 266\ne 168 313\ne 168 411\ne 168 471\ne 168 677\ne 168 752\ne 168 946\ne 168 975\ne 168 1008\ne 169 170\ne 169 177\ne 169 183\ne 169 393\ne 169 637\ne 169 709\ne 169 714\ne 169 821\ne 169 953\ne 169 980\ne 169 985\ne 169 993\ne 169 1000\ne 169 1002\ne 170 171\ne 170 176\ne 170 547\ne 170 740\ne 170 900\ne 170 925\ne 171 182\ne 171 183\ne 171 205\ne 171 208\ne 171 445\ne 171 622\ne 171 832\ne 171 845\ne 171 862\ne 171 913\ne 171 987\ne 171 1011\ne 171 1033\ne 171 1051\ne 172 173\ne 172 178\ne 172 180\ne 172 225\ne 172 242\ne 172 306\ne 172 360\ne 172 447\ne 172 675\ne 172 711\ne 173 174\ne 173 358\ne 173 383\ne 173 385\ne 173 386\ne 173 505\ne 173 666\ne 173 670\ne 173 712\ne 173 717\ne 173 1006\ne 173 1017\ne 174 201\ne 174 211\ne 174 291\ne 174 358\ne 174 551\ne 174 640\ne 174 699\ne 174 702\ne 174 708\ne 174 716\ne 174 1012\ne 174 1042\ne 175 176\ne 175 177\ne 175 280\ne 175 281\ne 175 485\ne 175 891\ne 176 177\ne 176 208\ne 176 553\ne 177 222\ne 177 306\ne 177 307\ne 177 456\ne 177 508\ne 177 996\ne 178 179\ne 178 313\ne 178 322\ne 178 520\ne 178 731\ne 179 180\ne 179 230\ne 179 613\ne 179 632\ne 179 696\ne 180 231\ne 180 307\ne 180 632\ne 180 697\ne 180 901\ne 180 1059\ne 181 182\ne 181 194\ne 181 348\ne 181 350\ne 181 356\ne 181 360\ne 181 494\ne 181 574\ne 181 575\ne 181 719\ne 181 731\ne 182 183\ne 182 326\ne 182 341\ne 182 351\ne 182 855\ne 182 1008\ne 182 1009\ne 182 1022\ne 182 1046\ne 183 184\ne 183 194\ne 183 714\ne 183 728\ne 183 775\ne 183 902\ne 183 968\ne 183 1027\ne 183 1066\ne 184 186\ne 184 193\ne 184 326\ne 184 390\ne 184 728\ne 184 770\ne 184 804\ne 184 822\ne 184 977\ne 185 186\ne 185 190\ne 185 193\ne 185 197\ne 185 262\ne 185 270\ne 185 635\ne 186 187\ne 186 188\ne 186 193\ne 186 228\ne 186 243\ne 186 637\ne 186 675\ne 186 678\ne 186 763\ne 186 770\ne 186 931\ne 187 188\ne 187 320\ne 187 456\ne 187 552\ne 187 558\ne 187 628\ne 187 1065\ne 188 189\ne 188 215\ne 188 259\ne 188 262\ne 188 263\ne 188 412\ne 188 427\ne 188 435\ne 188 559\ne 188 800\ne 188 801\ne 188 885\ne 188 948\ne 188 967\ne 188 999\ne 188 1055\ne 189 314\ne 189 343\ne 189 544\ne 189 583\ne 189 609\ne 189 736\ne 189 737\ne 189 761\ne 189 780\ne 189 810\ne 189 836\ne 189 851\ne 189 959\ne 189 987\ne 189 1064\ne 190 191\ne 190 265\ne 190 298\ne 190 331\ne 190 338\ne 190 455\ne 190 664\ne 190 699\ne 191 192\ne 191 214\ne 192 198\ne 192 248\ne 192 523\ne 192 569\ne 192 635\ne 192 664\ne 192 672\ne 192 762\ne 193 194\ne 193 263\ne 193 345\ne 193 508\ne 193 646\ne 193 727\ne 193 731\ne 193 809\ne 194 195\ne 194 221\ne 194 601\ne 194 604\ne 194 605\ne 194 673\ne 194 727\ne 194 1065\ne 195 196\ne 195 223\ne 195 263\ne 195 367\ne 195 419\ne 195 425\ne 195 456\ne 195 603\ne 195 626\ne 195 674\ne 195 872\ne 195 913\ne 195 967\ne 195 968\ne 195 1053\ne 196 220\ne 196 251\ne 196 480\ne 196 790\ne 196 857\ne 196 858\ne 196 869\ne 196 872\ne 197 198\ne 197 347\ne 197 465\ne 197 681\ne 198 199\ne 198 290\ne 198 660\ne 198 691\ne 198 981\ne 199 371\ne 199 404\ne 199 405\ne 199 614\ne 199 618\ne 199 726\ne 199 981\ne 200 201\ne 200 242\ne 200 340\ne 200 699\ne 201 202\ne 201 203\ne 201 340\ne 201 370\ne 201 378\ne 202 224\ne 202 430\ne 202 769\ne 202 838\ne 202 991\ne 202 1006\ne 202 1065\ne 203 204\ne 203 209\ne 203 274\ne 203 370\ne 203 466\ne 203 467\ne 203 702\ne 203 732\ne 204 205\ne 204 467\ne 204 709\ne 204 733\ne 204 734\ne 204 748\ne 204 985\ne 204 991\ne 204 1003\ne 204 1019\ne 205 207\ne 205 209\ne 205 249\ne 205 250\ne 205 372\ne 205 391\ne 205 846\ne 205 864\ne 205 921\ne 205 1051\ne 206 207\ne 206 308\ne 206 350\ne 206 467\ne 206 719\ne 206 723\ne 206 735\ne 207 208\ne 207 209\ne 207 366\ne 207 467\ne 208 304\ne 208 367\ne 208 480\ne 208 720\ne 208 725\ne 208 844\ne 209 210\ne 209 340\ne 209 371\ne 209 397\ne 209 399\ne 209 539\ne 209 585\ne 209 618\ne 209 650\ne 210 454\ne 210 476\ne 210 617\ne 210 644\ne 210 648\ne 210 732\ne 210 749\ne 211 212\ne 211 290\ne 211 338\ne 211 421\ne 212 213\ne 212 244\ne 212 254\ne 212 283\ne 212 312\ne 212 323\ne 212 331\ne 212 381\ne 212 384\ne 212 421\ne 212 1042\ne 213 214\ne 213 283\ne 213 416\ne 213 420\ne 213 630\ne 213 634\ne 213 667\ne 213 711\ne 213 1048\ne 214 215\ne 214 238\ne 214 255\ne 214 298\ne 214 300\ne 214 668\ne 214 781\ne 214 1050\ne 215 246\ne 215 298\ne 215 456\ne 215 559\ne 215 711\ne 215 781\ne 215 949\ne 215 995\ne 215 1044\ne 215 1049\ne 216 217\ne 216 221\ne 216 387\ne 216 424\ne 216 465\ne 216 481\ne 216 601\ne 216 602\ne 217 218\ne 217 221\ne 217 649\ne 217 681\ne 217 692\ne 217 1025\ne 218 219\ne 218 235\ne 218 424\ne 218 425\ne 218 626\ne 218 655\ne 219 220\ne 219 257\ne 219 368\ne 219 373\ne 219 571\ne 219 655\ne 219 818\ne 219 820\ne 219 824\ne 219 873\ne 219 907\ne 220 373\ne 220 388\ne 220 425\ne 220 426\ne 220 441\ne 220 514\ne 220 818\ne 220 870\ne 221 222\ne 221 316\ne 221 335\ne 221 348\ne 221 388\ne 221 425\ne 221 606\ne 221 684\ne 221 693\ne 221 739\ne 221 775\ne 221 778\ne 221 821\ne 221 950\ne 221 1025\ne 222 223\ne 222 366\ne 222 388\ne 222 426\ne 222 510\ne 222 526\ne 222 649\ne 222 652\ne 222 653\ne 222 791\ne 222 1002\ne 222 1023\ne 223 252\ne 223 328\ne 223 329\ne 223 366\ne 223 399\ne 223 454\ne 223 456\ne 223 704\ne 223 728\ne 223 743\ne 223 774\ne 223 822\ne 223 825\ne 223 920\ne 223 950\ne 223 962\ne 223 1066\ne 224 225\ne 224 242\ne 224 301\ne 224 327\ne 224 383\ne 224 432\ne 224 516\ne 224 524\ne 224 556\ne 224 567\ne 224 568\ne 224 628\ne 224 697\ne 224 765\ne 224 771\ne 224 929\ne 224 954\ne 225 226\ne 225 297\ne 225 313\ne 225 337\ne 225 929\ne 225 989\ne 225 1007\ne 225 1017\ne 225 1046\ne 226 227\ne 226 337\ne 226 461\ne 226 567\ne 226 714\ne 226 813\ne 226 854\ne 226 993\ne 226 1029\ne 226 1030\ne 227 297\ne 227 620\ne 227 674\ne 227 714\ne 227 750\ne 227 968\ne 227 988\ne 227 1000\ne 228 229\ne 228 244\ne 228 270\ne 228 272\ne 228 320\ne 228 324\ne 228 326\ne 228 402\ne 228 731\ne 228 785\ne 228 808\ne 228 889\ne 228 958\ne 228 982\ne 229 230\ne 229 271\ne 229 272\ne 229 342\ne 229 487\ne 229 582\ne 229 661\ne 229 759\ne 229 763\ne 229 798\ne 229 877\ne 229 973\ne 230 231\ne 230 271\ne 230 494\ne 230 504\ne 230 589\ne 230 641\ne 230 643\ne 230 646\ne 230 696\ne 230 730\ne 230 731\ne 230 827\ne 230 1018\ne 231 251\ne 231 360\ne 231 494\ne 231 505\ne 231 697\ne 231 759\ne 231 898\ne 231 905\ne 231 1017\ne 231 1058\ne 232 233\ne 232 493\ne 232 570\ne 232 678\ne 232 679\ne 232 691\ne 232 702\ne 232 881\ne 233 234\ne 233 545\ne 233 548\ne 233 690\ne 233 702\ne 233 708\ne 234 296\ne 234 297\ne 234 548\ne 234 550\ne 234 552\ne 234 874\ne 235 333\ne 235 624\ne 235 626\ne 235 630\ne 235 963\ne 236 237\ne 236 247\ne 236 305\ne 236 334\ne 236 428\ne 236 446\ne 236 927\ne 236 984\ne 236 985\ne 236 1016\ne 236 1031\ne 237 269\ne 237 305\ne 237 468\ne 237 720\ne 237 880\ne 237 1031\ne 237 1064\ne 238 239\ne 238 254\ne 238 298\ne 238 318\ne 238 721\ne 239 240\ne 239 639\ne 239 700\ne 239 703\ne 239 721\ne 239 1050\ne 240 241\ne 240 417\ne 240 562\ne 240 623\ne 240 665\ne 240 667\ne 241 427\ne 241 438\ne 241 795\ne 241 877\ne 241 901\ne 241 1050\ne 241 1063\ne 242 243\ne 242 301\ne 242 322\ne 242 344\ne 242 369\ne 242 381\ne 242 394\ne 242 431\ne 242 506\ne 242 515\ne 242 530\ne 242 555\ne 242 675\ne 242 699\ne 243 244\ne 243 252\ne 243 344\ne 243 558\ne 243 559\ne 243 627\ne 243 651\ne 243 771\ne 243 822\ne 244 245\ne 244 324\ne 244 338\ne 244 352\ne 244 381\ne 244 385\ne 244 447\ne 244 786\ne 244 877\ne 245 246\ne 245 298\ne 245 318\ne 245 320\ne 245 338\ne 245 355\ne 245 398\ne 245 421\ne 245 475\ne 245 558\ne 245 668\ne 245 722\ne 246 358\ne 246 408\ne 246 640\ne 246 668\ne 246 712\ne 246 741\ne 246 786\ne 246 1052\ne 247 248\ne 247 305\ne 247 330\ne 247 434\ne 248 249\ne 248 384\ne 248 511\ne 248 585\ne 248 984\ne 249 250\ne 249 266\ne 249 305\ne 249 307\ne 249 470\ne 249 581\ne 249 585\ne 249 614\ne 249 615\ne 249 633\ne 249 641\ne 249 827\ne 249 880\ne 249 897\ne 249 984\ne 249 991\ne 250 251\ne 250 351\ne 250 501\ne 250 720\ne 250 1056\ne 251 307\ne 251 540\ne 251 604\ne 251 643\ne 251 710\ne 251 805\ne 251 843\ne 251 902\ne 252 253\ne 252 454\ne 252 626\ne 252 649\ne 252 650\ne 252 678\ne 252 962\ne 253 254\ne 253 323\ne 253 328\ne 253 541\ne 253 542\ne 253 611\ne 253 636\ne 253 878\ne 254 255\ne 254 331\ne 254 415\ne 254 703\ne 254 704\ne 255 391\ne 255 416\ne 255 698\ne 255 716\ne 255 820\ne 255 878\ne 255 916\ne 255 1042\ne 255 1060\ne 256 257\ne 256 261\ne 256 347\ne 256 368\ne 256 513\ne 256 532\ne 257 258\ne 257 259\ne 257 284\ne 257 287\ne 257 293\ne 257 513\ne 257 514\ne 257 956\ne 257 957\ne 257 1037\ne 258 259\ne 258 288\ne 258 386\ne 258 412\ne 258 474\ne 258 571\ne 258 589\ne 258 892\ne 259 260\ne 259 289\ne 259 341\ne 259 347\ne 259 430\ne 259 661\ne 259 787\ne 259 828\ne 259 834\ne 259 886\ne 259 973\ne 259 1022\ne 259 1044\ne 260 289\ne 260 293\ne 260 366\ne 260 372\ne 260 448\ne 260 534\ne 260 589\ne 260 801\ne 260 835\ne 260 839\ne 260 1054\ne 260 1058\ne 261 262\ne 261 275\ne 261 608\ne 261 629\ne 261 721\ne 262 263\ne 262 298\ne 262 314\ne 262 316\ne 262 328\ne 262 347\ne 262 417\ne 262 437\ne 262 482\ne 262 626\ne 262 627\ne 262 629\ne 262 701\ne 263 264\ne 263 282\ne 263 423\ne 263 544\ne 263 809\ne 263 888\ne 264 273\ne 264 525\ne 264 598\ne 264 646\ne 264 701\ne 264 802\ne 264 807\ne 264 885\ne 264 888\ne 264 913\ne 265 266\ne 265 350\ne 265 414\ne 265 415\ne 265 585\ne 265 723\ne 266 267\ne 266 312\ne 266 351\ne 266 416\ne 266 470\ne 266 495\ne 266 658\ne 266 700\ne 266 724\ne 266 782\ne 266 783\ne 266 833\ne 266 937\ne 266 980\ne 267 268\ne 267 294\ne 267 310\ne 267 396\ne 267 411\ne 267 414\ne 267 416\ne 267 431\ne 267 433\ne 267 434\ne 267 457\ne 267 460\ne 267 516\ne 267 1060\ne 268 269\ne 268 305\ne 268 330\ne 268 334\ne 268 395\ne 268 460\ne 268 484\ne 268 688\ne 268 707\ne 268 897\ne 269 334\ne 269 433\ne 269 461\ne 269 525\ne 269 812\ne 269 813\ne 269 836\ne 269 923\ne 270 271\ne 270 380\ne 270 437\ne 270 560\ne 270 726\ne 270 731\ne 271 438\ne 271 493\ne 271 504\ne 271 561\ne 271 587\ne 271 616\ne 271 635\ne 271 691\ne 271 726\ne 272 273\ne 272 337\ne 272 474\ne 272 606\ne 272 726\ne 272 728\ne 272 730\ne 272 807\ne 272 965\ne 272 1027\ne 272 1028\ne 273 320\ne 273 468\ne 273 572\ne 273 597\ne 273 657\ne 273 722\ne 273 756\ne 273 947\ne 273 965\ne 274 275\ne 274 276\ne 274 284\ne 274 354\ne 274 570\ne 274 638\ne 274 666\ne 274 702\ne 274 748\ne 275 276\ne 275 323\ne 275 421\ne 275 545\ne 275 595\ne 275 599\ne 275 608\ne 275 648\ne 275 690\ne 275 732\ne 275 936\ne 276 277\ne 276 369\ne 276 416\ne 276 546\ne 276 600\ne 276 631\ne 276 703\ne 276 867\ne 277 329\ne 277 343\ne 277 354\ne 277 395\ne 277 743\ne 278 279\ne 278 308\ne 278 378\ne 278 397\ne 278 409\ne 278 681\ne 279 280\ne 279 305\ne 279 410\ne 279 460\ne 279 633\ne 279 681\ne 280 281\ne 280 308\ne 280 495\ne 280 684\ne 280 725\ne 281 282\ne 281 410\ne 281 411\ne 281 499\ne 282 316\ne 282 329\ne 282 360\ne 282 435\ne 282 453\ne 282 502\ne 282 654\ne 282 876\ne 283 284\ne 283 287\ne 283 311\ne 283 332\ne 283 400\ne 283 417\ne 283 459\ne 283 472\ne 283 587\ne 283 692\ne 283 1036\ne 284 285\ne 284 286\ne 284 400\ne 284 457\ne 284 458\ne 284 1005\ne 285 286\ne 285 381\ne 285 383\ne 285 666\ne 285 667\ne 285 1004\ne 286 316\ne 286 319\ne 286 332\ne 286 546\ne 286 584\ne 286 686\ne 286 841\ne 287 288\ne 287 289\ne 287 293\ne 287 333\ne 287 420\ne 287 473\ne 287 655\ne 287 738\ne 287 1039\ne 287 1040\ne 288 289\ne 288 573\ne 288 589\ne 289 735\ne 289 890\ne 289 910\ne 290 291\ne 290 371\ne 290 563\ne 290 564\ne 291 292\ne 291 371\ne 291 372\ne 291 384\ne 291 385\ne 291 476\ne 291 534\ne 291 788\ne 291 978\ne 291 981\ne 292 293\ne 292 363\ne 292 427\ne 292 476\ne 292 512\ne 292 564\ne 292 565\ne 292 640\ne 292 661\ne 292 734\ne 292 743\ne 292 823\ne 292 825\ne 292 877\ne 292 1043\ne 293 336\ne 293 400\ne 293 405\ne 293 526\ne 293 532\ne 293 533\ne 293 589\ne 293 591\ne 293 696\ne 293 1038\ne 293 1059\ne 294 295\ne 294 347\ne 294 370\ne 294 430\ne 294 431\ne 294 658\ne 294 667\ne 294 690\ne 294 713\ne 294 778\ne 294 950\ne 294 992\ne 295 296\ne 295 304\ne 295 359\ne 295 430\ne 295 712\ne 295 947\ne 296 297\ne 296 313\ne 296 422\ne 296 599\ne 296 690\ne 296 767\ne 296 947\ne 296 988\ne 297 551\ne 297 556\ne 297 708\ne 297 766\ne 297 989\ne 298 299\ne 298 455\ne 298 483\ne 298 700\ne 299 300\ne 299 375\ne 299 700\ne 299 721\ne 299 722\ne 299 729\ne 300 420\ne 300 606\ne 300 698\ne 300 721\ne 300 733\ne 300 994\ne 300 1050\ne 301 302\ne 301 374\ne 301 382\ne 301 450\ne 301 822\ne 302 303\ne 302 448\ne 302 449\ne 302 568\ne 303 304\ne 303 337\ne 303 352\ne 303 447\ne 303 707\ne 303 720\ne 303 730\ne 303 1056\ne 304 375\ne 304 486\ne 304 544\ne 304 1369\ne 305 306\ne 305 315\ne 305 349\ne 305 355\ne 305 397\ne 306 307\ne 306 506\ne 306 717\ne 306 791\ne 307 450\ne 307 507\ne 307 526\ne 307 538\ne 307 613\ne 307 621\ne 307 708\ne 307 858\ne 307 911\ne 307 953\ne 307 971\ne 307 990\ne 308 309\ne 308 378\ne 308 379\ne 308 499\ne 308 684\ne 308 745\ne 309 310\ne 309 414\ne 309 451\ne 309 452\ne 309 515\ne 309 520\ne 309 723\ne 309 778\ne 310 379\ne 310 450\ne 310 453\ne 310 513\ne 310 516\ne 310 707\ne 310 724\ne 310 739\ne 310 777\ne 310 952\ne 310 956\ne 310 1013\ne 310 1021\ne 311 312\ne 311 348\ne 311 363\ne 311 423\ne 311 439\ne 311 447\ne 311 458\ne 311 459\ne 311 461\ne 311 575\ne 311 589\ne 311 693\ne 311 711\ne 311 809\ne 311 830\ne 311 833\ne 311 890\ne 311 893\ne 311 1030\ne 311 1036\ne 311 1040\ne 312 313\ne 312 327\ne 312 352\ne 312 382\ne 312 422\ne 312 615\ne 312 711\ne 312 783\ne 312 1041\ne 312 1042\ne 313 326\ne 313 327\ne 313 1018\ne 313 1030\ne 314 315\ne 314 317\ne 314 395\ne 314 482\ne 314 542\ne 314 544\ne 314 592\ne 314 608\ne 314 617\ne 314 636\ne 314 735\ne 314 811\ne 314 890\ne 314 912\ne 315 316\ne 315 354\ne 315 395\ne 315 397\ne 315 502\ne 315 571\ne 315 581\ne 315 584\ne 315 618\ne 315 681\ne 315 682\ne 315 889\ne 315 1064\ne 316 354\ne 316 410\ne 316 434\ne 316 435\ne 316 791\ne 317 367\ne 317 444\ne 317 566\ne 317 569\ne 317 607\ne 317 626\ne 317 650\ne 317 679\ne 317 912\ne 317 987\ne 318 319\ne 318 668\ne 318 732\ne 319 320\ne 319 545\ne 319 560\ne 319 570\ne 319 616\ne 319 657\ne 319 945\ne 320 321\ne 320 356\ne 320 552\ne 320 560\ne 320 571\ne 320 574\ne 320 643\ne 320 690\ne 320 710\ne 320 820\ne 320 888\ne 320 945\ne 320 983\ne 320 1052\ne 321 325\ne 321 556\ne 321 710\ne 321 712\ne 321 805\ne 322 323\ne 322 327\ne 322 361\ne 322 518\ne 322 519\ne 322 629\ne 323 324\ne 323 327\ne 323 630\ne 323 930\ne 324 325\ne 324 361\ne 324 472\ne 325 326\ne 325 327\ne 325 341\ne 325 352\ne 325 361\ne 325 363\ne 325 809\ne 326 352\ne 326 364\ne 326 731\ne 326 785\ne 326 1046\ne 327 328\ne 327 345\ne 327 412\ne 327 512\ne 327 599\ne 327 691\ne 327 696\ne 327 799\ne 327 811\ne 327 890\ne 327 930\ne 327 1016\ne 328 329\ne 328 345\ne 328 389\ne 328 440\ne 328 464\ne 328 543\ne 328 612\ne 328 628\ne 328 800\ne 328 878\ne 329 344\ne 329 369\ne 329 376\ne 329 377\ne 329 398\ne 329 499\ne 329 794\ne 329 1062\ne 330 331\ne 330 334\ne 330 686\ne 330 706\ne 331 332\ne 331 669\ne 331 699\ne 332 333\ne 332 575\ne 332 909\ne 333 446\ne 333 488\ne 333 526\ne 333 655\ne 333 699\ne 333 749\ne 334 335\ne 334 336\ne 334 400\ne 334 517\ne 334 531\ne 334 812\ne 334 939\ne 334 940\ne 335 336\ne 335 453\ne 335 693\ne 335 1060\ne 336 337\ne 336 606\ne 336 697\ne 336 706\ne 336 707\ne 336 734\ne 336 755\ne 336 898\ne 336 956\ne 336 985\ne 336 994\ne 336 1014\ne 336 1054\ne 337 447\ne 337 568\ne 337 769\ne 337 786\ne 337 789\ne 337 790\ne 337 807\ne 337 830\ne 337 1015\ne 337 1033\ne 338 339\ne 338 358\ne 338 370\ne 338 397\ne 338 506\ne 338 664\ne 338 713\ne 339 340\ne 339 508\ne 339 625\ne 339 722\ne 339 735\ne 339 811\ne 340 353\ne 340 371\ne 340 394\ne 340 449\ne 340 450\ne 340 651\ne 340 846\ne 340 978\ne 341 342\ne 341 343\ne 341 351\ne 341 358\ne 341 386\ne 341 445\ne 341 683\ne 341 710\ne 341 736\ne 341 856\ne 341 962\ne 341 973\ne 341 1061\ne 342 343\ne 342 610\ne 342 616\ne 342 643\ne 342 661\ne 342 728\ne 342 737\ne 342 743\ne 342 804\ne 342 884\ne 342 902\ne 342 903\ne 342 934\ne 342 945\ne 343 542\ne 343 543\ne 343 631\ne 343 834\ne 343 878\ne 344 345\ne 344 360\ne 344 361\ne 344 745\ne 345 346\ne 345 385\ne 345 395\ne 345 507\ne 345 562\ne 345 604\ne 345 616\ne 345 646\ne 345 804\ne 345 805\ne 345 808\ne 345 888\ne 345 931\ne 346 671\ne 346 717\ne 346 1016\ne 347 348\ne 347 370\ne 347 376\ne 347 378\ne 347 448\ne 347 465\ne 347 472\ne 347 473\ne 347 691\ne 348 349\ne 348 402\ne 348 439\ne 348 474\ne 348 739\ne 348 1022\ne 348 1026\ne 349 465\ne 349 466\ne 349 468\ne 349 597\ne 350 351\ne 350 355\ne 351 352\ne 351 495\ne 351 720\ne 351 1044\ne 351 1066\ne 352 353\ne 352 382\ne 352 786\ne 352 822\ne 352 1056\ne 353 399\ne 353 467\ne 353 528\ne 353 565\ne 353 600\ne 353 707\ne 353 722\ne 353 734\ne 353 822\ne 354 355\ne 354 356\ne 354 357\ne 354 365\ne 354 469\ne 354 876\ne 354 1061\ne 355 356\ne 355 467\ne 355 470\ne 356 468\ne 356 471\ne 356 927\ne 356 1065\ne 357 358\ne 357 618\ne 357 702\ne 357 743\ne 357 859\ne 358 470\ne 358 628\ne 358 710\ne 358 786\ne 358 919\ne 358 920\ne 359 360\ne 359 456\ne 359 710\ne 359 713\ne 360 450\ne 360 495\ne 360 731\ne 360 958\ne 360 1046\ne 361 362\ne 361 731\ne 362 363\ne 362 459\ne 362 472\ne 362 496\ne 362 518\ne 362 564\ne 362 588\ne 362 745\ne 363 364\ne 363 385\ne 363 472\ne 363 475\ne 363 512\ne 363 589\ne 363 712\ne 363 744\ne 363 828\ne 363 829\ne 363 851\ne 363 888\ne 364 402\ne 364 496\ne 364 826\ne 364 827\ne 364 851\ne 364 1043\ne 364 1045\ne 365 366\ne 365 385\ne 365 500\ne 365 717\ne 365 791\ne 366 367\ne 366 405\ne 366 650\ne 366 735\ne 366 921\ne 366 1024\ne 367 544\ne 367 553\ne 367 572\ne 367 680\ne 367 683\ne 367 685\ne 367 987\ne 368 369\ne 368 373\ne 368 376\ne 368 497\ne 369 370\ne 369 374\ne 369 457\ne 369 481\ne 369 499\ne 369 506\ne 369 704\ne 370 371\ne 371 372\ne 371 448\ne 371 565\ne 372 551\ne 372 615\ne 372 694\ne 372 788\ne 372 846\ne 372 1056\ne 373 374\ne 373 387\ne 373 424\ne 373 440\ne 373 453\ne 373 513\ne 373 591\ne 374 375\ne 374 453\ne 374 600\ne 375 389\ne 375 426\ne 375 528\ne 375 698\ne 375 704\ne 375 996\ne 376 377\ne 376 477\ne 376 497\ne 377 378\ne 377 445\ne 377 498\ne 378 379\ne 378 396\ne 378 449\ne 378 627\ne 378 787\ne 378 838\ne 379 744\ne 379 776\ne 379 787\ne 379 1021\ne 379 1053\ne 380 381\ne 380 402\ne 380 496\ne 381 382\ne 381 383\ne 381 457\ne 381 515\ne 382 383\ne 382 495\ne 382 496\ne 383 402\ne 383 458\ne 383 516\ne 383 712\ne 383 786\ne 383 1004\ne 383 1042\ne 383 1063\ne 384 385\ne 384 512\ne 384 587\ne 384 593\ne 384 615\ne 384 634\ne 384 669\ne 385 386\ne 385 395\ne 385 447\ne 385 475\ne 385 651\ne 385 789\ne 385 827\ne 385 933\ne 385 982\ne 385 1056\ne 386 492\ne 386 494\ne 386 495\ne 386 501\ne 386 571\ne 386 634\ne 386 684\ne 386 685\ne 386 745\ne 386 854\ne 386 884\ne 386 912\ne 386 932\ne 387 388\ne 387 405\ne 387 664\ne 388 389\ne 388 391\ne 388 406\ne 388 502\ne 388 694\ne 388 920\ne 388 943\ne 389 390\ne 389 544\ne 389 591\ne 389 612\ne 389 636\ne 389 646\ne 389 680\ne 389 694\ne 389 959\ne 390 508\ne 390 628\ne 390 637\ne 390 680\ne 390 760\ne 390 853\ne 390 924\ne 391 392\ne 391 406\ne 391 441\ne 391 501\ne 391 526\ne 391 612\ne 391 622\ne 391 671\ne 391 748\ne 391 814\ne 391 817\ne 391 903\ne 392 393\ne 392 404\ne 392 476\ne 392 550\ne 392 612\ne 392 617\ne 392 657\ne 392 660\ne 392 661\ne 392 810\ne 392 820\ne 392 821\ne 392 981\ne 392 983\ne 393 436\ne 393 471\ne 393 605\ne 393 709\ne 393 762\ne 393 810\ne 393 817\ne 393 850\ne 393 868\ne 393 919\ne 393 921\ne 393 922\ne 393 924\ne 393 925\ne 393 969\ne 393 987\ne 393 1051\ne 393 1065\ne 394 395\ne 394 530\ne 394 600\ne 394 811\ne 395 396\ne 395 449\ne 395 526\ne 395 688\ne 395 836\ne 395 837\ne 395 896\ne 395 940\ne 395 941\ne 396 409\ne 396 411\ne 396 499\ne 396 570\ne 396 631\ne 396 746\ne 396 894\ne 396 1062\ne 397 398\ne 397 465\ne 397 498\ne 397 506\ne 398 399\ne 398 440\ne 398 571\ne 398 745\ne 399 440\ne 399 477\ne 399 565\ne 399 823\ne 400 401\ne 400 416\ne 400 440\ne 400 691\ne 400 814\ne 401 402\ne 401 416\ne 401 479\ne 401 686\ne 401 689\ne 401 831\ne 401 833\ne 402 438\ne 402 472\ne 402 474\ne 402 496\ne 402 511\ne 402 578\ne 402 826\ne 402 1063\ne 403 404\ne 403 557\ne 403 598\ne 403 599\ne 403 612\ne 403 642\ne 403 648\ne 404 597\ne 404 660\ne 405 406\ne 405 476\ne 405 553\ne 405 591\ne 405 614\ne 405 704\ne 406 407\ne 406 606\ne 406 698\ne 406 850\ne 406 870\ne 406 942\ne 406 1024\ne 406 1038\ne 407 408\ne 407 476\ne 407 594\ne 407 698\ne 407 733\ne 407 788\ne 407 840\ne 407 868\ne 407 920\ne 407 1020\ne 408 469\ne 408 475\ne 408 594\ne 408 789\ne 408 829\ne 408 848\ne 408 876\ne 409 410\ne 409 414\ne 409 499\ne 409 541\ne 410 411\ne 410 414\ne 410 629\ne 410 634\ne 410 687\ne 410 778\ne 411 412\ne 411 435\ne 411 688\ne 411 758\ne 411 777\ne 411 874\ne 411 889\ne 411 1016\ne 412 413\ne 412 431\ne 412 620\ne 412 629\ne 412 630\ne 412 675\ne 412 677\ne 412 689\ne 412 711\ne 412 890\ne 412 928\ne 413 464\ne 413 529\ne 413 537\ne 413 600\ne 413 622\ne 413 631\ne 413 800\ne 413 835\ne 413 866\ne 413 905\ne 413 1018\ne 414 415\ne 414 515\ne 415 416\ne 415 520\ne 415 541\ne 416 631\ne 416 690\ne 416 715\ne 416 1047\ne 417 418\ne 417 421\ne 417 423\ne 417 424\ne 417 427\ne 417 431\ne 417 434\ne 417 437\ne 417 440\ne 417 535\ne 417 564\ne 417 592\ne 417 689\ne 417 738\ne 418 419\ne 418 548\ne 418 554\ne 418 689\ne 418 711\ne 418 783\ne 419 420\ne 419 552\ne 419 553\ne 419 655\ne 419 658\ne 419 674\ne 419 704\ne 419 739\ne 419 858\ne 419 873\ne 419 996\ne 419 1000\ne 419 1001\ne 420 491\ne 420 992\ne 421 422\ne 421 520\ne 421 564\ne 421 593\ne 421 639\ne 421 640\ne 421 690\ne 422 423\ne 422 565\ne 422 599\ne 422 640\ne 423 427\ne 423 565\ne 423 590\ne 423 988\ne 424 425\ne 424 603\ne 424 738\ne 425 426\ne 425 427\ne 425 572\ne 425 825\ne 426 510\ne 426 527\ne 426 655\ne 426 908\ne 426 996\ne 426 1040\ne 427 428\ne 427 429\ne 427 432\ne 427 441\ne 427 527\ne 427 552\ne 427 583\ne 427 640\ne 427 751\ne 427 816\ne 427 999\ne 427 1036\ne 428 429\ne 428 433\ne 428 434\ne 428 435\ne 428 458\ne 428 747\ne 428 833\ne 428 861\ne 428 862\ne 428 927\ne 428 946\ne 428 1063\ne 429 430\ne 429 659\ne 429 948\ne 429 1026\ne 430 432\ne 430 433\ne 430 710\ne 430 767\ne 430 769\ne 430 777\ne 430 784\ne 430 795\ne 430 847\ne 430 872\ne 430 887\ne 431 432\ne 431 484\ne 431 555\ne 431 559\ne 431 600\ne 431 700\ne 431 711\ne 432 433\ne 432 525\ne 432 556\ne 432 803\ne 432 816\ne 432 928\ne 432 989\ne 432 1049\ne 433 458\ne 433 461\ne 433 746\ne 433 782\ne 433 952\ne 433 1032\ne 433 1047\ne 434 457\ne 434 511\ne 434 570\ne 434 739\ne 435 436\ne 435 758\ne 435 769\ne 435 775\ne 435 790\ne 435 841\ne 435 872\ne 435 948\ne 435 958\ne 435 1008\ne 435 1060\ne 435 1061\ne 435 1062\ne 435 1064\ne 435 1065\ne 436 488\ne 436 661\ne 436 734\ne 436 741\ne 436 746\ne 436 750\ne 436 753\ne 436 755\ne 436 756\ne 436 757\ne 436 759\ne 436 760\ne 436 762\ne 436 766\ne 436 768\ne 436 770\ne 436 772\ne 436 775\ne 436 781\ne 436 782\ne 436 785\ne 436 787\ne 436 788\ne 436 792\ne 436 799\ne 436 812\ne 436 818\ne 436 826\ne 436 839\ne 436 841\ne 436 848\ne 436 852\ne 436 854\ne 436 857\ne 436 860\ne 436 861\ne 436 863\ne 436 866\ne 436 868\ne 436 875\ne 436 879\ne 436 881\ne 436 885\ne 436 902\ne 436 906\ne 436 915\ne 436 963\ne 436 974\ne 436 993\ne 436 1003\ne 436 1012\ne 436 1031\ne 436 1037\ne 436 1501\ne 437 438\ne 437 439\ne 437 560\ne 438 439\ne 438 561\ne 438 701\ne 439 574\ne 439 597\ne 439 659\ne 439 726\ne 439 738\ne 439 965\ne 439 966\ne 440 441\ne 440 512\ne 440 517\ne 440 535\ne 440 537\ne 440 562\ne 440 591\ne 440 600\ne 441 442\ne 441 527\ne 441 800\ne 441 803\ne 441 814\ne 441 815\ne 441 870\ne 442 458\ne 442 512\ne 442 514\ne 442 594\ne 442 799\ne 442 806\ne 442 829\ne 442 861\ne 442 893\ne 443 444\ne 443 446\ne 443 449\ne 443 522\ne 443 569\ne 443 979\ne 444 445\ne 444 478\ne 444 647\ne 445 477\ne 445 547\ne 445 919\ne 445 934\ne 445 979\ne 445 987\ne 445 1010\ne 446 447\ne 446 569\ne 446 686\ne 446 720\ne 446 833\ne 446 1033\ne 447 586\ne 447 664\ne 447 706\ne 447 726\ne 447 791\ne 447 1016\ne 447 1059\ne 448 449\ne 448 532\ne 448 534\ne 448 573\ne 448 588\ne 449 450\ne 449 526\ne 449 683\ne 449 706\ne 449 720\ne 449 735\ne 449 979\ne 449 1054\ne 450 451\ne 450 453\ne 450 461\ne 450 495\ne 450 625\ne 450 714\ne 450 765\ne 450 845\ne 450 912\ne 450 923\ne 450 971\ne 450 980\ne 450 1032\ne 450 1066\ne 451 452\ne 451 460\ne 451 538\ne 451 673\ne 452 453\ne 452 455\ne 452 466\ne 453 456\ne 453 467\ne 453 517\ne 453 858\ne 453 943\ne 454 455\ne 454 603\ne 454 727\ne 455 456\ne 455 466\ne 455 603\ne 456 467\ne 456 558\ne 456 709\ne 456 914\ne 456 1013\ne 457 458\ne 457 459\ne 457 512\ne 457 513\ne 457 783\ne 458 514\ne 458 834\ne 458 1005\ne 458 1016\ne 458 1043\ne 458 1044\ne 459 460\ne 459 481\ne 459 588\ne 460 461\ne 460 745\ne 461 567\ne 461 744\ne 461 833\ne 461 898\ne 461 994\ne 461 1035\ne 462 463\ne 462 560\ne 462 673\ne 462 687\ne 462 727\ne 463 464\ne 463 504\ne 463 588\ne 463 629\ne 463 687\ne 464 479\ne 464 499\ne 464 541\ne 464 543\ne 464 613\ne 464 629\ne 464 631\ne 465 466\ne 465 660\ne 466 467\ne 466 549\ne 467 468\ne 467 470\ne 467 550\ne 467 722\ne 467 1044\ne 467 1053\ne 468 471\ne 468 719\ne 468 926\ne 468 943\ne 468 949\ne 468 1003\ne 468 1022\ne 469 470\ne 469 471\ne 469 475\ne 469 581\ne 469 638\ne 469 668\ne 469 828\ne 469 848\ne 469 933\ne 470 471\ne 470 517\ne 470 919\ne 471 494\ne 471 622\ne 471 826\ne 471 859\ne 471 925\ne 471 946\ne 471 1017\ne 471 1018\ne 471 1019\ne 472 473\ne 472 587\ne 472 667\ne 472 828\ne 473 474\ne 473 477\ne 473 701\ne 473 886\ne 473 992\ne 474 494\ne 474 606\ne 474 677\ne 474 892\ne 474 893\ne 474 1011\ne 475 476\ne 475 593\ne 475 983\ne 476 477\ne 476 550\ne 476 552\ne 476 563\ne 476 593\ne 476 704\ne 476 732\ne 476 919\ne 477 534\ne 477 552\ne 477 592\ne 477 650\ne 477 655\ne 477 840\ne 478 479\ne 478 542\ne 478 647\ne 479 480\ne 479 498\ne 479 553\ne 479 619\ne 479 622\ne 479 634\ne 479 686\ne 479 715\ne 480 622\ne 480 674\ne 480 688\ne 480 858\ne 480 874\ne 481 482\ne 481 483\ne 481 489\ne 481 692\ne 481 693\ne 482 483\ne 482 490\ne 482 726\ne 483 491\ne 483 652\ne 483 654\ne 484 485\ne 484 525\ne 484 530\ne 484 531\ne 484 535\ne 484 598\ne 484 600\ne 485 486\ne 485 711\ne 486 528\ne 486 557\ne 486 562\ne 486 670\ne 486 712\ne 486 805\ne 487 488\ne 487 531\ne 487 567\ne 487 582\ne 487 752\ne 487 866\ne 487 911\ne 487 939\ne 487 954\ne 488 567\ne 488 605\ne 488 697\ne 488 807\ne 488 856\ne 488 863\ne 488 901\ne 488 908\ne 488 909\ne 488 910\ne 488 913\ne 488 955\ne 488 963\ne 488 1028\ne 488 1033\ne 488 1039\ne 488 1057\ne 489 490\ne 489 491\ne 489 706\ne 490 491\ne 490 579\ne 490 609\ne 490 701\ne 491 641\ne 491 700\ne 491 718\ne 491 955\ne 491 997\ne 492 493\ne 492 570\ne 492 584\ne 492 666\ne 492 681\ne 492 932\ne 493 494\ne 493 503\ne 494 503\ne 494 605\ne 494 621\ne 494 632\ne 494 695\ne 494 719\ne 494 855\ne 494 884\ne 495 496\ne 495 725\ne 495 745\ne 495 1045\ne 495 1063\ne 496 731\ne 496 783\ne 497 498\ne 497 499\ne 497 560\ne 497 570\ne 497 571\ne 497 592\ne 498 499\ne 498 713\ne 500 501\ne 500 584\ne 500 687\ne 501 502\ne 501 584\ne 501 688\ne 501 983\ne 502 544\ne 502 653\ne 502 684\ne 502 837\ne 502 876\ne 502 1064\ne 503 504\ne 503 505\ne 503 623\ne 504 505\ne 504 588\ne 504 644\ne 504 645\ne 505 540\ne 505 665\ne 505 670\ne 505 699\ne 506 507\ne 506 508\ne 506 539\ne 506 628\ne 507 508\ne 507 523\ne 507 531\ne 507 557\ne 507 562\ne 507 647\ne 507 648\ne 507 911\ne 507 924\ne 508 509\ne 508 558\ne 508 637\ne 508 693\ne 509 510\ne 509 647\ne 509 649\ne 509 655\ne 509 679\ne 509 680\ne 509 727\ne 510 642\ne 510 680\ne 510 729\ne 511 512\ne 511 516\ne 511 547\ne 511 739\ne 511 823\ne 511 861\ne 511 899\ne 512 513\ne 512 516\ne 512 518\ne 512 562\ne 512 593\ne 512 892\ne 513 514\ne 513 739\ne 513 825\ne 514 862\ne 514 952\ne 514 1037\ne 514 1040\ne 515 516\ne 515 518\ne 515 634\ne 515 745\ne 516 517\ne 516 734\ne 516 744\ne 516 894\ne 516 897\ne 516 991\ne 516 992\ne 517 546\ne 517 552\ne 517 600\ne 517 622\ne 517 628\ne 517 815\ne 517 927\ne 518 519\ne 518 564\ne 519 520\ne 519 608\ne 520 532\ne 520 538\ne 520 593\ne 520 608\ne 520 704\ne 520 723\ne 521 522\ne 521 526\ne 521 530\ne 521 532\ne 521 535\ne 521 538\ne 521 662\ne 521 663\ne 522 523\ne 522 526\ne 522 533\ne 522 647\ne 522 649\ne 522 754\ne 523 524\ne 523 530\ne 523 531\ne 523 635\ne 524 525\ne 524 526\ne 524 530\ne 524 557\ne 524 646\ne 524 694\ne 524 897\ne 525 527\ne 525 531\ne 525 557\ne 525 803\ne 525 852\ne 526 527\ne 526 617\ne 526 618\ne 526 621\ne 526 642\ne 526 743\ne 526 754\ne 526 791\ne 526 863\ne 526 1059\ne 527 528\ne 527 535\ne 527 642\ne 527 801\ne 527 893\ne 528 529\ne 528 550\ne 528 600\ne 528 612\ne 528 640\ne 528 700\ne 528 703\ne 528 916\ne 529 610\ne 529 612\ne 529 613\ne 529 677\ne 529 767\ne 529 915\ne 529 990\ne 530 645\ne 530 664\ne 531 693\ne 531 701\ne 531 808\ne 531 852\ne 531 867\ne 531 877\ne 531 896\ne 532 533\ne 532 564\ne 532 588\ne 532 705\ne 532 706\ne 533 534\ne 533 587\ne 534 566\ne 534 568\ne 534 587\ne 534 650\ne 534 839\ne 534 979\ne 535 536\ne 536 537\ne 536 573\ne 536 613\ne 536 621\ne 536 689\ne 537 571\ne 537 622\ne 537 689\ne 537 884\ne 537 892\ne 538 539\ne 538 540\ne 538 585\ne 538 715\ne 539 540\ne 539 699\ne 539 704\ne 539 716\ne 540 575\ne 540 604\ne 540 713\ne 540 717\ne 541 542\ne 541 608\ne 541 631\ne 542 543\ne 542 616\ne 543 544\ne 543 643\ne 544 590\ne 544 837\ne 545 546\ne 545 552\ne 545 607\ne 545 647\ne 545 935\ne 546 547\ne 547 637\ne 547 900\ne 547 906\ne 548 549\ne 549 550\ne 549 563\ne 549 660\ne 549 702\ne 549 783\ne 550 551\ne 550 658\ne 550 702\ne 550 858\ne 550 1019\ne 550 1020\ne 550 1043\ne 550 1051\ne 551 810\ne 551 1012\ne 551 1013\ne 551 1041\ne 551 1051\ne 552 553\ne 552 556\ne 552 599\ne 552 642\ne 552 749\ne 552 868\ne 552 927\ne 552 935\ne 552 992\ne 553 554\ne 553 579\ne 553 607\ne 553 680\ne 553 900\ne 553 942\ne 554 577\ne 554 578\ne 554 592\ne 554 614\ne 554 740\ne 554 883\ne 554 899\ne 555 556\ne 555 558\ne 556 557\ne 556 599\ne 556 712\ne 556 914\ne 556 1032\ne 557 642\ne 557 805\ne 557 923\ne 557 924\ne 558 559\ne 559 693\ne 559 950\ne 559 1060\ne 560 561\ne 560 597\ne 560 713\ne 561 562\ne 561 592\ne 561 596\ne 561 597\ne 562 641\ne 562 667\ne 562 669\ne 562 703\ne 562 743\ne 562 745\ne 562 884\ne 563 564\ne 563 660\ne 564 565\ne 564 691\ne 564 783\ne 566 567\ne 566 569\ne 566 573\ne 566 620\ne 566 747\ne 566 833\ne 566 875\ne 566 890\ne 566 903\ne 567 568\ne 567 746\ne 567 836\ne 567 838\ne 567 890\ne 567 912\ne 567 973\ne 567 998\ne 568 651\ne 568 979\ne 568 1057\ne 569 672\ne 569 682\ne 569 683\ne 569 986\ne 570 571\ne 570 747\ne 571 572\ne 571 573\ne 571 583\ne 571 747\ne 571 819\ne 571 874\ne 571 889\ne 571 1053\ne 572 583\ne 572 685\ne 572 844\ne 573 574\ne 573 655\ne 573 674\ne 573 682\ne 574 575\ne 574 596\ne 574 643\ne 574 749\ne 575 654\ne 575 909\ne 576 577\ne 576 581\ne 576 585\ne 576 587\ne 576 592\ne 576 650\ne 577 578\ne 577 585\ne 577 618\ne 578 579\ne 578 586\ne 578 587\ne 579 580\ne 579 581\ne 579 652\ne 579 676\ne 579 683\ne 579 686\ne 579 706\ne 579 755\ne 579 1002\ne 580 652\ne 580 688\ne 580 707\ne 580 755\ne 580 777\ne 580 895\ne 580 941\ne 580 1023\ne 581 582\ne 581 583\ne 581 584\ne 581 879\ne 581 896\ne 581 911\ne 581 921\ne 581 982\ne 582 583\ne 582 587\ne 582 589\ne 582 828\ne 582 839\ne 582 889\ne 582 903\ne 582 1036\ne 583 590\ne 583 592\ne 583 594\ne 583 840\ne 583 851\ne 583 883\ne 583 907\ne 584 686\ne 585 586\ne 585 644\ne 585 664\ne 586 587\ne 586 706\ne 587 588\ne 587 589\ne 587 592\ne 587 634\ne 588 589\ne 589 590\ne 589 615\ne 589 898\ne 589 941\ne 590 591\ne 590 592\ne 591 641\ne 591 642\ne 591 738\ne 591 870\ne 591 907\ne 592 593\ne 593 594\ne 593 638\ne 593 644\ne 593 811\ne 594 638\ne 594 640\ne 594 641\ne 594 797\ne 594 810\ne 594 925\ne 594 992\ne 594 1018\ne 595 596\ne 595 598\ne 595 648\ne 595 656\ne 595 701\ne 596 613\ne 596 629\ne 596 677\ne 596 690\ne 597 598\ne 597 625\ne 597 656\ne 597 657\ne 597 726\ne 598 599\ne 598 645\ne 598 701\ne 598 749\ne 599 600\ne 599 722\ne 599 802\ne 599 811\ne 599 936\ne 600 803\ne 600 867\ne 601 602\ne 601 603\ne 602 664\ne 602 665\ne 603 626\ne 603 749\ne 604 605\ne 604 616\ne 604 902\ne 604 909\ne 604 922\ne 605 606\ne 605 617\ne 605 646\ne 605 680\ne 605 911\ne 605 912\ne 605 913\ne 605 969\ne 605 971\ne 606 609\ne 606 680\ne 606 685\ne 606 724\ne 606 729\ne 606 733\ne 606 736\ne 606 798\ne 606 850\ne 606 985\ne 606 1039\ne 607 608\ne 607 679\ne 607 900\ne 608 609\ne 608 636\ne 609 631\ne 609 892\ne 609 900\ne 609 911\ne 609 936\ne 609 959\ne 610 611\ne 610 613\ne 610 631\ne 610 695\ne 610 703\ne 610 821\ne 610 884\ne 610 915\ne 610 953\ne 610 954\ne 611 612\ne 611 636\ne 611 648\ne 611 703\ne 611 917\ne 612 648\ne 612 701\ne 612 917\ne 612 924\ne 612 934\ne 612 936\ne 613 614\ne 613 643\ne 613 648\ne 613 690\ne 614 615\ne 614 783\ne 615 670\ne 615 1056\ne 616 617\ne 616 643\ne 616 647\ne 616 691\ne 616 727\ne 617 618\ne 617 646\ne 617 657\ne 617 662\ne 617 727\ne 617 735\ne 617 737\ne 617 910\ne 618 650\ne 618 663\ne 618 864\ne 619 620\ne 619 623\ne 619 626\ne 619 629\ne 619 674\ne 620 621\ne 620 622\ne 620 672\ne 620 674\ne 620 689\ne 620 708\ne 620 750\ne 620 967\ne 620 1001\ne 621 622\ne 621 901\ne 622 831\ne 622 857\ne 622 903\ne 622 904\ne 622 939\ne 622 942\ne 622 943\ne 623 624\ne 624 625\ne 624 656\ne 624 969\ne 625 713\ne 625 735\ne 625 969\ne 626 627\ne 626 967\ne 627 628\ne 627 912\ne 627 1055\ne 628 924\ne 628 1065\ne 628 1066\ne 629 630\ne 630 631\ne 631 866\ne 631 878\ne 632 633\ne 632 634\ne 632 676\ne 632 695\ne 632 718\ne 632 740\ne 632 891\ne 632 975\ne 633 634\ne 633 725\ne 633 739\ne 633 991\ne 633 1011\ne 634 667\ne 634 675\ne 634 778\ne 635 636\ne 635 645\ne 635 646\ne 635 676\ne 635 699\ne 635 701\ne 635 763\ne 636 637\ne 636 679\ne 636 691\ne 636 704\ne 636 821\ne 636 959\ne 637 679\ne 637 760\ne 637 912\ne 637 980\ne 637 998\ne 637 999\ne 638 639\ne 638 797\ne 639 640\ne 639 667\ne 639 668\ne 639 703\ne 639 796\ne 640 700\ne 640 712\ne 640 767\ne 640 796\ne 640 936\ne 640 1042\ne 641 642\ne 641 644\ne 641 646\ne 641 700\ne 641 729\ne 641 798\ne 641 865\ne 641 871\ne 641 891\ne 641 893\ne 641 897\ne 641 992\ne 641 1059\ne 642 643\ne 642 647\ne 642 749\ne 642 869\ne 642 934\ne 643 805\ne 644 645\ne 645 646\ne 645 662\ne 645 664\ne 645 811\ne 646 763\ne 646 807\ne 646 810\ne 646 941\ne 646 977\ne 647 648\ne 647 934\ne 648 660\ne 649 650\ne 649 655\ne 649 682\ne 649 705\ne 649 1002\ne 650 651\ne 650 921\ne 651 912\ne 652 653\ne 652 687\ne 652 778\ne 653 654\ne 653 682\ne 653 683\ne 653 684\ne 655 693\ne 655 704\ne 655 908\ne 655 1010\ne 656 657\ne 656 660\ne 656 662\ne 657 658\ne 657 701\ne 657 732\ne 657 756\ne 657 966\ne 657 969\ne 658 659\ne 658 690\ne 658 700\ne 658 718\ne 658 732\ne 658 739\ne 658 784\ne 658 938\ne 658 947\ne 658 970\ne 658 1013\ne 659 689\ne 659 833\ne 660 662\ne 660 691\ne 660 811\ne 661 691\ne 661 751\ne 661 767\ne 661 799\ne 661 814\ne 661 834\ne 661 851\ne 661 881\ne 661 892\ne 661 907\ne 661 910\ne 661 959\ne 661 966\ne 661 981\ne 661 1043\ne 662 663\ne 663 664\ne 664 669\ne 664 673\ne 664 682\ne 665 666\ne 665 669\ne 666 667\ne 666 669\ne 666 1006\ne 667 668\ne 667 712\ne 667 795\ne 667 992\ne 668 733\ne 668 741\ne 668 743\ne 668 945\ne 668 946\ne 668 950\ne 669 670\ne 669 671\ne 670 671\ne 671 904\ne 671 1006\ne 672 673\ne 672 675\ne 672 714\ne 672 764\ne 673 674\ne 673 682\ne 673 713\ne 673 714\ne 673 778\ne 674 874\ne 675 676\ne 675 929\ne 676 677\ne 676 699\ne 676 821\ne 676 944\ne 677 696\ne 677 700\ne 677 701\ne 677 767\ne 677 865\ne 677 910\ne 678 679\ne 678 727\ne 678 728\ne 678 961\ne 679 680\ne 679 960\ne 680 728\ne 680 908\ne 680 934\ne 680 960\ne 680 1002\ne 681 682\ne 681 684\ne 681 838\ne 682 683\ne 683 714\ne 683 720\ne 683 838\ne 683 986\ne 683 1002\ne 683 1064\ne 684 685\ne 684 719\ne 684 776\ne 684 778\ne 684 838\ne 685 725\ne 685 736\ne 685 865\ne 685 898\ne 686 687\ne 686 688\ne 686 939\ne 687 688\ne 687 791\ne 688 720\ne 688 790\ne 688 813\ne 688 827\ne 688 895\ne 688 939\ne 689 690\ne 689 691\ne 689 751\ne 689 988\ne 690 691\ne 690 708\ne 690 767\ne 690 820\ne 691 783\ne 692 693\ne 692 998\ne 693 694\ne 693 809\ne 693 853\ne 693 941\ne 693 998\ne 693 1010\ne 694 837\ne 695 696\ne 695 697\ne 695 938\ne 695 974\ne 695 1019\ne 696 697\ne 696 910\ne 697 698\ne 697 699\ne 697 763\ne 697 914\ne 697 944\ne 697 954\ne 697 1012\ne 697 1049\ne 697 1057\ne 698 704\ne 698 774\ne 698 779\ne 698 908\ne 698 916\ne 698 959\ne 698 1000\ne 698 1021\ne 698 1043\ne 699 700\ne 699 704\ne 699 706\ne 700 701\ne 700 1049\ne 700 1050\ne 701 885\ne 701 936\ne 702 703\ne 702 1019\ne 703 704\ne 703 743\ne 703 916\ne 704 721\ne 704 783\ne 704 821\ne 704 980\ne 705 706\ne 706 707\ne 707 729\ne 707 897\ne 707 941\ne 708 709\ne 708 710\ne 708 766\ne 708 873\ne 708 970\ne 708 991\ne 708 1048\ne 709 710\ne 709 781\ne 709 820\ne 709 946\ne 709 962\ne 709 967\ne 710 713\ne 710 714\ne 710 843\ne 710 872\ne 710 958\ne 710 1029\ne 711 712\ne 711 989\ne 711 1048\ne 712 795\ne 713 717\ne 714 764\ne 714 777\ne 714 904\ne 714 912\ne 714 943\ne 715 716\ne 715 717\ne 716 717\ne 717 843\ne 717 922\ne 718 719\ne 718 723\ne 718 778\ne 719 720\ne 719 730\ne 719 735\ne 719 1009\ne 719 1028\ne 720 1033\ne 721 732\ne 722 729\ne 722 732\ne 722 733\ne 722 947\ne 723 724\ne 724 725\ne 724 971\ne 725 844\ne 725 891\ne 725 1011\ne 726 727\ne 726 730\ne 727 728\ne 727 791\ne 728 737\ne 728 790\ne 728 801\ne 728 821\ne 728 948\ne 728 961\ne 729 730\ne 729 1040\ne 730 731\ne 732 733\ne 733 734\ne 733 756\ne 733 784\ne 733 936\ne 733 950\ne 733 972\ne 733 992\ne 733 999\ne 734 744\ne 734 771\ne 734 782\ne 734 786\ne 734 816\ne 734 823\ne 734 860\ne 734 867\ne 734 916\ne 734 919\ne 734 978\ne 734 994\ne 734 1021\ne 734 1060\ne 735 736\ne 736 737\ne 736 849\ne 736 856\ne 736 1009\ne 736 1024\ne 736 1054\ne 737 756\ne 737 863\ne 737 864\ne 737 865\ne 737 977\ne 738 739\ne 738 749\ne 739 740\ne 739 824\ne 739 844\ne 739 862\ne 739 957\ne 739 1053\ne 740 821\ne 741 742\ne 741 781\ne 741 795\ne 741 796\ne 741 848\ne 742 743\ne 742 774\ne 742 859\ne 742 863\ne 742 916\ne 743 877\ne 744 745\ne 744 854\ne 744 898\ne 744 1045\ne 746 747\ne 746 787\ne 746 834\ne 746 836\ne 746 866\ne 747 748\ne 747 819\ne 747 834\ne 747 881\ne 747 903\ne 747 932\ne 747 945\ne 748 758\ne 748 797\ne 748 847\ne 748 850\ne 748 867\ne 748 922\ne 748 936\ne 748 1005\ne 748 1006\ne 748 1019\ne 748 1061\ne 749 913\ne 750 751\ne 750 764\ne 750 766\ne 750 816\ne 750 875\ne 750 928\ne 751 752\ne 751 767\ne 751 831\ne 751 928\ne 751 988\ne 752 764\ne 752 780\ne 752 838\ne 752 899\ne 752 900\ne 752 929\ne 752 939\ne 752 945\ne 752 961\ne 752 981\ne 752 1028\ne 753 754\ne 753 930\ne 753 932\ne 753 935\ne 753 937\ne 753 1025\ne 754 863\ne 754 934\ne 754 940\ne 754 953\ne 754 979\ne 754 1002\ne 755 879\ne 755 939\ne 755 942\ne 755 944\ne 755 993\ne 756 784\ne 756 885\ne 756 945\ne 756 948\ne 756 1003\ne 757 758\ne 757 796\ne 757 797\ne 757 951\ne 757 953\ne 757 955\ne 757 956\ne 758 873\ne 759 768\ne 759 902\ne 759 958\ne 759 1027\ne 760 761\ne 760 770\ne 760 906\ne 760 959\ne 760 960\ne 761 845\ne 761 854\ne 761 912\ne 761 987\ne 761 1055\ne 762 763\ne 762 764\ne 762 976\ne 762 978\ne 762 981\ne 762 984\ne 762 986\ne 763 798\ne 763 885\ne 763 931\ne 763 944\ne 763 959\ne 763 977\ne 764 765\ne 764 929\ne 764 986\ne 765 816\ne 765 817\ne 765 845\ne 765 953\ne 765 956\ne 765 958\ne 765 969\ne 765 978\ne 765 979\ne 765 1035\ne 765 1063\ne 766 767\ne 766 990\ne 766 1012\ne 767 784\ne 767 936\ne 767 1047\ne 767 1052\ne 768 769\ne 768 887\ne 768 1004\ne 768 1037\ne 769 956\ne 769 958\ne 769 1004\ne 769 1015\ne 769 1026\ne 769 1034\ne 770 771\ne 770 785\ne 770 929\ne 770 961\ne 770 1007\ne 771 786\ne 771 822\ne 771 962\ne 771 1007\ne 771 1055\ne 772 773\ne 772 787\ne 772 974\ne 772 1011\ne 773 774\ne 773 786\ne 773 799\ne 773 853\ne 773 964\ne 773 972\ne 774 794\ne 774 800\ne 774 962\ne 774 968\ne 774 1023\ne 774 1024\ne 775 776\ne 775 777\ne 775 779\ne 775 849\ne 775 853\ne 775 862\ne 775 1021\ne 775 1022\ne 775 1023\ne 775 1025\ne 775 1027\ne 776 777\ne 776 780\ne 776 838\ne 776 854\ne 776 1009\ne 777 778\ne 777 943\ne 779 780\ne 779 821\ne 779 915\ne 779 944\ne 779 959\ne 780 813\ne 780 883\ne 780 942\ne 780 1009\ne 780 1052\ne 781 993\ne 781 1029\ne 781 1048\ne 782 784\ne 782 832\ne 782 880\ne 782 937\ne 782 1007\ne 782 1041\ne 782 1043\ne 782 1045\ne 782 1047\ne 782 1049\ne 782 1051\ne 783 1043\ne 784 862\ne 784 938\ne 784 946\ne 784 947\ne 784 949\ne 784 1000\ne 784 1020\ne 784 1049\ne 785 786\ne 785 826\ne 785 1052\ne 786 789\ne 786 1029\ne 786 1042\ne 787 956\ne 787 1054\ne 787 1055\ne 788 789\ne 788 839\ne 788 972\ne 788 1012\ne 788 1057\ne 789 804\ne 789 829\ne 789 836\ne 789 854\ne 789 933\ne 789 973\ne 789 1017\ne 789 1056\ne 789 1058\ne 790 791\ne 790 801\ne 790 863\ne 790 1023\ne 792 793\ne 792 842\ne 792 882\ne 792 951\ne 792 964\ne 792 976\ne 793 794\ne 793 806\ne 793 829\ne 793 840\ne 793 849\ne 793 853\ne 793 860\ne 794 800\ne 794 1062\ne 795 796\ne 795 828\ne 795 887\ne 795 1004\ne 795 1006\ne 795 1048\ne 796 797\ne 796 916\ne 796 1050\ne 797 798\ne 797 848\ne 797 850\ne 798 856\ne 798 871\ne 798 894\ne 798 907\ne 798 918\ne 798 934\ne 798 955\ne 798 975\ne 798 984\ne 798 1050\ne 799 800\ne 799 802\ne 799 804\ne 799 806\ne 799 807\ne 799 810\ne 799 928\ne 799 930\ne 799 1015\ne 799 1041\ne 800 801\ne 800 804\ne 800 857\ne 800 878\ne 801 807\ne 802 803\ne 802 810\ne 802 847\ne 802 868\ne 802 936\ne 803 815\ne 803 847\ne 803 867\ne 804 805\ne 804 836\ne 804 902\ne 804 905\ne 804 924\ne 804 931\ne 804 954\ne 804 977\ne 805 837\ne 805 1056\ne 806 850\ne 806 867\ne 806 906\ne 806 907\ne 806 924\ne 806 951\ne 806 1014\ne 807 808\ne 807 809\ne 807 893\ne 807 910\ne 808 809\ne 808 888\ne 808 909\ne 810 811\ne 810 1018\ne 812 814\ne 812 815\ne 812 817\ne 812 852\ne 812 1031\ne 812 1035\ne 813 836\ne 813 842\ne 813 939\ne 814 831\ne 814 1005\ne 814 1036\ne 814 1038\ne 814 1047\ne 815 816\ne 815 817\ne 815 842\ne 815 868\ne 816 1029\ne 816 1035\ne 816 1055\ne 817 923\ne 817 924\ne 817 1014\ne 818 819\ne 818 857\ne 818 908\ne 818 1037\ne 819 847\ne 819 854\ne 819 882\ne 819 1052\ne 819 1064\ne 820 821\ne 820 950\ne 822 1066\ne 823 824\ne 823 881\ne 823 962\ne 824 825\ne 824 1021\ne 824 1053\ne 825 1021\ne 826 828\ne 826 831\ne 826 861\ne 826 1022\ne 827 982\ne 828 829\ne 828 860\ne 828 886\ne 828 933\ne 828 1036\ne 829 830\ne 830 832\ne 830 853\ne 830 965\ne 830 989\ne 830 1022\ne 830 1036\ne 830 1041\ne 831 832\ne 831 939\ne 831 1047\ne 832 833\ne 832 875\ne 832 1033\ne 833 1010\ne 834 835\ne 834 890\ne 835 836\ne 835 890\ne 835 903\ne 836 837\ne 836 863\ne 836 903\ne 836 940\ne 836 1054\ne 836 1064\ne 837 1056\ne 838 932\ne 838 1025\ne 838 1028\ne 838 1064\ne 839 840\ne 839 875\ne 839 921\ne 840 856\ne 840 868\ne 840 882\ne 840 886\ne 840 907\ne 840 908\ne 840 921\ne 841 842\ne 841 909\ne 841 922\ne 841 939\ne 841 945\ne 841 1004\ne 841 1005\ne 842 843\ne 842 1029\ne 842 1047\ne 842 1052\ne 843 847\ne 843 922\ne 843 1017\ne 844 862\ne 844 947\ne 844 996\ne 845 846\ne 845 952\ne 845 990\ne 845 1045\ne 845 1046\ne 845 1054\ne 846 976\ne 846 978\ne 846 1054\ne 847 1015\ne 847 1017\ne 847 1020\ne 848 849\ne 848 856\ne 848 879\ne 848 1061\ne 849 850\ne 849 1024\ne 849 1061\ne 850 1014\ne 850 1037\ne 851 854\ne 851 892\ne 851 982\ne 851 983\ne 851 1018\ne 851 1030\ne 852 853\ne 852 885\ne 852 924\ne 853 908\ne 853 998\ne 854 855\ne 854 932\ne 854 1017\ne 854 1029\ne 854 1045\ne 855 856\ne 855 882\ne 855 974\ne 855 975\ne 855 1009\ne 856 886\ne 856 918\ne 857 859\ne 857 902\ne 857 967\ne 858 873\ne 859 864\ne 859 1008\ne 859 1019\ne 859 1061\ne 860 894\ne 860 917\ne 860 918\ne 860 998\ne 860 1062\ne 861 862\ne 861 906\ne 861 925\ne 861 984\ne 862 926\ne 862 952\ne 862 957\ne 862 985\ne 862 1000\ne 862 1011\ne 862 1022\ne 863 864\ne 863 869\ne 863 895\ne 863 990\ne 863 1023\ne 863 1038\ne 863 1054\ne 864 882\ne 864 921\ne 864 976\ne 864 1003\ne 864 1008\ne 864 1028\ne 864 1064\ne 865 884\ne 865 891\ne 865 895\ne 866 867\ne 866 915\ne 866 1047\ne 867 916\ne 867 936\ne 867 1047\ne 868 869\ne 868 872\ne 868 887\ne 868 913\ne 868 935\ne 868 942\ne 868 1000\ne 868 1052\ne 869 870\ne 869 871\ne 869 913\ne 869 934\ne 870 871\ne 870 907\ne 870 1038\ne 871 880\ne 871 887\ne 871 977\ne 871 997\ne 871 1049\ne 872 873\ne 872 874\ne 872 920\ne 872 943\ne 872 1065\ne 873 874\ne 875 986\ne 875 987\ne 876 920\ne 876 1061\ne 877 1059\ne 878 917\ne 878 930\ne 878 931\ne 878 959\ne 878 962\ne 878 1062\ne 879 880\ne 879 1064\ne 880 976\ne 880 984\ne 880 990\ne 880 1011\ne 881 882\ne 881 899\ne 881 960\ne 881 961\ne 881 1019\ne 882 883\ne 882 1020\ne 883 899\ne 883 942\ne 884 892\ne 884 904\ne 884 905\ne 884 911\ne 885 886\ne 885 1049\ne 886 887\ne 886 1039\ne 887 992\ne 889 890\ne 889 1016\ne 890 910\ne 890 1030\ne 891 975\ne 892 899\ne 893 1040\ne 893 1059\ne 894 896\ne 895 896\ne 895 897\ne 895 898\ne 895 901\ne 896 897\ne 896 941\ne 898 1054\ne 899 900\ne 900 935\ne 900 942\ne 900 960\ne 900 987\ne 900 1001\ne 901 955\ne 901 1059\ne 902 904\ne 902 953\ne 904 905\ne 904 953\ne 905 954\ne 906 907\ne 907 934\ne 907 959\ne 908 1000\ne 908 1002\ne 908 1039\ne 909 1036\ne 910 966\ne 911 912\ne 913 914\ne 914 1049\ne 914 1051\ne 915 916\ne 915 917\ne 915 974\ne 916 917\ne 916 1019\ne 916 1050\ne 917 918\ne 917 959\ne 918 962\ne 918 974\ne 919 978\ne 919 980\ne 919 1051\ne 920 950\ne 921 962\ne 921 987\ne 921 1002\ne 921 1024\ne 922 1006\ne 922 1065\ne 923 1032\ne 924 934\ne 924 990\ne 925 926\ne 925 927\ne 925 984\ne 926 927\ne 926 985\ne 927 991\ne 927 1032\ne 927 1065\ne 928 929\ne 928 964\ne 928 989\ne 929 944\ne 929 1027\ne 930 931\ne 930 936\ne 930 972\ne 930 1042\ne 931 933\ne 931 940\ne 932 933\ne 932 1006\ne 932 1063\ne 933 940\ne 933 1006\ne 934 935\ne 935 936\ne 935 945\ne 935 1001\ne 937 938\ne 937 984\ne 937 1042\ne 937 1050\ne 937 1063\ne 938 957\ne 938 1001\ne 938 1019\ne 938 1050\ne 939 940\ne 939 1033\ne 940 979\ne 942 943\ne 942 1000\ne 944 975\ne 944 1027\ne 945 946\ne 945 948\ne 945 1052\ne 946 949\ne 947 996\ne 948 949\ne 948 950\ne 949 950\ne 950 1060\ne 951 952\ne 951 990\ne 951 997\ne 951 1038\ne 951 1047\ne 952 956\ne 953 954\ne 953 984\ne 953 990\ne 955 997\ne 955 1050\ne 956 957\ne 956 1014\ne 957 1001\ne 957 1025\ne 957 1026\ne 958 1027\ne 958 1029\ne 958 1046\ne 958 1063\ne 959 960\ne 960 961\ne 960 987\ne 961 962\ne 961 1008\ne 962 967\ne 962 972\ne 962 1002\ne 962 1008\ne 962 1062\ne 963 964\ne 963 966\ne 963 967\ne 963 969\ne 963 972\ne 963 1034\ne 963 1035\ne 964 965\ne 964 968\ne 964 1027\ne 964 1029\ne 965 966\ne 965 1022\ne 966 1026\ne 966 1036\ne 967 968\ne 967 987\ne 967 1001\ne 967 1055\ne 968 1000\ne 969 970\ne 970 971\ne 970 1013\ne 970 1051\ne 971 1051\ne 972 973\ne 972 998\ne 972 1057\ne 973 1058\ne 974 975\ne 974 1008\ne 975 1011\ne 976 977\ne 976 1007\ne 976 1027\ne 978 979\ne 979 986\ne 979 1033\ne 979 1054\ne 980 994\ne 980 995\ne 980 1010\ne 980 1066\ne 981 982\ne 982 983\ne 984 985\ne 986 987\ne 986 1033\ne 988 1030\ne 989 1041\ne 989 1048\ne 991 992\ne 993 994\ne 993 998\ne 993 1000\ne 994 995\ne 994 998\ne 995 999\ne 995 1000\ne 996 1000\ne 997 1049\ne 998 999\ne 998 1025\ne 998 1036\ne 999 1000\ne 1000 1001\ne 1002 1023\ne 1002 1025\ne 1003 1028\ne 1003 1031\ne 1004 1005\ne 1004 1006\ne 1005 1036\ne 1005 1037\ne 1006 1017\ne 1007 1008\ne 1007 1046\ne 1009 1028\ne 1011 1027\ne 1011 1063\ne 1012 1014\ne 1012 1017\ne 1012 1019\ne 1014 1015\ne 1015 1016\ne 1015 1031\ne 1017 1018\ne 1019 1020\ne 1021 1044\ne 1021 1060\ne 1021 1066\ne 1022 1026\ne 1022 1044\ne 1023 1024\ne 1025 1026\ne 1026 1034\ne 1026 1036\ne 1027 1028\ne 1028 1033\ne 1029 1048\ne 1029 1055\ne 1031 1033\ne 1031 1034\ne 1032 1051\ne 1033 1034\ne 1034 1035\ne 1035 1036\ne 1036 1039\ne 1036 1042\ne 1036 1048\ne 1037 1038\ne 1037 1039\ne 1038 1039\ne 1039 1040\ne 1041 1042\ne 1041 1043\ne 1042 1048\ne 1043 1044\ne 1045 1046\ne 1045 1063\ne 1047 1048\ne 1049 1050\ne 1057 1058\ne 1057 1059\ne 1058 1059\ne 1060 1062\ne 1061 1064\ne 1067 1068\ne 1067 1145\ne 1067 1176\ne 1067 1179\ne 1067 1183\ne 1067 1303\ne 1067 1548\ne 1068 1069\ne 1068 1134\ne 1068 1326\ne 1068 1339\ne 1068 1381\ne 1068 1475\ne 1068 1823\ne 1069 1070\ne 1069 1092\ne 1069 1237\ne 1069 1346\ne 1069 1347\ne 1069 1371\ne 1069 1434\ne 1069 1481\ne 1069 1548\ne 1069 1823\ne 1070 1071\ne 1070 1114\ne 1070 1134\ne 1070 1142\ne 1070 1147\ne 1070 1235\ne 1070 1245\ne 1070 1358\ne 1070 1372\ne 1070 1375\ne 1070 1378\ne 1070 1481\ne 1070 1485\ne 1070 1556\ne 1070 1585\ne 1070 1659\ne 1070 1674\ne 1070 1705\ne 1070 1763\ne 1070 1789\ne 1070 1805\ne 1070 1822\ne 1070 1923\ne 1070 2016\ne 1071 1075\ne 1071 1079\ne 1071 1083\ne 1071 1096\ne 1071 1100\ne 1071 1104\ne 1071 1108\ne 1071 1114\ne 1071 1118\ne 1071 1121\ne 1071 1125\ne 1071 1138\ne 1071 1148\ne 1071 1152\ne 1071 1175\ne 1071 1182\ne 1071 1186\ne 1071 1197\ne 1071 1207\ne 1071 1215\ne 1071 1229\ne 1071 1236\ne 1071 1249\ne 1071 1254\ne 1071 1261\ne 1071 1280\ne 1071 1285\ne 1071 1292\ne 1071 1296\ne 1071 1302\ne 1071 1311\ne 1071 1316\ne 1071 1325\ne 1071 1329\ne 1071 1334\ne 1071 1338\ne 1071 1347\ne 1071 1362\ne 1071 1369\ne 1071 1391\ne 1071 1402\ne 1071 1418\ne 1071 1429\ne 1071 1437\ne 1071 1444\ne 1071 1455\ne 1071 1473\ne 1071 1478\ne 1071 1495\ne 1071 1501\ne 1071 1533\ne 1071 1579\ne 1071 1590\ne 1071 1594\ne 1071 1616\ne 1071 1645\ne 1071 1763\ne 1071 1839\ne 1071 1845\ne 1071 1857\ne 1071 1895\ne 1071 1910\ne 1071 1912\ne 1071 1936\ne 1072 1073\ne 1072 1094\ne 1072 1098\ne 1072 1130\ne 1072 1149\ne 1072 1158\ne 1072 1217\ne 1072 1233\ne 1072 1237\ne 1072 1240\ne 1072 1243\ne 1072 1246\ne 1072 1330\ne 1072 1344\ne 1072 1475\ne 1072 1483\ne 1072 1613\ne 1072 1642\ne 1072 1661\ne 1072 1697\ne 1072 1783\ne 1073 1074\ne 1073 1181\ne 1073 1246\ne 1073 1317\ne 1073 1381\ne 1073 1422\ne 1073 1683\ne 1073 1743\ne 1073 1760\ne 1073 2073\ne 1074 1075\ne 1074 1228\ne 1074 1240\ne 1074 1288\ne 1074 1347\ne 1074 1793\ne 1074 2014\ne 1074 2073\ne 1075 1137\ne 1075 1161\ne 1075 1175\ne 1075 1594\ne 1075 1760\ne 1075 1802\ne 1075 1849\ne 1075 1956\ne 1075 2039\ne 1075 2085\ne 1076 1077\ne 1076 1150\ne 1076 1160\ne 1076 1174\ne 1076 1265\ne 1076 1268\ne 1076 1271\ne 1076 1274\ne 1076 1343\ne 1077 1078\ne 1077 1226\ne 1077 1227\ne 1077 1265\ne 1077 1387\ne 1077 1435\ne 1077 1436\ne 1077 1519\ne 1077 1546\ne 1077 1791\ne 1078 1079\ne 1078 1102\ne 1078 1152\ne 1078 1184\ne 1078 1300\ne 1078 1305\ne 1078 1344\ne 1078 1359\ne 1078 1489\ne 1078 1525\ne 1078 1661\ne 1078 1667\ne 1078 1668\ne 1078 1694\ne 1078 1786\ne 1079 1081\ne 1079 1085\ne 1079 1089\ne 1079 1090\ne 1079 1092\ne 1079 1103\ne 1079 1106\ne 1079 1108\ne 1079 1112\ne 1079 1117\ne 1079 1120\ne 1079 1131\ne 1079 1153\ne 1079 1161\ne 1079 1165\ne 1079 1166\ne 1079 1185\ne 1079 1230\ne 1079 1240\ne 1079 1258\ne 1079 1271\ne 1079 1288\ne 1079 1310\ne 1079 1333\ne 1079 1359\ne 1079 1363\ne 1079 1366\ne 1079 1369\ne 1079 1370\ne 1079 1373\ne 1079 1376\ne 1079 1379\ne 1079 1414\ne 1079 1436\ne 1079 1438\ne 1079 1488\ne 1079 1512\ne 1079 1513\ne 1079 1515\ne 1079 1529\ne 1079 1540\ne 1079 1548\ne 1079 1549\ne 1079 1561\ne 1079 1564\ne 1079 1570\ne 1079 1573\ne 1079 1578\ne 1079 1585\ne 1079 1605\ne 1079 1627\ne 1079 1638\ne 1079 1662\ne 1079 1663\ne 1079 1678\ne 1079 1706\ne 1079 1717\ne 1079 1729\ne 1079 1739\ne 1079 1769\ne 1079 1794\ne 1079 1796\ne 1080 1081\ne 1080 1101\ne 1080 1139\ne 1080 1179\ne 1080 1246\ne 1080 1317\ne 1080 1403\ne 1080 1406\ne 1080 1409\ne 1080 1412\ne 1080 1415\ne 1080 1419\ne 1080 1426\ne 1080 1430\ne 1080 1441\ne 1080 1509\ne 1080 1607\ne 1080 1681\ne 1080 1747\ne 1080 1778\ne 1080 1800\ne 1081 1082\ne 1081 1140\ne 1081 1288\ne 1081 1406\ne 1081 1424\ne 1081 1608\ne 1081 1708\ne 1081 1718\ne 1081 1719\ne 1081 1941\ne 1082 1083\ne 1082 1324\ne 1082 1328\ne 1082 1360\ne 1082 1412\ne 1082 1428\ne 1082 1444\ne 1082 1578\ne 1082 1890\ne 1082 2118\ne 1083 1086\ne 1083 1121\ne 1083 1264\ne 1083 1295\ne 1083 1361\ne 1083 1454\ne 1083 1469\ne 1083 1504\ne 1083 1630\ne 1083 1656\ne 1083 1708\ne 1083 1726\ne 1083 1756\ne 1083 2053\ne 1084 1085\ne 1084 1201\ne 1084 1274\ne 1084 1319\ne 1084 1339\ne 1084 1430\ne 1084 1449\ne 1084 1452\ne 1084 1456\ne 1084 1459\ne 1084 1462\ne 1084 1465\ne 1084 1470\ne 1084 1505\ne 1084 1544\ne 1084 1565\ne 1084 1586\ne 1084 1713\ne 1084 1725\ne 1084 1734\ne 1084 1780\ne 1085 1086\ne 1085 1092\ne 1085 1272\ne 1085 1456\ne 1085 1468\ne 1085 1469\ne 1085 1545\ne 1085 1680\ne 1085 1735\ne 1085 1988\ne 1086 1334\ne 1086 1376\ne 1086 1465\ne 1086 1523\ne 1086 1759\ne 1086 1879\ne 1086 1900\ne 1087 1088\ne 1087 1119\ne 1087 1122\ne 1087 1217\ne 1087 1355\ne 1087 1508\ne 1087 1513\ne 1087 1599\ne 1088 1089\ne 1088 1123\ne 1088 1262\ne 1088 1307\ne 1088 1367\ne 1088 1508\ne 1088 1512\ne 1088 1633\ne 1089 1119\ne 1089 1138\ne 1089 1144\ne 1089 1275\ne 1089 1379\ne 1089 1426\ne 1089 1427\ne 1089 1462\ne 1089 1509\ne 1089 1530\ne 1089 1546\ne 1089 1584\ne 1089 1620\ne 1089 1713\ne 1090 1091\ne 1090 1516\ne 1090 1519\ne 1090 1524\ne 1090 1527\ne 1090 1709\ne 1091 1092\ne 1091 1339\ne 1091 1522\ne 1091 1531\ne 1091 1562\ne 1091 1614\ne 1091 1658\ne 1092 1169\ne 1092 1238\ne 1092 1339\ne 1092 1359\ne 1092 1523\ne 1092 1532\ne 1092 1615\ne 1092 1636\ne 1092 1659\ne 1092 1664\ne 1092 1665\ne 1092 1782\ne 1092 1813\ne 1092 1912\ne 1092 1939\ne 1092 1941\ne 1092 1985\ne 1092 1992\ne 1092 2081\ne 1092 2097\ne 1093 1094\ne 1093 1383\ne 1093 1530\ne 1093 1534\ne 1093 1537\ne 1093 1540\ne 1093 1543\ne 1093 1641\ne 1094 1095\ne 1094 1159\ne 1094 1238\ne 1094 1414\ne 1094 1422\ne 1094 1467\ne 1094 1536\ne 1094 1544\ne 1094 1568\ne 1094 1626\ne 1094 1627\ne 1094 1642\ne 1094 1688\ne 1094 1767\ne 1095 1096\ne 1095 1240\ne 1095 1429\ne 1095 1536\ne 1095 1540\ne 1095 1545\ne 1095 1551\ne 1095 1615\ne 1095 1892\ne 1095 2048\ne 1096 1128\ne 1096 1159\ne 1096 1169\ne 1096 1182\ne 1096 1242\ne 1096 1431\ne 1096 1432\ne 1096 1458\ne 1096 1469\ne 1096 1521\ne 1096 1622\ne 1096 1690\ne 1096 1782\ne 1096 1988\ne 1097 1098\ne 1097 1155\ne 1097 1341\ne 1097 1395\ne 1097 1546\ne 1097 1549\ne 1097 1554\ne 1097 1596\ne 1097 1660\ne 1098 1099\ne 1098 1155\ne 1098 1278\ne 1098 1481\ne 1098 1525\ne 1098 1548\ne 1098 1550\ne 1098 1556\ne 1098 1603\ne 1098 1627\ne 1098 1640\ne 1098 1651\ne 1098 1661\ne 1099 1100\ne 1099 1206\ne 1099 1229\ne 1099 1233\ne 1099 1372\ne 1099 1526\ne 1099 1550\ne 1099 1552\ne 1099 1596\ne 1099 1742\ne 1099 1960\ne 1099 2062\ne 1099 2112\ne 1099 2113\ne 1100 1186\ne 1100 1306\ne 1100 1506\ne 1100 1507\ne 1100 1536\ne 1100 1551\ne 1100 1627\ne 1100 1736\ne 1100 1807\ne 1100 1809\ne 1100 1860\ne 1100 1869\ne 1100 1936\ne 1100 1960\ne 1100 1981\ne 1100 1989\ne 1101 1102\ne 1101 1183\ne 1101 1184\ne 1101 1451\ne 1101 1557\ne 1101 1562\ne 1101 1565\ne 1101 1568\ne 1102 1103\ne 1102 1167\ne 1102 1353\ne 1102 1750\ne 1102 1800\ne 1103 1104\ne 1103 1431\ne 1103 1470\ne 1103 1540\ne 1103 1565\ne 1103 1641\ne 1103 1646\ne 1103 1654\ne 1103 1655\ne 1103 1717\ne 1104 1471\ne 1104 1473\ne 1104 1566\ne 1104 1567\ne 1104 1645\ne 1104 1646\ne 1104 1750\ne 1104 1944\ne 1104 2089\ne 1105 1106\ne 1105 1107\ne 1105 1126\ne 1105 1433\ne 1105 1571\ne 1105 1574\ne 1105 1712\ne 1106 1107\ne 1106 1261\ne 1106 1438\ne 1106 1575\ne 1106 1605\ne 1106 1668\ne 1106 1707\ne 1106 1856\ne 1107 1108\ne 1107 1261\ne 1107 1284\ne 1107 1393\ne 1107 1422\ne 1107 1442\ne 1107 1544\ne 1107 1669\ne 1107 1691\ne 1107 1693\ne 1107 1745\ne 1107 1781\ne 1107 1922\ne 1107 1999\ne 1108 1111\ne 1108 1134\ne 1108 1144\ne 1108 1159\ne 1108 1171\ne 1108 1174\ne 1108 1176\ne 1108 1200\ne 1108 1209\ne 1108 1223\ne 1108 1233\ne 1108 1239\ne 1108 1251\ne 1108 1257\ne 1108 1279\ne 1108 1284\ne 1108 1286\ne 1108 1289\ne 1108 1293\ne 1108 1297\ne 1108 1300\ne 1108 1301\ne 1108 1322\ne 1108 1331\ne 1108 1351\ne 1108 1356\ne 1108 1381\ne 1108 1392\ne 1108 1398\ne 1108 1399\ne 1108 1406\ne 1108 1443\ne 1108 1451\ne 1108 1456\ne 1108 1461\ne 1108 1467\ne 1108 1492\ne 1108 1501\ne 1108 1534\ne 1108 1576\ne 1108 1591\ne 1108 1596\ne 1108 1599\ne 1108 1612\ne 1108 1617\ne 1108 1631\ne 1108 1644\ne 1108 1646\ne 1108 1669\ne 1108 1671\ne 1108 1675\ne 1108 1685\ne 1108 1696\ne 1108 1699\ne 1108 1702\ne 1108 1722\ne 1108 1733\ne 1108 1756\ne 1108 1760\ne 1108 1766\ne 1108 1773\ne 1108 1813\ne 1108 1817\ne 1108 1818\ne 1108 1830\ne 1108 1863\ne 1108 2027\ne 1108 2101\ne 1109 1110\ne 1109 1112\ne 1109 1129\ne 1109 1445\ne 1109 1576\ne 1109 1580\ne 1109 1583\ne 1110 1111\ne 1110 1297\ne 1110 1317\ne 1110 1464\ne 1110 1629\ne 1110 1770\ne 1110 1888\ne 1111 1144\ne 1111 1204\ne 1111 1268\ne 1111 1308\ne 1111 1309\ne 1111 1331\ne 1111 1341\ne 1111 1418\ne 1111 1580\ne 1111 1629\ne 1111 1768\ne 1111 1771\ne 1111 1797\ne 1111 1799\ne 1111 1810\ne 1112 1113\ne 1112 1117\ne 1112 1207\ne 1112 1464\ne 1112 1561\ne 1112 1576\ne 1112 1619\ne 1112 1650\ne 1112 1651\ne 1112 1657\ne 1113 1114\ne 1113 1232\ne 1113 1366\ne 1113 1418\ne 1113 1447\ne 1113 1580\ne 1113 1581\ne 1114 1154\ne 1114 1225\ne 1114 1232\ne 1114 1346\ne 1114 1545\ne 1114 1619\ne 1114 1654\ne 1114 1680\ne 1114 1699\ne 1114 1776\ne 1114 1777\ne 1114 1790\ne 1114 1956\ne 1115 1116\ne 1115 1119\ne 1115 1482\ne 1115 1610\ne 1115 1613\ne 1115 1617\ne 1115 1620\ne 1115 1625\ne 1115 1628\ne 1116 1117\ne 1116 1185\ne 1116 1395\ne 1116 1517\ne 1116 1543\ne 1116 1544\ne 1116 1571\ne 1116 1580\ne 1116 1582\ne 1116 1611\ne 1117 1118\ne 1117 1129\ne 1117 1241\ne 1117 1517\ne 1117 1573\ne 1117 1612\ne 1117 1618\ne 1117 1771\ne 1118 1203\ne 1118 1207\ne 1118 1401\ne 1118 1455\ne 1118 1518\ne 1118 1582\ne 1118 1612\ne 1118 1656\ne 1118 1971\ne 1118 1979\ne 1118 1991\ne 1118 2007\ne 1118 2060\ne 1119 1120\ne 1119 1509\ne 1119 1542\ne 1119 1628\ne 1120 1121\ne 1120 1131\ne 1120 1170\ne 1120 1217\ne 1120 1219\ne 1120 1297\ne 1120 1562\ne 1120 1568\ne 1120 1574\ne 1120 1600\ne 1120 1613\ne 1120 1614\ne 1120 1728\ne 1120 1792\ne 1121 1218\ne 1121 1236\ne 1121 1297\ne 1121 1299\ne 1121 1464\ne 1121 1494\ne 1121 1539\ne 1121 1542\ne 1121 1559\ne 1121 1592\ne 1121 1602\ne 1121 1615\ne 1121 1619\ne 1121 1636\ne 1121 1683\ne 1121 1724\ne 1121 1745\ne 1121 1793\ne 1121 1805\ne 1121 1946\ne 1121 1947\ne 1121 2118\ne 1122 1123\ne 1122 1631\ne 1122 1635\ne 1122 1638\ne 1122 1684\ne 1123 1124\ne 1123 1155\ne 1123 1177\ne 1123 1307\ne 1123 1398\ne 1123 1460\ne 1123 1461\ne 1123 1525\ne 1123 1632\ne 1123 1746\ne 1123 1757\ne 1124 1125\ne 1124 1156\ne 1124 1197\ne 1124 1231\ne 1124 1366\ne 1124 1367\ne 1124 1632\ne 1124 1638\ne 1124 1749\ne 1124 1758\ne 1124 1902\ne 1125 1128\ne 1125 1140\ne 1125 1152\ne 1125 1156\ne 1125 1162\ne 1125 1398\ne 1125 1491\ne 1125 1553\ne 1125 1556\ne 1125 1640\ne 1125 1719\ne 1125 1784\ne 1125 1785\ne 1125 2105\ne 1126 1127\ne 1126 1397\ne 1126 1605\ne 1126 1666\ne 1126 1669\ne 1126 1730\ne 1127 1128\ne 1127 1159\ne 1127 1167\ne 1127 1398\ne 1127 1568\ne 1127 1574\ne 1127 1666\ne 1127 1670\ne 1127 1689\ne 1127 1710\ne 1127 1727\ne 1127 1814\ne 1128 1575\ne 1128 1605\ne 1128 1670\ne 1128 1690\ne 1128 1794\ne 1129 1130\ne 1129 1157\ne 1129 1509\ne 1129 1584\ne 1129 1612\ne 1129 1672\ne 1130 1131\ne 1130 1158\ne 1130 1235\ne 1130 1241\ne 1130 1585\ne 1130 1672\ne 1130 1770\ne 1131 1236\ne 1131 1246\ne 1131 1259\ne 1131 1273\ne 1131 1274\ne 1131 1382\ne 1131 1509\ne 1131 1511\ne 1131 1515\ne 1131 1544\ne 1131 1563\ne 1131 1698\ne 1131 1804\ne 1131 1814\ne 1131 1898\ne 1131 1992\ne 1132 1133\ne 1132 1160\ne 1132 1606\ne 1132 1675\ne 1132 1678\ne 1132 1681\ne 1133 1134\ne 1133 1158\ne 1133 1187\ne 1133 1313\ne 1133 1372\ne 1133 1587\ne 1133 1603\ne 1133 1669\ne 1133 2018\ne 1134 1173\ne 1134 1188\ne 1134 1585\ne 1134 1598\ne 1134 1699\ne 1134 1703\ne 1134 1704\ne 1134 1822\ne 1135 1136\ne 1135 1139\ne 1135 1141\ne 1135 1143\ne 1135 1167\ne 1135 1304\ne 1135 1313\ne 1135 1336\ne 1135 1697\ne 1135 1700\ne 1135 1703\ne 1135 1706\ne 1135 1709\ne 1135 1712\ne 1135 1863\ne 1136 1137\ne 1136 1139\ne 1136 1144\ne 1136 1275\ne 1136 1317\ne 1136 1676\ne 1136 1760\ne 1136 1983\ne 1137 1138\ne 1137 1140\ne 1137 1270\ne 1137 1275\ne 1137 1288\ne 1137 1472\ne 1137 1677\ne 1137 1706\ne 1137 1798\ne 1137 1802\ne 1137 1808\ne 1137 1978\ne 1137 1979\ne 1137 1983\ne 1137 2122\ne 1138 1142\ne 1138 1144\ne 1138 1190\ne 1138 1211\ne 1138 1232\ne 1138 1254\ne 1138 1342\ne 1138 1390\ne 1138 1394\ne 1138 1418\ne 1138 1428\ne 1138 1510\ne 1138 1542\ne 1138 1621\ne 1138 1624\ne 1138 1633\ne 1138 1677\ne 1138 1716\ne 1138 1758\ne 1138 1858\ne 1138 1925\ne 1138 2006\ne 1139 1140\ne 1139 1398\ne 1139 1534\ne 1139 1538\ne 1139 1542\ne 1139 1559\ne 1139 1800\ne 1139 1921\ne 1140 1315\ne 1140 1473\ne 1140 1706\ne 1140 1921\ne 1140 2123\ne 1141 1142\ne 1141 1433\ne 1141 1542\ne 1141 1598\ne 1141 1612\ne 1141 1656\ne 1141 1657\ne 1141 1701\ne 1141 1712\ne 1141 1756\ne 1141 1805\ne 1141 1972\ne 1142 1168\ne 1142 1189\ne 1142 1232\ne 1142 1341\ne 1142 1392\ne 1142 1434\ne 1142 1554\ne 1142 1572\ne 1142 1577\ne 1142 1584\ne 1142 1612\ne 1142 1674\ne 1142 1871\ne 1143 1144\ne 1143 1232\ne 1143 1474\ne 1143 1543\ne 1143 1580\ne 1143 1699\ne 1143 1959\ne 1144 1210\ne 1144 1389\ne 1144 1537\ne 1144 1676\ne 1144 1757\ne 1144 1925\ne 1145 1146\ne 1145 1641\ne 1145 1644\ne 1145 1714\ne 1145 1717\ne 1145 1747\ne 1146 1147\ne 1146 1170\ne 1146 1231\ne 1146 1482\ne 1146 1547\ne 1146 1585\ne 1146 1597\ne 1146 1638\ne 1146 1657\ne 1146 1661\ne 1146 1720\ne 1147 1148\ne 1147 1358\ne 1147 1466\ne 1147 1492\ne 1147 1539\ne 1147 1618\ne 1147 1644\ne 1147 1648\ne 1147 1686\ne 1147 1742\ne 1147 1966\ne 1147 1973\ne 1147 2095\ne 1148 1292\ne 1148 1325\ne 1148 1432\ne 1148 1631\ne 1148 1638\ne 1148 1748\ne 1148 1884\ne 1148 1940\ne 1148 1973\ne 1149 1150\ne 1149 1153\ne 1149 1155\ne 1149 1263\ne 1149 1384\ne 1149 1527\ne 1149 1672\ne 1149 1737\ne 1149 1740\ne 1149 1743\ne 1149 1746\ne 1149 1751\ne 1149 1754\ne 1149 1817\ne 1150 1151\ne 1150 1209\ne 1150 1398\ne 1150 1511\ne 1150 1558\ne 1150 1683\ne 1150 1724\ne 1150 1746\ne 1150 1784\ne 1150 1791\ne 1150 2093\ne 1151 1152\ne 1151 1182\ne 1151 1199\ne 1151 1200\ne 1151 1259\ne 1151 1265\ne 1151 1267\ne 1151 1286\ne 1151 1667\ne 1151 1690\ne 1151 1698\ne 1151 1740\ne 1151 1741\ne 1151 1791\ne 1151 2092\ne 1152 1178\ne 1152 1193\ne 1152 1227\ne 1152 1260\ne 1152 1300\ne 1152 1306\ne 1152 1354\ne 1152 1365\ne 1152 1477\ne 1152 1485\ne 1152 1490\ne 1152 1495\ne 1152 1504\ne 1152 1526\ne 1152 1690\ne 1152 1724\ne 1152 1742\ne 1152 1750\ne 1152 2028\ne 1152 2029\ne 1152 2078\ne 1153 1154\ne 1153 1156\ne 1153 1240\ne 1153 1264\ne 1153 1379\ne 1153 1385\ne 1153 1527\ne 1153 1618\ne 1153 1619\ne 1153 1749\ne 1153 1753\ne 1153 1779\ne 1153 1784\ne 1153 1793\ne 1153 1817\ne 1153 1845\ne 1153 1886\ne 1153 1930\ne 1153 1949\ne 1153 2048\ne 1153 2053\ne 1153 2095\ne 1154 1249\ne 1154 1366\ne 1154 1386\ne 1154 1740\ne 1154 1779\ne 1154 1970\ne 1154 1994\ne 1155 1156\ne 1155 1336\ne 1155 1398\ne 1155 1528\ne 1155 1547\ne 1155 1552\ne 1155 1555\ne 1155 1652\ne 1155 1695\ne 1155 1696\ne 1155 1751\ne 1156 1295\ne 1156 1477\ne 1156 1478\ne 1156 1528\ne 1156 1549\ne 1156 1550\ne 1156 1552\ne 1156 1601\ne 1156 1654\ne 1156 1686\ne 1156 1753\ne 1156 1866\ne 1156 1953\ne 1157 1158\ne 1157 1250\ne 1157 1573\ne 1157 1702\ne 1157 1757\ne 1158 1159\ne 1158 1164\ne 1158 1176\ne 1158 1234\ne 1158 1242\ne 1158 1259\ne 1158 1484\ne 1158 1714\ne 1158 1738\ne 1159 1164\ne 1159 1168\ne 1159 1201\ne 1159 1257\ne 1159 1382\ne 1159 1458\ne 1159 1572\ne 1159 1617\ne 1159 1689\ne 1159 1715\ne 1159 1725\ne 1159 1876\ne 1160 1161\ne 1160 1568\ne 1160 1760\ne 1160 1764\ne 1160 1767\ne 1160 1770\ne 1161 1162\ne 1161 1271\ne 1161 1275\ne 1161 1615\ne 1161 1678\ne 1161 1682\ne 1161 1723\ne 1161 1760\ne 1161 2036\ne 1162 1366\ne 1162 1440\ne 1162 1616\ne 1162 1711\ne 1162 1762\ne 1162 1764\ne 1162 1772\ne 1162 2036\ne 1162 2078\ne 1163 1164\ne 1163 1165\ne 1163 1276\ne 1163 1278\ne 1163 1603\ne 1163 1684\ne 1163 1773\ne 1163 1778\ne 1163 1780\ne 1164 1194\ne 1164 1256\ne 1164 1268\ne 1164 1279\ne 1164 1317\ne 1164 1520\ne 1164 1521\ne 1164 1604\ne 1164 1691\ne 1164 1774\ne 1164 1778\ne 1164 1781\ne 1165 1166\ne 1165 1237\ne 1165 1362\ne 1165 1424\ne 1165 1521\ne 1165 1613\ne 1165 1620\ne 1165 1739\ne 1165 1773\ne 1165 1776\ne 1166 1239\ne 1166 1276\ne 1166 1377\ne 1166 1436\ne 1166 1487\ne 1166 1614\ne 1166 1616\ne 1166 1876\ne 1167 1168\ne 1167 1170\ne 1167 1281\ne 1167 1352\ne 1167 1470\ne 1167 1554\ne 1167 1574\ne 1167 1671\ne 1167 1673\ne 1167 1771\ne 1167 1783\ne 1167 1786\ne 1167 1788\ne 1167 1791\ne 1167 1794\ne 1167 1797\ne 1167 1800\ne 1167 1803\ne 1168 1169\ne 1168 1189\ne 1168 1211\ne 1168 1322\ne 1168 1339\ne 1168 1470\ne 1168 1703\ne 1168 1915\ne 1169 1579\ne 1169 1637\ne 1169 1659\ne 1169 1794\ne 1169 1915\ne 1169 1991\ne 1170 1230\ne 1170 1445\ne 1170 1539\ne 1170 1568\ne 1170 1651\ne 1170 1661\ne 1170 1679\ne 1170 1698\ne 1170 1791\ne 1171 1172\ne 1171 1282\ne 1171 1388\ne 1171 1557\ne 1171 1587\ne 1171 1610\ne 1171 1818\ne 1172 1173\ne 1172 1198\ne 1172 1256\ne 1172 1277\ne 1172 1304\ne 1172 1313\ne 1172 1330\ne 1172 1331\ne 1172 2002\ne 1173 1174\ne 1173 1188\ne 1173 1698\ne 1173 1788\ne 1173 1789\ne 1174 1175\ne 1174 1227\ne 1174 1269\ne 1174 1270\ne 1174 1271\ne 1174 1443\ne 1174 1698\ne 1174 1760\ne 1174 1837\ne 1174 1889\ne 1174 2035\ne 1174 2075\ne 1174 2093\ne 1175 1271\ne 1175 1416\ne 1175 1444\ne 1175 1789\ne 1175 1790\ne 1175 1801\ne 1175 1837\ne 1175 2074\ne 1176 1177\ne 1176 1180\ne 1176 1229\ne 1176 1279\ne 1176 1484\ne 1176 1548\ne 1176 1644\ne 1176 1757\ne 1176 1823\ne 1176 2020\ne 1176 2058\ne 1177 1178\ne 1177 1180\ne 1177 1183\ne 1177 1237\ne 1177 1291\ne 1177 1451\ne 1177 1476\ne 1177 1512\ne 1177 1525\ne 1177 1738\ne 1177 1739\ne 1177 1753\ne 1177 1966\ne 1178 1184\ne 1178 1186\ne 1178 1199\ne 1178 1204\ne 1178 1278\ne 1178 1279\ne 1178 1306\ne 1178 1309\ne 1178 1451\ne 1178 1502\ne 1178 1640\ne 1178 1692\ne 1178 1778\ne 1178 2094\ne 1179 1180\ne 1179 1183\ne 1179 1255\ne 1179 1381\ne 1179 1397\ne 1179 1398\ne 1179 1666\ne 1179 1668\ne 1179 1719\ne 1179 1747\ne 1179 1856\ne 1180 1181\ne 1180 1259\ne 1180 1260\ne 1180 1406\ne 1180 1500\ne 1180 1553\ne 1180 1719\ne 1180 1748\ne 1180 1855\ne 1180 1874\ne 1180 1898\ne 1180 1974\ne 1180 2075\ne 1181 1182\ne 1181 1237\ne 1181 1251\ne 1181 1255\ne 1181 1256\ne 1181 1280\ne 1181 1308\ne 1181 1331\ne 1181 1400\ne 1181 1423\ne 1181 1425\ne 1181 1521\ne 1181 1762\ne 1181 2045\ne 1181 2072\ne 1182 1203\ne 1182 1257\ne 1182 1264\ne 1182 1267\ne 1182 1314\ne 1182 1402\ne 1182 1405\ne 1182 1411\ne 1182 1423\ne 1182 1453\ne 1182 1589\ne 1182 1683\ne 1182 1711\ne 1182 1716\ne 1182 1729\ne 1182 1736\ne 1182 1748\ne 1182 1779\ne 1182 1827\ne 1182 1892\ne 1182 2041\ne 1183 1184\ne 1183 1475\ne 1183 1752\ne 1184 1185\ne 1184 1303\ne 1184 1305\ne 1185 1186\ne 1185 1319\ne 1185 1351\ne 1185 1415\ne 1185 1446\ne 1185 1480\ne 1185 1522\ne 1185 1565\ne 1185 1571\ne 1185 1607\ne 1185 1625\ne 1185 1627\ne 1185 1752\ne 1186 1320\ne 1186 1342\ne 1186 1347\ne 1186 1351\ne 1186 1385\ne 1186 1408\ne 1186 1411\ne 1186 1416\ne 1186 1421\ne 1186 1448\ne 1186 1481\ne 1186 1523\ne 1186 1566\ne 1186 1582\ne 1186 1640\ne 1186 1693\ne 1186 1753\ne 1186 1782\ne 1186 1906\ne 1186 1907\ne 1186 1953\ne 1187 1188\ne 1187 1191\ne 1187 1194\ne 1187 1198\ne 1187 1199\ne 1187 1201\ne 1187 1204\ne 1187 1307\ne 1187 1508\ne 1187 1515\ne 1187 1516\ne 1187 1689\ne 1187 1737\ne 1187 1830\ne 1188 1189\ne 1188 1194\ne 1188 1213\ne 1188 1321\ne 1188 1374\ne 1188 1375\ne 1188 1443\ne 1188 1771\ne 1188 2021\ne 1189 1190\ne 1189 1201\ne 1189 1239\ne 1189 1771\ne 1189 2079\ne 1189 2081\ne 1190 1211\ne 1190 1270\ne 1190 1302\ne 1190 1400\ne 1190 1423\ne 1190 1443\ne 1190 1456\ne 1190 1462\ne 1190 1514\ne 1190 1693\ne 1190 2097\ne 1190 2125\ne 1190 2129\ne 1191 1192\ne 1191 1204\ne 1191 1300\ne 1191 1348\ne 1191 1399\ne 1191 1525\ne 1191 1526\ne 1191 2100\ne 1192 1193\ne 1192 1213\ne 1192 1216\ne 1192 1300\ne 1192 1301\ne 1192 1344\ne 1192 1511\ne 1192 2099\ne 1193 1214\ne 1193 1233\ne 1193 1302\ne 1193 1344\ne 1193 1345\ne 1193 1413\ne 1193 1443\ne 1193 1476\ne 1193 1526\ne 1193 1785\ne 1193 1903\ne 1193 2076\ne 1193 2099\ne 1193 2128\ne 1194 1195\ne 1194 1268\ne 1194 1381\ne 1194 1433\ne 1194 1434\ne 1194 1442\ne 1194 1499\ne 1194 1517\ne 1194 1518\ne 1194 1563\ne 1194 1611\ne 1194 1612\ne 1195 1196\ne 1195 1269\ne 1195 1284\ne 1195 1493\ne 1195 1500\ne 1195 1518\ne 1195 1774\ne 1195 1830\ne 1195 1971\ne 1195 2021\ne 1195 2064\ne 1195 2075\ne 1196 1197\ne 1196 1284\ne 1196 1289\ne 1196 1434\ne 1196 1439\ne 1196 1523\ne 1196 1693\ne 1196 1758\ne 1196 1759\ne 1196 1763\ne 1196 1859\ne 1196 1871\ne 1196 1932\ne 1196 2086\ne 1197 1386\ne 1197 1444\ne 1197 1461\ne 1197 1478\ne 1197 1564\ne 1197 1636\ne 1197 1811\ne 1197 1859\ne 1197 1902\ne 1198 1199\ne 1198 1305\ne 1198 1344\ne 1198 1445\ne 1198 1446\ne 1198 1499\ne 1198 1557\ne 1198 1560\ne 1198 1688\ne 1198 1698\ne 1198 2128\ne 1199 1200\ne 1199 1213\ne 1199 1256\ne 1199 1335\ne 1199 1381\ne 1199 1411\ne 1199 1425\ne 1199 1778\ne 1199 2023\ne 1200 1223\ne 1200 1296\ne 1200 1306\ne 1200 1336\ne 1200 1558\ne 1200 1570\ne 1200 1669\ne 1200 1730\ne 1200 1736\ne 1200 1824\ne 1200 2071\ne 1201 1202\ne 1201 1203\ne 1201 1399\ne 1201 1411\ne 1201 1459\ne 1201 1572\ne 1201 1582\ne 1201 1882\ne 1201 1988\ne 1202 1203\ne 1202 1370\ne 1202 1400\ne 1202 1452\ne 1202 1518\ne 1202 1573\ne 1203 1314\ne 1203 1401\ne 1203 1470\ne 1203 1984\ne 1203 2045\ne 1204 1205\ne 1204 1482\ne 1204 1496\ne 1204 1582\ne 1204 1685\ne 1204 1692\ne 1204 1881\ne 1205 1206\ne 1205 1292\ne 1205 1418\ne 1205 1488\ne 1205 1515\ne 1205 1526\ne 1205 1881\ne 1205 2060\ne 1206 1207\ne 1206 1299\ne 1206 1372\ne 1206 1592\ne 1206 1593\ne 1206 2061\ne 1207 1221\ne 1207 1314\ne 1207 1429\ne 1207 1576\ne 1207 1648\ne 1207 1909\ne 1207 1926\ne 1207 1948\ne 1208 1209\ne 1208 1212\ne 1208 1216\ne 1208 1219\ne 1208 1312\ne 1208 1414\ne 1208 1721\ne 1208 1728\ne 1209 1210\ne 1209 1267\ne 1209 1268\ne 1209 1301\ne 1209 1414\ne 1209 1494\ne 1209 1533\ne 1209 1683\ne 1209 1722\ne 1209 2068\ne 1209 2091\ne 1210 1211\ne 1210 1268\ne 1210 1282\ne 1210 1530\ne 1210 1715\ne 1211 1252\ne 1211 1286\ne 1211 1324\ne 1211 1419\ne 1211 1431\ne 1211 1457\ne 1211 1530\ne 1211 1532\ne 1211 1533\ne 1211 1534\ne 1211 1800\ne 1211 1914\ne 1211 2015\ne 1212 1213\ne 1212 1262\ne 1212 1267\ne 1212 1307\ne 1212 1359\ne 1212 1667\ne 1212 1729\ne 1212 1731\ne 1212 1746\ne 1213 1214\ne 1213 1216\ne 1213 1223\ne 1213 1350\ne 1213 1359\ne 1213 1381\ne 1213 1512\ne 1213 1834\ne 1213 2081\ne 1214 1215\ne 1214 1228\ne 1214 1347\ne 1214 1360\ne 1214 1368\ne 1214 1375\ne 1214 1378\ne 1214 1413\ne 1214 1425\ne 1214 1448\ne 1214 1834\ne 1215 1223\ne 1215 1230\ne 1215 1296\ne 1215 1448\ne 1215 1539\ne 1215 1579\ne 1215 1833\ne 1215 1838\ne 1215 2076\ne 1216 1217\ne 1216 1219\ne 1216 1282\ne 1216 1348\ne 1216 1413\ne 1216 2091\ne 1217 1218\ne 1217 1511\ne 1217 1512\ne 1217 1714\ne 1217 1728\ne 1217 1770\ne 1218 1233\ne 1218 1402\ne 1218 1599\ne 1218 1643\ne 1218 1683\ne 1218 1954\ne 1218 1975\ne 1218 2067\ne 1218 2091\ne 1218 2098\ne 1219 1220\ne 1219 1359\ne 1219 1482\ne 1219 1494\ne 1219 1499\ne 1219 1688\ne 1220 1221\ne 1220 1360\ne 1220 1413\ne 1220 1414\ne 1220 1488\ne 1220 1494\ne 1221 1347\ne 1221 1370\ne 1221 1463\ne 1221 1488\ne 1221 1493\ne 1221 1499\ne 1221 1518\ne 1221 1560\ne 1221 1636\ne 1221 1909\ne 1222 1223\ne 1222 1226\ne 1222 1230\ne 1222 1321\ne 1222 1449\ne 1222 1730\ne 1223 1224\ne 1223 1230\ne 1223 1322\ne 1223 1350\ne 1223 1833\ne 1223 2037\ne 1223 2038\ne 1223 2057\ne 1223 2124\ne 1224 1225\ne 1224 1449\ne 1224 1456\ne 1224 1507\ne 1224 1647\ne 1224 1659\ne 1224 1680\ne 1224 1736\ne 1224 1853\ne 1224 1854\ne 1224 2019\ne 1224 2049\ne 1224 2081\ne 1224 2107\ne 1224 2124\ne 1225 1476\ne 1225 1501\ne 1225 1581\ne 1225 1647\ne 1225 1687\ne 1225 1699\ne 1225 1822\ne 1225 1842\ne 1225 1860\ne 1225 1919\ne 1225 1957\ne 1225 1959\ne 1225 1960\ne 1225 1964\ne 1225 1966\ne 1225 1969\ne 1225 1994\ne 1225 2040\ne 1225 2076\ne 1225 2113\ne 1226 1227\ne 1226 1300\ne 1226 1317\ne 1226 1356\ne 1226 1388\ne 1226 1757\ne 1226 1797\ne 1226 2037\ne 1227 1228\ne 1227 1230\ne 1227 1288\ne 1227 1309\ne 1227 1337\ne 1227 1354\ne 1227 1392\ne 1227 1404\ne 1227 1437\ne 1227 1758\ne 1227 1761\ne 1227 1787\ne 1227 1838\ne 1227 1890\ne 1227 1942\ne 1227 2037\ne 1227 2078\ne 1228 1229\ne 1228 1337\ne 1228 1378\ne 1228 1789\ne 1228 2012\ne 1229 1242\ne 1229 1280\ne 1229 1548\ne 1229 1645\ne 1229 1758\ne 1229 2058\ne 1229 2061\ne 1229 2062\ne 1230 1231\ne 1230 1244\ne 1230 1353\ne 1230 1446\ne 1230 1570\ne 1230 1578\ne 1230 1680\ne 1230 1698\ne 1231 1232\ne 1231 1496\ne 1231 1556\ne 1231 1758\ne 1231 1843\ne 1232 1496\ne 1232 1564\ne 1232 1657\ne 1232 1706\ne 1232 1959\ne 1233 1234\ne 1233 1235\ne 1233 1240\ne 1233 1247\ne 1233 1290\ne 1233 1299\ne 1233 1331\ne 1233 1378\ne 1233 1476\ne 1233 1536\ne 1233 1742\ne 1233 1817\ne 1233 2011\ne 1233 2040\ne 1233 2073\ne 1234 1235\ne 1234 1242\ne 1234 1248\ne 1234 1458\ne 1234 1702\ne 1234 1774\ne 1234 1779\ne 1234 1886\ne 1234 2018\ne 1234 2045\ne 1234 2050\ne 1234 2058\ne 1234 2065\ne 1234 2067\ne 1235 1236\ne 1235 1241\ne 1235 1612\ne 1235 1805\ne 1235 1965\ne 1235 1990\ne 1236 1247\ne 1236 1248\ne 1236 1270\ne 1236 1273\ne 1236 1510\ne 1236 1687\ne 1236 1897\ne 1236 1910\ne 1236 1927\ne 1236 1978\ne 1236 2052\ne 1236 2076\ne 1236 2098\ne 1236 2116\ne 1237 1238\ne 1237 1243\ne 1237 1245\ne 1237 1290\ne 1237 1307\ne 1237 1371\ne 1237 1425\ne 1237 1512\ne 1237 1740\ne 1237 1776\ne 1238 1239\ne 1238 1423\ne 1238 1448\ne 1238 1450\ne 1238 1451\ne 1238 1570\ne 1238 1731\ne 1238 1735\ne 1238 1777\ne 1238 1782\ne 1238 2071\ne 1238 2082\ne 1239 1266\ne 1239 1276\ne 1239 1356\ne 1239 1423\ne 1239 1616\ne 1239 1705\ne 1239 1764\ne 1239 1767\ne 1239 1773\ne 1239 1781\ne 1239 2077\ne 1239 2107\ne 1240 1241\ne 1240 1242\ne 1240 1345\ne 1240 1346\ne 1240 1550\ne 1240 1956\ne 1241 1242\ne 1241 1273\ne 1241 1618\ne 1242 1287\ne 1242 1371\ne 1242 1372\ne 1242 1521\ne 1242 1573\ne 1242 2061\ne 1243 1244\ne 1243 1378\ne 1243 1387\ne 1243 1585\ne 1243 1796\ne 1244 1245\ne 1244 1295\ne 1244 1678\ne 1244 1697\ne 1244 1761\ne 1245 1296\ne 1245 1372\ne 1245 1697\ne 1245 1762\ne 1245 1966\ne 1245 2124\ne 1246 1247\ne 1246 1259\ne 1246 1413\ne 1246 1415\ne 1246 1421\ne 1246 1425\ne 1246 1559\ne 1246 1639\ne 1246 1640\ne 1246 1784\ne 1246 1796\ne 1247 1248\ne 1247 1391\ne 1247 1406\ne 1247 1416\ne 1247 1920\ne 1247 2073\ne 1247 2074\ne 1247 2087\ne 1247 2111\ne 1248 1249\ne 1248 1259\ne 1248 1779\ne 1248 1793\ne 1248 1840\ne 1248 1967\ne 1248 2033\ne 1248 2092\ne 1248 2131\ne 1249 1251\ne 1249 1258\ne 1249 1391\ne 1249 1455\ne 1249 1793\ne 1249 1835\ne 1249 1869\ne 1249 1887\ne 1249 2042\ne 1250 1251\ne 1250 1255\ne 1250 1258\ne 1250 1262\ne 1250 1327\ne 1250 1335\ne 1250 1700\ne 1251 1252\ne 1251 1253\ne 1251 1258\ne 1251 1293\ne 1251 1308\ne 1251 1702\ne 1251 1740\ne 1251 1743\ne 1251 1828\ne 1251 1835\ne 1251 1996\ne 1252 1253\ne 1252 1385\ne 1252 1521\ne 1252 1617\ne 1252 1623\ne 1252 1693\ne 1252 2130\ne 1253 1254\ne 1253 1280\ne 1253 1324\ne 1253 1327\ne 1253 1328\ne 1253 1477\ne 1253 1492\ne 1253 1500\ne 1253 1624\ne 1253 1865\ne 1253 1866\ne 1253 1950\ne 1253 2013\ne 1253 2032\ne 1253 2064\ne 1253 2120\ne 1254 1379\ne 1254 1408\ne 1254 1609\ne 1254 1648\ne 1254 1674\ne 1254 1801\ne 1254 1802\ne 1254 1826\ne 1254 1845\ne 1254 1875\ne 1254 1901\ne 1254 1916\ne 1254 2024\ne 1254 2052\ne 1254 2129\ne 1255 1256\ne 1255 1330\ne 1255 1363\ne 1255 1396\ne 1255 1403\ne 1255 1520\ne 1255 1729\ne 1255 1764\ne 1256 1257\ne 1256 1279\ne 1257 1263\ne 1257 1313\ne 1257 1588\ne 1257 1634\ne 1257 1700\ne 1257 1729\ne 1257 1737\ne 1257 1827\ne 1258 1259\ne 1258 1328\ne 1258 1410\ne 1258 1573\ne 1258 1711\ne 1258 1792\ne 1258 1796\ne 1258 1874\ne 1259 1260\ne 1259 1286\ne 1259 1666\ne 1259 1669\ne 1259 1670\ne 1259 1738\ne 1259 1792\ne 1259 2130\ne 1260 1261\ne 1260 1288\ne 1260 1328\ne 1260 1432\ne 1260 1484\ne 1260 1490\ne 1260 1521\ne 1260 1668\ne 1260 1691\ne 1260 1739\ne 1260 1937\ne 1260 1978\ne 1260 2032\ne 1260 2033\ne 1260 2118\ne 1261 1285\ne 1261 1316\ne 1261 1545\ne 1261 1855\ne 1261 1922\ne 1261 1923\ne 1261 1934\ne 1261 1937\ne 1262 1263\ne 1262 1412\ne 1262 1530\ne 1262 1746\ne 1263 1264\ne 1263 1355\ne 1263 1725\ne 1263 1756\ne 1263 2046\ne 1264 1436\ne 1264 1469\ne 1264 1470\ne 1264 1679\ne 1264 1683\ne 1264 1791\ne 1264 2046\ne 1265 1266\ne 1265 1307\ne 1265 1405\ne 1265 1764\ne 1266 1267\ne 1266 1268\ne 1266 1405\ne 1266 1435\ne 1266 1443\ne 1267 1289\ne 1267 1495\ne 1267 1834\ne 1267 1903\ne 1267 2056\ne 1267 2071\ne 1267 2130\ne 1268 1269\ne 1268 1274\ne 1268 1339\ne 1268 1435\ne 1268 1531\ne 1268 1532\ne 1268 1767\ne 1268 1797\ne 1269 1270\ne 1269 1532\ne 1269 1774\ne 1269 1798\ne 1269 1799\ne 1269 1813\ne 1269 2050\ne 1269 2056\ne 1269 2068\ne 1269 2084\ne 1270 1272\ne 1270 1274\ne 1270 1314\ne 1270 1315\ne 1270 1437\ne 1270 1456\ne 1270 1911\ne 1270 1929\ne 1270 1986\ne 1270 2116\ne 1271 1272\ne 1271 1373\ne 1271 1415\ne 1271 1532\ne 1271 1784\ne 1271 1788\ne 1271 1800\ne 1272 1273\ne 1272 1274\ne 1272 1431\ne 1272 1532\ne 1273 1369\ne 1273 1432\ne 1273 1545\ne 1273 1785\ne 1273 1790\ne 1273 1909\ne 1274 1275\ne 1274 1405\ne 1274 1436\ne 1274 1462\ne 1274 1464\ne 1274 1604\ne 1274 1650\ne 1274 1683\ne 1274 1715\ne 1275 1519\ne 1275 1541\ne 1275 1682\ne 1275 1709\ne 1275 1713\ne 1275 1797\ne 1275 1814\ne 1276 1277\ne 1276 1355\ne 1276 1403\ne 1276 1486\ne 1277 1278\ne 1277 1309\ne 1277 1319\ne 1277 1348\ne 1277 1377\ne 1277 1388\ne 1277 1396\ne 1277 1446\ne 1277 1449\ne 1277 1486\ne 1277 2107\ne 1278 1279\ne 1278 1348\ne 1278 1481\ne 1278 1485\ne 1278 1695\ne 1278 1699\ne 1278 1732\ne 1278 1776\ne 1278 2113\ne 1279 1280\ne 1279 1303\ne 1279 1320\ne 1279 1363\ne 1279 1365\ne 1279 1733\ne 1279 1846\ne 1279 2115\ne 1280 1311\ne 1280 1363\ne 1280 1521\ne 1280 1624\ne 1280 1776\ne 1280 1846\ne 1280 2014\ne 1280 2060\ne 1280 2109\ne 1280 2114\ne 1281 1282\ne 1281 1286\ne 1281 1452\ne 1281 1489\ne 1281 1530\ne 1281 1546\ne 1281 1666\ne 1281 1667\ne 1282 1283\ne 1282 1286\ne 1282 1714\ne 1282 1746\ne 1282 1757\ne 1282 2090\ne 1283 1284\ne 1283 1300\ne 1283 1489\ne 1283 1490\ne 1283 1691\ne 1283 1720\ne 1284 1285\ne 1284 1322\ne 1284 1433\ne 1284 1438\ne 1284 1636\ne 1284 1720\ne 1284 1883\ne 1284 1885\ne 1284 1889\ne 1284 1938\ne 1284 1972\ne 1285 1438\ne 1285 1453\ne 1285 1490\ne 1285 1491\ne 1285 1506\ne 1285 1579\ne 1285 1883\ne 1285 1935\ne 1286 1287\ne 1286 1381\ne 1286 1400\ne 1286 1413\ne 1286 1453\ne 1286 1490\ne 1286 1671\ne 1286 1749\ne 1286 1758\ne 1286 1804\ne 1286 1840\ne 1286 1843\ne 1286 1886\ne 1286 2015\ne 1286 2090\ne 1287 1288\ne 1287 1431\ne 1287 1453\ne 1287 1491\ne 1287 1575\ne 1287 1591\ne 1287 1714\ne 1287 1717\ne 1287 1718\ne 1287 1856\ne 1287 2067\ne 1287 2088\ne 1288 1317\ne 1288 1393\ne 1288 1394\ne 1288 1431\ne 1288 1464\ne 1288 1519\ne 1288 1521\ne 1288 1769\ne 1288 1793\ne 1288 1808\ne 1288 1839\ne 1288 1887\ne 1288 1890\ne 1288 1985\ne 1288 2015\ne 1288 2027\ne 1288 2131\ne 1289 1290\ne 1289 1307\ne 1289 1366\ne 1289 1392\ne 1289 1448\ne 1289 1497\ne 1289 1581\ne 1289 1589\ne 1289 1621\ne 1289 1632\ne 1289 1633\ne 1289 1693\ne 1289 1762\ne 1289 1830\ne 1289 1836\ne 1289 1994\ne 1289 2019\ne 1290 1291\ne 1290 1362\ne 1290 1378\ne 1290 1402\ne 1290 1994\ne 1290 2054\ne 1290 2072\ne 1290 2082\ne 1290 2111\ne 1291 1292\ne 1291 1402\ne 1291 1526\ne 1291 1632\ne 1291 1779\ne 1291 1878\ne 1291 1919\ne 1291 2058\ne 1291 2094\ne 1291 2095\ne 1292 1362\ne 1292 1685\ne 1292 1739\ne 1292 1779\ne 1292 1815\ne 1292 2033\ne 1292 2053\ne 1292 2065\ne 1293 1294\ne 1293 1309\ne 1293 1335\ne 1293 1337\ne 1293 1385\ne 1293 1389\ne 1293 1391\ne 1293 1467\ne 1293 1796\ne 1293 1850\ne 1293 1873\ne 1293 1954\ne 1293 2023\ne 1293 2047\ne 1294 1295\ne 1294 1336\ne 1294 1337\ne 1294 1407\ne 1294 1552\ne 1294 1647\ne 1294 1726\ne 1294 1824\ne 1294 1828\ne 1294 1863\ne 1294 1942\ne 1294 2038\ne 1295 1296\ne 1295 1336\ne 1295 1559\ne 1295 1569\ne 1295 1654\ne 1295 1706\ne 1295 1708\ne 1295 1711\ne 1295 1761\ne 1295 1795\ne 1295 1796\ne 1295 1892\ne 1295 2083\ne 1296 1316\ne 1296 1425\ne 1296 1559\ne 1296 1570\ne 1296 1762\ne 1296 1824\ne 1296 1963\ne 1296 1970\ne 1296 2082\ne 1296 2123\ne 1297 1298\ne 1297 1558\ne 1297 1635\ne 1297 1743\ne 1297 1744\ne 1297 1756\ne 1297 1767\ne 1297 1946\ne 1298 1299\ne 1298 1610\ne 1298 1613\ne 1298 1755\ne 1298 1767\ne 1298 1773\ne 1299 1361\ne 1299 1362\ne 1299 1613\ne 1299 1615\ne 1299 1617\ne 1299 1939\ne 1300 1398\ne 1300 1689\ne 1300 1691\ne 1300 1695\ne 1300 2028\ne 1301 1302\ne 1301 1312\ne 1301 1370\ne 1301 1399\ne 1301 1493\ne 1301 1511\ne 1301 1992\ne 1301 2049\ne 1301 2050\ne 1301 2081\ne 1301 2096\ne 1302 1334\ne 1302 1370\ne 1302 1533\ne 1302 1785\ne 1302 1945\ne 1302 2096\ne 1302 2129\ne 1303 1304\ne 1303 1319\ne 1303 1363\ne 1303 1383\ne 1303 1786\ne 1304 1305\ne 1304 1704\ne 1304 1765\ne 1304 1768\ne 1304 1786\ne 1304 2115\ne 1305 1306\ne 1305 1482\ne 1305 1627\ne 1305 1688\ne 1305 1730\ne 1305 1732\ne 1306 1492\ne 1306 1503\ne 1306 1860\ne 1306 1942\ne 1306 1966\ne 1306 2115\ne 1306 2128\ne 1307 1308\ne 1307 1366\ne 1307 1387\ne 1307 1409\ne 1307 1434\ne 1307 1446\ne 1307 1459\ne 1307 1496\ne 1307 1571\ne 1307 1580\ne 1307 1595\ne 1307 1620\ne 1307 1740\ne 1307 1764\ne 1308 1309\ne 1308 1317\ne 1308 1409\ne 1308 1623\ne 1308 1624\ne 1308 1692\ne 1308 1716\ne 1308 1836\ne 1308 1887\ne 1309 1310\ne 1309 1389\ne 1309 1403\ne 1309 1417\ne 1309 1446\ne 1309 1450\ne 1309 1512\ne 1309 1851\ne 1309 1942\ne 1310 1311\ne 1310 1363\ne 1310 1383\ne 1310 1385\ne 1310 1403\ne 1310 1420\ne 1310 1463\ne 1310 1486\ne 1310 1540\ne 1310 1623\ne 1310 1733\ne 1310 1787\ne 1311 1423\ne 1311 1473\ne 1311 1705\ne 1311 1733\ne 1311 1777\ne 1311 1806\ne 1311 1851\ne 1311 2117\ne 1312 1313\ne 1312 1370\ne 1312 1395\ne 1312 1499\ne 1313 1314\ne 1313 1449\ne 1313 1576\ne 1313 1650\ne 1313 2049\ne 1314 1315\ne 1314 1331\ne 1314 1370\ne 1314 1372\ne 1314 1535\ne 1314 1646\ne 1314 1650\ne 1314 1679\ne 1314 1680\ne 1314 1698\ne 1314 1706\ne 1314 1892\ne 1314 1945\ne 1314 1962\ne 1314 2049\ne 1314 2056\ne 1315 1316\ne 1315 1416\ne 1315 1566\ne 1315 1785\ne 1315 2121\ne 1316 1372\ne 1316 1605\ne 1316 1669\ne 1316 1708\ne 1316 1775\ne 1316 1870\ne 1316 1908\ne 1316 1967\ne 1317 1318\ne 1317 1519\ne 1317 1691\ne 1317 1714\ne 1317 1715\ne 1317 1743\ne 1317 2027\ne 1318 1319\ne 1318 1388\ne 1318 1393\ne 1318 1606\ne 1318 1607\ne 1318 1676\ne 1318 1701\ne 1318 1943\ne 1319 1320\ne 1319 1396\ne 1319 1480\ne 1319 1768\ne 1319 1769\ne 1320 1456\ne 1320 1481\ne 1320 1763\ne 1320 1781\ne 1320 1885\ne 1320 1943\ne 1320 1981\ne 1320 2107\ne 1320 2125\ne 1321 1322\ne 1321 1326\ne 1321 1412\ne 1321 1433\ne 1321 1578\ne 1321 1597\ne 1322 1323\ne 1322 1324\ne 1322 1349\ne 1322 1352\ne 1322 1358\ne 1322 1578\ne 1322 1579\ne 1322 2021\ne 1322 2022\ne 1322 2102\ne 1323 1324\ne 1323 1353\ne 1323 1451\ne 1323 1477\ne 1323 1539\ne 1323 1636\ne 1323 1654\ne 1323 1957\ne 1324 1325\ne 1324 1354\ne 1324 1406\ne 1324 1412\ne 1324 1495\ne 1324 1726\ne 1324 1852\ne 1324 1893\ne 1324 1899\ne 1324 1951\ne 1324 2038\ne 1324 2087\ne 1324 2109\ne 1325 1354\ne 1325 1358\ne 1325 1431\ne 1325 1437\ne 1325 1513\ne 1325 1599\ne 1325 1654\ne 1325 1866\ne 1325 1900\ne 1325 1904\ne 1325 2119\ne 1325 2123\ne 1326 1327\ne 1326 1340\ne 1326 1673\ne 1326 1694\ne 1326 1786\ne 1327 1328\ne 1327 1363\ne 1327 1379\ne 1327 1381\ne 1327 1393\ne 1327 1412\ne 1327 1482\ne 1327 1502\ne 1327 1547\ne 1327 1691\ne 1327 1692\ne 1327 1694\ne 1327 1766\ne 1328 1329\ne 1328 1347\ne 1328 1488\ne 1328 1609\ne 1328 1874\ne 1328 1953\ne 1329 1338\ne 1329 1590\ne 1329 1663\ne 1329 1711\ne 1329 1766\ne 1329 1867\ne 1329 1872\ne 1329 1950\ne 1329 1953\ne 1329 1978\ne 1330 1331\ne 1330 1415\ne 1330 1479\ne 1330 1480\ne 1330 1650\ne 1330 1788\ne 1331 1332\ne 1331 1377\ne 1331 1416\ne 1331 1481\ne 1331 1535\ne 1331 1560\ne 1331 1723\ne 1331 1765\ne 1331 1789\ne 1331 1847\ne 1331 1848\ne 1331 1898\ne 1331 2002\ne 1331 2045\ne 1332 1333\ne 1332 1359\ne 1332 1375\ne 1332 1461\ne 1332 1476\ne 1332 1479\ne 1332 1481\ne 1332 1496\ne 1332 1498\ne 1332 1499\ne 1332 1522\ne 1332 1525\ne 1332 1581\ne 1332 2125\ne 1333 1334\ne 1333 1370\ne 1333 1395\ne 1333 1399\ne 1333 1460\ne 1333 1525\ne 1333 1549\ne 1333 1753\ne 1333 1772\ne 1333 1962\ne 1334 1399\ne 1334 1498\ne 1334 1526\ne 1334 1590\ne 1334 1877\ne 1334 1878\ne 1334 1901\ne 1334 1988\ne 1335 1336\ne 1335 1445\ne 1335 1502\ne 1335 1625\ne 1335 1791\ne 1335 1796\ne 1336 1503\ne 1336 1558\ne 1336 1569\ne 1336 1626\ne 1336 1652\ne 1336 1681\ne 1336 1700\ne 1336 1756\ne 1336 1791\ne 1337 1338\ne 1337 1402\ne 1337 1539\ne 1337 1671\ne 1337 1791\ne 1337 1793\ne 1337 1795\ne 1337 1872\ne 1337 2030\ne 1337 2092\ne 1337 2093\ne 1338 1385\ne 1338 1533\ne 1338 1637\ne 1338 1662\ne 1338 1722\ne 1338 1787\ne 1338 1821\ne 1338 2012\ne 1338 2030\ne 1339 1340\ne 1339 1341\ne 1339 1349\ne 1339 1419\ne 1339 1635\ne 1339 1703\ne 1339 1731\ne 1339 1767\ne 1339 1813\ne 1340 1341\ne 1340 1388\ne 1340 1486\ne 1340 1610\ne 1340 1660\ne 1340 1664\ne 1340 1673\ne 1340 1713\ne 1340 1755\ne 1340 1797\ne 1340 2001\ne 1341 1342\ne 1341 1434\ne 1341 1481\ne 1341 1611\ne 1341 1665\ne 1341 1696\ne 1341 1768\ne 1341 1932\ne 1342 1394\ne 1342 1408\ne 1342 1419\ne 1342 1460\ne 1342 1808\ne 1343 1344\ne 1343 1373\ne 1343 1443\ne 1343 1462\ne 1343 1474\ne 1343 1746\ne 1344 1345\ne 1344 1370\ne 1344 1475\ne 1344 1525\ne 1344 1698\ne 1344 1746\ne 1345 1346\ne 1345 1373\ne 1345 1560\ne 1345 1749\ne 1345 1790\ne 1346 1347\ne 1346 1475\ne 1346 1476\ne 1346 1564\ne 1347 1381\ne 1347 1394\ne 1347 1425\ne 1347 1500\ne 1347 1518\ne 1347 1567\ne 1347 1719\ne 1347 1941\ne 1348 1349\ne 1348 1352\ne 1348 1376\ne 1348 1397\ne 1348 1465\ne 1348 1482\ne 1348 1524\ne 1348 1537\ne 1348 1652\ne 1348 1757\ne 1348 2101\ne 1349 1350\ne 1349 1351\ne 1349 1465\ne 1349 1522\ne 1349 1523\ne 1349 2070\ne 1350 1351\ne 1350 1446\ne 1350 1448\ne 1350 1731\ne 1350 1732\ne 1350 2069\ne 1351 1381\ne 1351 1384\ne 1351 1397\ne 1351 1611\ne 1351 1649\ne 1351 1751\ne 1351 1906\ne 1352 1353\ne 1352 1354\ne 1352 1358\ne 1352 1398\ne 1352 1485\ne 1352 1538\ne 1352 1720\ne 1352 1803\ne 1352 2104\ne 1352 2105\ne 1353 1354\ne 1353 1638\ne 1353 1654\ne 1354 1800\ne 1354 1955\ne 1354 1975\ne 1355 1356\ne 1355 1436\ne 1355 1628\ne 1355 1629\ne 1356 1357\ne 1356 1436\ne 1356 1437\ne 1356 1449\ne 1356 1450\ne 1356 1541\ne 1356 1599\ne 1356 1853\ne 1356 2043\ne 1356 2046\ne 1357 1358\ne 1357 1428\ne 1357 1492\ne 1357 1541\ne 1357 1577\ne 1357 1629\ne 1357 1630\ne 1357 1705\ne 1357 1726\ne 1357 1799\ne 1357 1808\ne 1357 1888\ne 1357 1890\ne 1357 1942\ne 1357 2108\ne 1358 1401\ne 1358 1465\ne 1358 1470\ne 1358 1591\ne 1358 1597\ne 1358 1598\ne 1358 1654\ne 1358 1656\ne 1358 1761\ne 1358 2103\ne 1358 2124\ne 1359 1360\ne 1359 1412\ne 1359 1435\ne 1359 1495\ne 1359 1496\ne 1359 1723\ne 1359 1732\ne 1359 1755\ne 1359 1778\ne 1359 1843\ne 1359 2015\ne 1359 2057\ne 1360 1361\ne 1360 1369\ne 1360 1424\ne 1360 1495\ne 1360 1777\ne 1360 2012\ne 1361 1362\ne 1361 1378\ne 1361 1487\ne 1361 1664\ne 1361 1755\ne 1361 1832\ne 1361 2012\ne 1361 2053\ne 1362 1616\ne 1362 1621\ne 1362 1773\ne 1362 1831\ne 1362 2054\ne 1363 1364\ne 1363 1520\ne 1363 1548\ne 1363 1765\ne 1364 1365\ne 1364 1440\ne 1364 1765\ne 1364 1786\ne 1364 1787\ne 1364 1794\ne 1365 1485\ne 1365 1671\ne 1365 1763\ne 1365 1786\ne 1365 1798\ne 1365 2059\ne 1365 2115\ne 1366 1367\ne 1366 1439\ne 1366 1447\ne 1366 1515\ne 1366 1887\ne 1367 1368\ne 1367 1513\ne 1367 1514\ne 1367 1633\ne 1368 1369\ne 1368 1402\ne 1368 1417\ne 1368 1512\ne 1368 1772\ne 1368 1785\ne 1368 1795\ne 1368 2121\ne 1369 1440\ne 1369 1551\ne 1369 1609\ne 1370 1371\ne 1370 1380\ne 1370 1414\ne 1370 1420\ne 1370 1462\ne 1371 1372\ne 1371 1571\ne 1371 1782\ne 1371 1856\ne 1372 1515\ne 1372 1572\ne 1372 1591\ne 1372 1603\ne 1372 1678\ne 1372 1686\ne 1372 1773\ne 1372 1923\ne 1372 1976\ne 1372 2018\ne 1372 2036\ne 1372 2055\ne 1373 1374\ne 1373 1443\ne 1373 1444\ne 1373 1564\ne 1373 1749\ne 1373 1810\ne 1374 1375\ne 1374 1479\ne 1374 1516\ne 1374 1517\ne 1374 1580\ne 1374 1585\ne 1374 1788\ne 1374 1843\ne 1375 1444\ne 1375 1515\ne 1375 1518\ne 1375 1578\ne 1375 1581\ne 1375 1772\ne 1375 1789\ne 1375 1804\ne 1375 1842\ne 1375 2017\ne 1375 2021\ne 1375 2078\ne 1375 2086\ne 1376 1377\ne 1376 1413\ne 1376 1428\ne 1376 1488\ne 1376 1504\ne 1376 1512\ne 1376 1523\ne 1376 1524\ne 1376 1526\ne 1376 1640\ne 1376 1654\ne 1376 1758\ne 1376 1776\ne 1376 1874\ne 1376 1895\ne 1376 1898\ne 1376 1955\ne 1376 1958\ne 1376 2095\ne 1376 2101\ne 1376 2105\ne 1377 1378\ne 1377 1392\ne 1377 1417\ne 1377 1447\ne 1377 1487\ne 1377 1680\ne 1377 1776\ne 1377 1848\ne 1377 2106\ne 1377 2107\ne 1378 1391\ne 1378 1392\ne 1378 2083\ne 1378 2095\ne 1379 1380\ne 1379 1382\ne 1379 1460\ne 1379 1547\ne 1379 1607\ne 1379 1609\ne 1379 1657\ne 1379 1673\ne 1379 1682\ne 1379 1701\ne 1379 1800\ne 1379 1876\ne 1379 1955\ne 1379 1977\ne 1380 1381\ne 1380 1419\ne 1380 1460\ne 1380 1462\ne 1380 1567\ne 1380 1636\ne 1380 1646\ne 1380 1649\ne 1380 1683\ne 1380 1746\ne 1380 1747\ne 1380 1954\ne 1380 2129\ne 1381 1419\ne 1381 1475\ne 1381 1499\ne 1381 1500\ne 1381 1856\ne 1382 1432\ne 1382 1509\ne 1382 1631\ne 1382 1634\ne 1382 1672\ne 1382 1691\ne 1382 1715\ne 1382 1744\ne 1382 1977\ne 1382 2052\ne 1383 1384\ne 1383 1733\ne 1383 1797\ne 1384 1385\ne 1384 1610\ne 1384 1625\ne 1384 1635\ne 1384 1681\ne 1384 1722\ne 1384 2010\ne 1385 1386\ne 1385 1421\ne 1385 1617\ne 1385 1625\ne 1385 1636\ne 1385 1639\ne 1385 1708\ne 1385 1755\ne 1385 1775\ne 1385 1885\ne 1385 1953\ne 1385 2010\ne 1385 2048\ne 1385 2117\ne 1386 1390\ne 1386 1621\ne 1386 1775\ne 1386 1777\ne 1386 1870\ne 1387 1388\ne 1387 1392\ne 1387 1426\ne 1387 1583\ne 1387 1584\ne 1387 1694\ne 1388 1389\ne 1388 1392\ne 1388 1695\ne 1388 1995\ne 1389 1390\ne 1389 1426\ne 1389 1537\ne 1390 1391\ne 1390 1392\ne 1390 1406\ne 1390 1417\ne 1390 1426\ne 1390 1428\ne 1390 1874\ne 1391 1417\ne 1391 1429\ne 1391 1796\ne 1391 1850\ne 1391 2111\ne 1392 1393\ne 1392 1410\ne 1392 1477\ne 1392 1577\ne 1392 1664\ne 1392 1756\ne 1392 1761\ne 1392 1864\ne 1392 1876\ne 1392 1955\ne 1392 1995\ne 1392 2081\ne 1393 1394\ne 1393 1410\ne 1393 1454\ne 1393 1505\ne 1393 1529\ne 1393 1608\ne 1393 1677\ne 1393 1693\ne 1393 1865\ne 1393 1943\ne 1394 1409\ne 1394 1434\ne 1394 1441\ne 1394 1442\ne 1394 1463\ne 1394 1564\ne 1394 1859\ne 1394 2127\ne 1395 1396\ne 1395 1399\ne 1395 1751\ne 1395 1771\ne 1396 1397\ne 1396 1734\ne 1396 1764\ne 1397 1398\ne 1397 1640\ne 1397 1974\ne 1398 1511\ne 1398 1553\ne 1398 1591\ne 1398 1720\ne 1398 1764\ne 1398 1814\ne 1399 1400\ne 1399 1401\ne 1399 1465\ne 1399 1582\ne 1399 1596\ne 1399 1877\ne 1399 2004\ne 1399 2005\ne 1400 1401\ne 1400 1518\ne 1400 1758\ne 1400 2125\ne 1401 1402\ne 1401 1671\ne 1401 1762\ne 1401 1771\ne 1401 1772\ne 1401 1799\ne 1401 1820\ne 1401 1963\ne 1401 2021\ne 1401 2050\ne 1401 2059\ne 1401 2079\ne 1401 2119\ne 1402 1512\ne 1402 1633\ne 1402 1834\ne 1402 1851\ne 1402 1854\ne 1402 1855\ne 1402 1872\ne 1402 1895\ne 1402 2080\ne 1402 2098\ne 1403 1404\ne 1403 1423\ne 1403 1435\ne 1403 1462\ne 1403 1571\ne 1403 1729\ne 1403 1778\ne 1404 1405\ne 1404 1573\ne 1404 1690\ne 1404 1787\ne 1404 1800\ne 1404 1876\ne 1405 1418\ne 1405 1436\ne 1405 1459\ne 1405 1514\ne 1405 1515\ne 1405 1716\ne 1405 1911\ne 1405 2043\ne 1406 1407\ne 1406 1408\ne 1406 1416\ne 1406 1423\ne 1406 1451\ne 1406 1510\ne 1406 1748\ne 1406 1775\ne 1406 1801\ne 1406 1921\ne 1406 2027\ne 1406 2038\ne 1406 2126\ne 1407 1408\ne 1407 1675\ne 1407 1681\ne 1407 1708\ne 1407 1726\ne 1407 1793\ne 1407 1802\ne 1407 1808\ne 1407 1869\ne 1407 1949\ne 1407 1967\ne 1407 1968\ne 1407 1999\ne 1407 2010\ne 1408 1607\ne 1408 1608\ne 1408 1696\ne 1408 1899\ne 1408 1943\ne 1409 1410\ne 1409 1425\ne 1409 1426\ne 1409 1810\ne 1410 1411\ne 1410 1450\ne 1410 1460\ne 1410 1572\ne 1410 1627\ne 1410 1669\ne 1410 1681\ne 1410 1711\ne 1410 1869\ne 1410 1870\ne 1410 1873\ne 1410 1953\ne 1410 1996\ne 1411 1736\ne 1411 1782\ne 1411 2081\ne 1412 1413\ne 1412 1435\ne 1412 1441\ne 1412 1443\ne 1412 1513\ne 1412 1530\ne 1412 1537\ne 1412 1538\ne 1412 1756\ne 1413 1414\ne 1413 1467\ne 1413 1504\ne 1413 1539\ne 1413 1804\ne 1413 2087\ne 1413 2091\ne 1414 1530\ne 1414 1531\ne 1414 1533\ne 1414 1662\ne 1415 1416\ne 1415 1420\ne 1416 1417\ne 1416 1560\ne 1416 1785\ne 1416 2109\ne 1416 2131\ne 1417 1418\ne 1417 1447\ne 1417 1851\ne 1417 1887\ne 1417 2121\ne 1418 1464\ne 1418 1532\ne 1418 1593\ne 1418 1630\ne 1418 1665\ne 1418 1772\ne 1418 1787\ne 1418 1799\ne 1418 1887\ne 1419 1420\ne 1419 1421\ne 1419 1422\ne 1419 1430\ne 1419 1534\ne 1419 1941\ne 1419 2126\ne 1420 1421\ne 1420 1532\ne 1420 1535\ne 1421 1533\ne 1421 1536\ne 1421 1992\ne 1421 2130\ne 1422 1423\ne 1422 1683\ne 1422 1767\ne 1422 1808\ne 1422 1924\ne 1423 1535\ne 1423 1693\ne 1423 1775\ne 1423 1851\ne 1423 1984\ne 1423 1985\ne 1424 1425\ne 1424 1521\ne 1424 1775\ne 1424 1778\ne 1425 1515\ne 1425 1560\ne 1425 1796\ne 1425 2023\ne 1425 2111\ne 1426 1427\ne 1426 1796\ne 1427 1428\ne 1427 1524\ne 1427 1537\ne 1427 1561\ne 1427 1583\ne 1427 1629\ne 1427 1653\ne 1427 1810\ne 1428 1429\ne 1428 1450\ne 1428 1537\ne 1428 1540\ne 1428 1577\ne 1428 1654\ne 1428 1777\ne 1428 1809\ne 1428 1893\ne 1428 1894\ne 1428 1916\ne 1428 1953\ne 1429 1467\ne 1429 1561\ne 1429 1891\ne 1429 1892\ne 1429 1916\ne 1429 2108\ne 1429 2110\ne 1430 1431\ne 1430 1450\ne 1430 1565\ne 1430 1782\ne 1430 1856\ne 1431 1432\ne 1431 1470\ne 1431 1715\ne 1431 1800\ne 1431 1986\ne 1431 2089\ne 1432 1609\ne 1432 1618\ne 1432 1637\ne 1432 1745\ne 1432 1748\ne 1432 1750\ne 1432 2052\ne 1433 1434\ne 1433 1438\ne 1433 1441\ne 1433 1562\ne 1434 1435\ne 1434 1439\ne 1434 1522\ne 1434 1546\ne 1434 1564\ne 1434 1571\ne 1434 1769\ne 1435 1436\ne 1436 1437\ne 1436 1513\ne 1436 1630\ne 1437 1616\ne 1437 1680\ne 1437 1759\ne 1437 1853\ne 1437 1911\ne 1437 2121\ne 1438 1439\ne 1438 1452\ne 1438 1489\ne 1438 1505\ne 1438 1518\ne 1438 1578\ne 1438 1656\ne 1439 1440\ne 1439 1518\ne 1439 1665\ne 1440 1454\ne 1440 1491\ne 1440 1593\ne 1440 1763\ne 1440 1769\ne 1440 2061\ne 1441 1442\ne 1441 1542\ne 1441 1562\ne 1442 1443\ne 1442 1510\ne 1442 1563\ne 1443 1444\ne 1443 1461\ne 1443 1514\ne 1443 1692\ne 1443 1852\ne 1443 1903\ne 1444 1809\ne 1444 1841\ne 1444 1852\ne 1444 2086\ne 1444 2118\ne 1445 1446\ne 1445 1467\ne 1445 1561\ne 1446 1447\ne 1446 1448\ne 1446 1522\ne 1446 1580\ne 1447 1448\ne 1447 1560\ne 1447 1561\ne 1448 1467\ne 1448 1523\ne 1448 1581\ne 1448 1777\ne 1448 1851\ne 1448 2069\ne 1448 2107\ne 1448 2128\ne 1449 1450\ne 1449 1577\ne 1449 1652\ne 1449 1658\ne 1449 1680\ne 1449 1699\ne 1449 1734\ne 1450 1451\ne 1450 1460\ne 1450 1512\ne 1450 1540\ne 1450 1716\ne 1450 1854\ne 1450 1892\ne 1450 1998\ne 1450 2047\ne 1450 2121\ne 1451 1557\ne 1451 1559\ne 1451 1560\ne 1451 1566\ne 1451 1636\ne 1451 1699\ne 1451 1749\ne 1451 1750\ne 1451 1810\ne 1451 1919\ne 1451 1949\ne 1451 1977\ne 1451 1997\ne 1452 1453\ne 1452 1470\ne 1452 1729\ne 1453 1454\ne 1453 1456\ne 1453 1471\ne 1453 1567\ne 1453 1759\ne 1453 1985\ne 1453 2008\ne 1454 1455\ne 1454 1609\ne 1454 1656\ne 1454 1677\ne 1454 1701\ne 1454 1711\ne 1454 1745\ne 1454 1759\ne 1454 2024\ne 1455 1573\ne 1455 1693\ne 1455 1702\ne 1455 1745\ne 1455 1825\ne 1455 1918\ne 1455 1989\ne 1456 1457\ne 1456 1471\ne 1456 1506\ne 1456 1566\ne 1456 1591\ne 1456 1677\ne 1456 1687\ne 1456 1736\ne 1456 1813\ne 1456 1879\ne 1456 1882\ne 1456 1968\ne 1457 1458\ne 1457 1469\ne 1457 1541\ne 1457 1615\ne 1457 1677\ne 1457 1682\ne 1457 1722\ne 1457 1725\ne 1457 1726\ne 1457 1875\ne 1457 1885\ne 1457 1886\ne 1457 2046\ne 1457 2048\ne 1458 1501\ne 1458 1536\ne 1458 1670\ne 1458 1774\ne 1458 1827\ne 1458 1875\ne 1458 1882\ne 1458 1915\ne 1458 1933\ne 1458 1984\ne 1458 1986\ne 1458 1987\ne 1458 1989\ne 1458 1990\ne 1458 2034\ne 1458 2052\ne 1458 2116\ne 1458 2130\ne 1459 1460\ne 1459 1595\ne 1459 1665\ne 1459 1876\ne 1460 1461\ne 1460 1514\ne 1460 1591\ne 1460 1753\ne 1460 1901\ne 1460 1902\ne 1460 1961\ne 1460 2005\ne 1460 2006\ne 1461 1474\ne 1461 1476\ne 1461 1564\ne 1461 1635\ne 1461 1696\ne 1461 1811\ne 1461 1959\ne 1461 2127\ne 1462 1463\ne 1462 1530\ne 1462 1563\ne 1462 1571\ne 1463 1464\ne 1463 1505\ne 1463 1636\ne 1463 1810\ne 1464 1505\ne 1464 1542\ne 1464 1630\ne 1464 1888\ne 1465 1466\ne 1465 1481\ne 1465 1505\ne 1465 1756\ne 1465 1879\ne 1466 1467\ne 1466 1481\ne 1466 1544\ne 1466 1751\ne 1466 1754\ne 1466 1896\ne 1466 1898\ne 1467 1503\ne 1467 1537\ne 1467 1539\ne 1467 1561\ne 1467 1576\ne 1467 1643\ne 1467 1891\ne 1467 2128\ne 1468 1469\ne 1468 1622\ne 1468 1663\ne 1468 1664\ne 1468 1677\ne 1468 1707\ne 1468 1713\ne 1469 1662\ne 1469 1725\ne 1470 1471\ne 1470 1541\ne 1470 1618\ne 1470 1656\ne 1470 1679\ne 1470 1769\ne 1471 1472\ne 1471 1671\ne 1471 1763\ne 1471 1915\ne 1471 1935\ne 1471 2007\ne 1471 2089\ne 1471 2103\ne 1472 1473\ne 1472 1541\ne 1472 1659\ne 1472 1763\ne 1472 1798\ne 1472 1853\ne 1472 1905\ne 1472 1933\ne 1472 1985\ne 1472 2085\ne 1473 1534\ne 1473 1540\ne 1473 1659\ne 1473 1854\ne 1473 1894\ne 1473 1913\ne 1473 1941\ne 1474 1475\ne 1474 1479\ne 1474 1564\ne 1474 1606\ne 1475 1476\ne 1475 1479\ne 1475 1694\ne 1475 1699\ne 1475 1752\ne 1475 1843\ne 1476 1477\ne 1476 1500\ne 1476 1753\ne 1476 1823\ne 1476 1842\ne 1476 1939\ne 1476 1954\ne 1476 2081\ne 1477 1478\ne 1477 1496\ne 1477 1685\ne 1477 1694\ne 1477 1695\ne 1477 1740\ne 1477 1742\ne 1477 1754\ne 1477 1776\ne 1477 1955\ne 1477 1993\ne 1478 1529\ne 1478 1594\ne 1478 1602\ne 1478 1665\ne 1478 1687\ne 1478 1696\ne 1478 1865\ne 1478 1900\ne 1478 1931\ne 1478 1970\ne 1478 2083\ne 1479 1480\ne 1479 1580\ne 1480 1481\ne 1480 1585\ne 1480 1606\ne 1481 1696\ne 1481 1755\ne 1481 1780\ne 1481 2112\ne 1482 1483\ne 1482 1486\ne 1482 1488\ne 1482 1489\ne 1482 1492\ne 1482 1496\ne 1482 1499\ne 1482 1502\ne 1482 1505\ne 1482 1600\ne 1482 1629\ne 1482 1657\ne 1482 1754\ne 1482 1803\ne 1483 1484\ne 1483 1613\ne 1483 1619\ne 1483 1754\ne 1483 1776\ne 1483 1848\ne 1484 1485\ne 1484 1617\ne 1484 1618\ne 1484 1720\ne 1484 1723\ne 1484 1739\ne 1484 1769\ne 1484 1804\ne 1484 1923\ne 1484 1938\ne 1484 2061\ne 1484 2065\ne 1484 2066\ne 1485 1556\ne 1485 2057\ne 1486 1487\ne 1486 1585\ne 1486 1629\ne 1486 1658\ne 1486 1704\ne 1486 1705\ne 1486 1755\ne 1487 1488\ne 1487 1630\ne 1487 1664\ne 1487 1705\ne 1488 1492\ne 1488 1630\ne 1488 1655\ne 1488 2053\ne 1489 1490\ne 1489 1668\ne 1489 1803\ne 1490 1491\ne 1490 1492\ne 1490 1637\ne 1490 1890\ne 1491 1575\ne 1491 1592\ne 1491 1720\ne 1491 1973\ne 1491 2061\ne 1491 2105\ne 1492 1493\ne 1492 1494\ne 1492 1497\ne 1492 1506\ne 1492 1592\ne 1492 1617\ne 1492 1648\ne 1492 1705\ne 1492 1816\ne 1492 1881\ne 1492 2064\ne 1492 2101\ne 1493 1494\ne 1493 1498\ne 1493 1499\ne 1493 1500\ne 1493 1523\ne 1493 1812\ne 1493 1898\ne 1493 1926\ne 1493 1927\ne 1493 1992\ne 1493 2011\ne 1493 2128\ne 1494 1495\ne 1494 1724\ne 1494 2013\ne 1494 2091\ne 1495 1497\ne 1495 1498\ne 1495 1775\ne 1495 1832\ne 1495 1834\ne 1495 1842\ne 1495 1849\ne 1495 1860\ne 1495 1912\ne 1495 1937\ne 1495 1952\ne 1496 1497\ne 1496 1549\ne 1496 1620\ne 1496 1624\ne 1496 1665\ne 1496 1765\ne 1496 1776\ne 1497 1498\ne 1497 1590\ne 1497 1621\ne 1497 1868\ne 1497 1881\ne 1497 1993\ne 1497 2054\ne 1497 2114\ne 1498 1523\ne 1498 1526\ne 1498 1811\ne 1498 1847\ne 1498 2017\ne 1498 2097\ne 1498 2112\ne 1499 1522\ne 1499 1576\ne 1499 1635\ne 1499 1804\ne 1500 1501\ne 1500 1823\ne 1500 1834\ne 1500 1840\ne 1500 1855\ne 1500 1906\ne 1500 1937\ne 1500 2013\ne 1500 2023\ne 1500 2073\ne 1500 2125\ne 1500 2126\ne 1500 2127\ne 1500 2129\ne 1500 2130\ne 1501 1553\ne 1501 1726\ne 1501 1799\ne 1501 1806\ne 1501 1811\ne 1501 1815\ne 1501 1818\ne 1501 1820\ne 1501 1821\ne 1501 1822\ne 1501 1824\ne 1501 1825\ne 1501 1827\ne 1501 1831\ne 1501 1833\ne 1501 1835\ne 1501 1837\ne 1501 1840\ne 1501 1846\ne 1501 1847\ne 1501 1850\ne 1501 1852\ne 1501 1853\ne 1501 1857\ne 1501 1864\ne 1501 1877\ne 1501 1883\ne 1501 1891\ne 1501 1904\ne 1501 1906\ne 1501 1913\ne 1501 1917\ne 1501 1919\ne 1501 1922\ne 1501 1925\ne 1501 1926\ne 1501 1928\ne 1501 1931\ne 1501 1933\ne 1501 1940\ne 1501 1944\ne 1501 1946\ne 1501 1950\ne 1501 1967\ne 1501 1971\ne 1501 1980\ne 1501 2028\ne 1501 2039\ne 1501 2058\ne 1501 2068\ne 1501 2077\ne 1501 2096\ne 1501 2102\ne 1502 1503\ne 1502 1504\ne 1502 1625\ne 1503 1504\ne 1503 1626\ne 1503 1766\ne 1504 1639\ne 1504 1662\ne 1504 1724\ne 1504 1791\ne 1504 1803\ne 1504 2030\ne 1504 2031\ne 1505 1506\ne 1505 1577\ne 1505 1582\ne 1505 1600\ne 1505 1602\ne 1505 1627\ne 1505 1656\ne 1505 1665\ne 1506 1507\ne 1506 1592\ne 1506 1865\ne 1506 1868\ne 1506 1879\ne 1506 1880\ne 1506 1935\ne 1507 1523\ne 1507 1577\ne 1507 1579\ne 1507 1659\ne 1507 1864\ne 1507 1871\ne 1507 1894\ne 1507 1926\ne 1507 1958\ne 1508 1509\ne 1508 1511\ne 1508 1514\ne 1508 1587\ne 1508 1634\ne 1508 2044\ne 1509 1510\ne 1509 1543\ne 1509 1712\ne 1510 1542\ne 1510 1612\ne 1510 1984\ne 1510 1999\ne 1510 2044\ne 1510 2052\ne 1510 2075\ne 1511 1512\ne 1511 1634\ne 1511 1751\ne 1511 1785\ne 1511 1898\ne 1511 2098\ne 1512 1651\ne 1512 1729\ne 1512 1771\ne 1512 1791\ne 1512 1856\ne 1512 2081\ne 1512 2124\ne 1513 1514\ne 1513 1597\ne 1513 1599\ne 1513 1638\ne 1513 1653\ne 1514 1515\ne 1514 1591\ne 1514 1748\ne 1514 1771\ne 1514 1785\ne 1514 1800\ne 1514 2044\ne 1514 2119\ne 1515 1516\ne 1515 1518\ne 1515 1526\ne 1515 1560\ne 1515 1690\ne 1515 1779\ne 1515 1830\ne 1515 1910\ne 1515 1977\ne 1515 1988\ne 1515 2036\ne 1515 2045\ne 1515 2097\ne 1515 2131\ne 1516 1517\ne 1516 1525\ne 1516 1603\ne 1516 1738\ne 1517 1518\ne 1517 1520\ne 1517 1531\ne 1518 1521\ne 1518 1532\ne 1518 1582\ne 1518 1923\ne 1518 2008\ne 1519 1520\ne 1519 1668\ne 1519 1792\ne 1520 1521\ne 1520 1531\ne 1520 1668\ne 1521 1532\ne 1521 1623\ne 1521 1774\ne 1521 1979\ne 1521 2078\ne 1522 1523\ne 1522 1524\ne 1522 1577\ne 1522 1578\ne 1522 1848\ne 1523 1579\ne 1523 1899\ne 1523 2070\ne 1523 2081\ne 1523 2108\ne 1523 2109\ne 1524 1525\ne 1524 1546\ne 1524 1653\ne 1525 1526\ne 1525 1810\ne 1526 1632\ne 1526 1809\ne 1526 1898\ne 1526 1963\ne 1526 2059\ne 1526 2100\ne 1527 1528\ne 1527 1625\ne 1527 1738\ne 1527 1752\ne 1527 1792\ne 1528 1529\ne 1528 1569\ne 1528 1653\ne 1528 1694\ne 1528 1752\ne 1529 1544\ne 1529 1564\ne 1529 1606\ne 1529 1608\ne 1529 1678\ne 1529 1694\ne 1529 1696\ne 1530 1531\ne 1530 1725\ne 1531 1532\ne 1531 1614\ne 1532 1533\ne 1532 1535\ne 1532 1615\ne 1532 1787\ne 1532 2109\ne 1532 2118\ne 1533 1536\ne 1533 1784\ne 1533 1991\ne 1533 2008\ne 1533 2014\ne 1533 2068\ne 1533 2087\ne 1534 1535\ne 1534 1536\ne 1534 1540\ne 1534 1646\ne 1534 1703\ne 1534 1733\ne 1534 1893\ne 1534 1913\ne 1534 1998\ne 1535 1536\ne 1535 1582\ne 1535 1984\ne 1536 1559\ne 1536 1687\ne 1536 1891\ne 1536 1924\ne 1536 1990\ne 1536 2011\ne 1536 2082\ne 1536 2083\ne 1536 2084\ne 1537 1538\ne 1537 1652\ne 1537 1732\ne 1537 1893\ne 1538 1539\ne 1538 1542\ne 1538 1766\ne 1538 1951\ne 1538 2057\ne 1539 1559\ne 1539 1671\ne 1539 1742\ne 1539 1957\ne 1539 1958\ne 1539 2076\ne 1540 1541\ne 1540 1658\ne 1540 2048\ne 1541 1542\ne 1541 1615\ne 1541 1617\ne 1541 1628\ne 1541 1658\ne 1541 1769\ne 1541 1797\ne 1541 1984\ne 1542 1599\ne 1542 1617\ne 1542 1657\ne 1542 1715\ne 1542 1720\ne 1542 1905\ne 1543 1544\ne 1543 1607\ne 1543 1712\ne 1544 1545\ne 1544 1563\ne 1544 1618\ne 1544 1684\ne 1544 1687\ne 1544 1699\ne 1544 1751\ne 1544 1780\ne 1545 1687\ne 1545 1739\ne 1545 1753\ne 1545 1923\ne 1545 1939\ne 1546 1547\ne 1546 1548\ne 1546 1554\ne 1546 1757\ne 1546 1758\ne 1547 1548\ne 1547 1555\ne 1547 1791\ne 1548 1556\ne 1548 1717\ne 1548 1719\ne 1549 1550\ne 1549 1590\ne 1549 1595\ne 1549 1596\ne 1549 1600\ne 1549 1663\ne 1549 1665\ne 1550 1551\ne 1550 1776\ne 1551 1593\ne 1551 1622\ne 1551 1627\ne 1551 1735\ne 1551 1777\ne 1551 1870\ne 1552 1553\ne 1552 1596\ne 1552 1632\ne 1552 1647\ne 1552 1817\ne 1552 1931\ne 1552 1976\ne 1552 2004\ne 1552 2019\ne 1553 1632\ne 1553 1670\ne 1553 1762\ne 1553 1872\ne 1553 1921\ne 1553 1928\ne 1553 1966\ne 1553 1973\ne 1553 1974\ne 1553 1975\ne 1553 1978\ne 1553 2020\ne 1553 2028\ne 1553 2093\ne 1553 2098\ne 1553 2104\ne 1553 2122\ne 1554 1555\ne 1554 1556\ne 1554 1771\ne 1555 1556\ne 1555 1644\ne 1555 1674\ne 1555 1766\ne 1556 1706\ne 1556 1765\ne 1556 1783\ne 1556 2020\ne 1556 2062\ne 1557 1558\ne 1557 1635\ne 1557 1649\ne 1557 1731\ne 1557 1746\ne 1557 1997\ne 1558 1559\ne 1558 1568\ne 1559 1568\ne 1559 1670\ne 1559 1686\ne 1559 1697\ne 1559 1760\ne 1559 1784\ne 1559 1920\ne 1559 1949\ne 1560 1561\ne 1560 1790\ne 1560 1810\ne 1560 2110\ne 1560 2128\ne 1561 1796\ne 1561 1848\ne 1562 1563\ne 1562 1564\ne 1562 1625\ne 1562 1635\ne 1562 1636\ne 1562 1657\ne 1563 1564\ne 1563 1778\ne 1565 1566\ne 1565 1649\ne 1565 1752\ne 1566 1567\ne 1566 1649\ne 1566 1753\ne 1566 2048\ne 1567 1609\ne 1567 1718\ne 1567 1749\ne 1567 1902\ne 1567 1941\ne 1567 2129\ne 1568 1569\ne 1568 1570\ne 1568 1688\ne 1569 1570\ne 1569 1653\ne 1569 1709\ne 1569 1710\ne 1570 1605\ne 1570 1730\ne 1570 1735\ne 1570 1764\ne 1571 1572\ne 1571 1573\ne 1571 1604\ne 1571 1693\ne 1572 1573\ne 1572 1588\ne 1572 1596\ne 1572 1622\ne 1572 1627\ne 1572 1712\ne 1572 1713\ne 1572 1976\ne 1572 1989\ne 1573 1574\ne 1573 1623\ne 1573 1702\ne 1573 1758\ne 1574 1575\ne 1574 1712\ne 1574 1714\ne 1574 1720\ne 1574 1744\ne 1574 1745\ne 1574 1792\ne 1575 1707\ne 1575 1745\ne 1575 1794\ne 1576 1577\ne 1576 1581\ne 1576 1612\ne 1576 1804\ne 1576 1888\ne 1576 1926\ne 1576 1964\ne 1577 1578\ne 1577 1581\ne 1577 1583\ne 1577 1627\ne 1577 1658\ne 1577 1957\ne 1578 1579\ne 1578 1804\ne 1578 1890\ne 1579 1927\ne 1579 2017\ne 1579 2102\ne 1579 2105\ne 1580 1581\ne 1580 1583\ne 1580 1699\ne 1580 1810\ne 1581 1582\ne 1581 1799\ne 1581 1809\ne 1581 1959\ne 1581 1962\ne 1581 2056\ne 1581 2057\ne 1582 1611\ne 1582 1617\ne 1582 1665\ne 1582 1687\ne 1582 1693\ne 1582 1880\ne 1582 1992\ne 1583 1584\ne 1583 1629\ne 1584 1585\ne 1584 1673\ne 1585 1597\ne 1585 1603\ne 1585 1658\ne 1585 1673\ne 1585 1769\ne 1585 1788\ne 1586 1587\ne 1586 1591\ne 1586 1595\ne 1586 1597\ne 1586 1600\ne 1586 1603\ne 1586 1727\ne 1586 1728\ne 1587 1588\ne 1587 1591\ne 1587 1598\ne 1587 1712\ne 1587 1714\ne 1587 1819\ne 1588 1589\ne 1588 1595\ne 1588 1596\ne 1588 1700\ne 1589 1590\ne 1589 1591\ne 1589 1595\ne 1589 1622\ne 1589 1711\ne 1589 1759\ne 1589 1962\ne 1590 1592\ne 1590 1596\ne 1590 1622\ne 1590 1868\ne 1590 1917\ne 1591 1592\ne 1591 1682\ne 1591 1683\ne 1591 1686\ne 1591 1707\ne 1591 1808\ne 1591 1819\ne 1591 1856\ne 1591 1928\ne 1591 2124\ne 1592 1593\ne 1592 1600\ne 1592 1707\ne 1592 1866\ne 1592 1958\ne 1593 1594\ne 1593 1615\ne 1593 1665\ne 1593 1677\ne 1593 1705\ne 1593 1765\ne 1593 1768\ne 1593 1981\ne 1594 1675\ne 1594 1677\ne 1594 1678\ne 1594 1742\ne 1594 1832\ne 1594 1980\ne 1594 2055\ne 1595 1710\ne 1595 1729\ne 1596 1758\ne 1596 1766\ne 1596 1873\ne 1596 1917\ne 1596 1932\ne 1596 1942\ne 1596 1961\ne 1597 1598\ne 1597 1629\ne 1597 1653\ne 1597 1770\ne 1597 1771\ne 1598 1599\ne 1598 1652\ne 1599 1631\ne 1599 1633\ne 1599 1652\ne 1599 1715\ne 1599 1904\ne 1599 2044\ne 1600 1601\ne 1601 1602\ne 1601 1638\ne 1601 1678\ne 1601 1686\ne 1601 1754\ne 1602 1636\ne 1602 1687\ne 1602 1754\ne 1602 1949\ne 1602 1957\ne 1603 1604\ne 1603 1605\ne 1603 1650\ne 1603 1780\ne 1604 1605\ne 1604 1764\ne 1604 1769\ne 1604 1781\ne 1605 1640\ne 1605 1669\ne 1605 1778\ne 1605 1782\ne 1606 1607\ne 1606 1673\ne 1606 1696\ne 1607 1608\ne 1607 1681\ne 1608 1609\ne 1608 1708\ne 1609 1655\ne 1609 1902\ne 1610 1611\ne 1610 1617\ne 1610 1672\ne 1610 1712\ne 1610 2000\ne 1611 1612\ne 1612 1702\ne 1612 1965\ne 1612 1971\ne 1613 1614\ne 1614 1615\ne 1614 1628\ne 1614 1725\ne 1614 1767\ne 1614 1848\ne 1615 1616\ne 1615 1723\ne 1615 1767\ne 1615 1923\ne 1615 2084\ne 1615 2085\ne 1615 2108\ne 1615 2116\ne 1616 1875\ne 1616 2077\ne 1616 2078\ne 1616 2106\ne 1616 2116\ne 1617 1618\ne 1617 1621\ne 1617 1664\ne 1617 1707\ne 1617 1814\ne 1617 1933\ne 1617 1992\ne 1617 2000\ne 1617 2057\ne 1618 1619\ne 1618 1644\ne 1618 1672\ne 1618 1745\ne 1618 1965\ne 1618 2007\ne 1619 1642\ne 1619 1643\ne 1619 1657\ne 1619 1679\ne 1619 1805\ne 1619 1948\ne 1619 1964\ne 1620 1621\ne 1620 1623\ne 1621 1622\ne 1621 1664\ne 1621 1777\ne 1621 1979\ne 1621 2097\ne 1622 1707\ne 1622 1870\ne 1622 1988\ne 1622 1989\ne 1623 1624\ne 1624 1758\ne 1624 2015\ne 1624 2125\ne 1625 1626\ne 1625 1662\ne 1625 1778\ne 1626 1627\ne 1626 1657\ne 1626 1661\ne 1626 1662\ne 1627 1706\ne 1627 1732\ne 1627 1734\ne 1627 1768\ne 1627 1808\ne 1627 1810\ne 1627 1949\ne 1628 1629\ne 1628 1725\ne 1629 1630\ne 1629 1756\ne 1629 1848\ne 1631 1632\ne 1631 1634\ne 1631 1638\ne 1631 1685\ne 1631 1812\ne 1631 1898\ne 1631 1940\ne 1631 1955\ne 1631 1968\ne 1632 1633\ne 1632 1811\ne 1632 1901\ne 1632 1903\ne 1632 1955\ne 1632 1977\ne 1632 2038\ne 1632 2063\ne 1633 1716\ne 1633 2044\ne 1633 2122\ne 1634 1737\ne 1634 1747\ne 1634 1748\ne 1634 2051\ne 1635 1636\ne 1635 1812\ne 1636 1637\ne 1636 1638\ne 1636 1648\ne 1636 1812\ne 1636 1884\ne 1636 1939\ne 1636 1954\ne 1636 2118\ne 1637 1648\ne 1637 1750\ne 1637 1909\ne 1638 1639\ne 1638 1720\ne 1638 1739\ne 1638 1747\ne 1639 1640\ne 1639 1661\ne 1639 1708\ne 1639 1814\ne 1640 1719\ne 1640 1974\ne 1641 1642\ne 1641 1646\ne 1641 1650\ne 1641 1652\ne 1641 1657\ne 1641 1715\ne 1642 1643\ne 1642 1650\ne 1642 1683\ne 1643 1644\ne 1643 1651\ne 1643 1652\ne 1644 1645\ne 1644 1646\ne 1644 1717\ne 1644 1741\ne 1644 1748\ne 1644 1751\ne 1644 1771\ne 1644 1820\ne 1644 2067\ne 1645 1717\ne 1645 1753\ne 1645 1772\ne 1645 1820\ne 1645 1842\ne 1645 1960\ne 1645 2006\ne 1645 2088\ne 1646 1647\ne 1646 1648\ne 1646 1649\ne 1646 1944\ne 1646 1961\ne 1646 1976\ne 1646 1986\ne 1646 2047\ne 1647 1648\ne 1647 1652\ne 1647 1654\ne 1647 1893\ne 1647 1904\ne 1647 1954\ne 1647 1968\ne 1647 2101\ne 1648 1655\ne 1648 1657\ne 1648 1659\ne 1648 1905\ne 1648 1916\ne 1648 1948\ne 1648 1972\ne 1649 1751\ne 1650 1651\ne 1650 1709\ne 1650 1729\ne 1651 1652\ne 1651 1771\ne 1652 1653\ne 1652 1654\ne 1652 1657\ne 1652 1699\ne 1653 1654\ne 1654 1655\ne 1654 1680\ne 1654 1963\ne 1654 2006\ne 1655 1656\ne 1655 1657\ne 1656 1706\ne 1656 1707\ne 1656 1803\ne 1656 1935\ne 1656 1972\ne 1657 1658\ne 1658 1659\ne 1658 1703\ne 1658 1709\ne 1658 1876\ne 1659 1703\ne 1659 1705\ne 1659 1706\ne 1659 1862\ne 1659 1875\ne 1659 1990\ne 1659 2057\ne 1659 2083\ne 1660 1661\ne 1660 1663\ne 1660 1713\ne 1660 1721\ne 1660 1766\ne 1661 1678\ne 1661 1694\ne 1661 1742\ne 1661 1755\ne 1662 1663\ne 1662 1690\ne 1662 1721\ne 1662 1722\ne 1662 1791\ne 1663 1664\ne 1663 1710\ne 1663 1766\ne 1663 1814\ne 1664 1665\ne 1664 1787\ne 1664 1867\ne 1664 1876\ne 1664 2001\ne 1665 1868\ne 1665 1932\ne 1666 1667\ne 1666 1668\ne 1667 1729\ne 1667 1730\ne 1668 1691\ne 1668 1814\ne 1669 1670\ne 1669 1681\ne 1669 1967\ne 1669 1974\ne 1669 1987\ne 1670 1671\ne 1670 1682\ne 1670 1711\ne 1670 1745\ne 1670 1976\ne 1670 1977\ne 1670 1978\ne 1670 2034\ne 1670 2036\ne 1671 1674\ne 1671 1745\ne 1671 1750\ne 1671 1789\ne 1671 1794\ne 1671 1798\ne 1671 1801\ne 1671 1863\ne 1671 1915\ne 1671 2050\ne 1671 2104\ne 1672 1673\ne 1672 1744\ne 1672 1965\ne 1673 1674\ne 1673 1701\ne 1674 1696\ne 1674 1957\ne 1674 1965\ne 1674 1976\ne 1674 2001\ne 1674 2024\ne 1675 1676\ne 1675 1678\ne 1675 1696\ne 1675 1760\ne 1675 1768\ne 1675 1886\ne 1675 1949\ne 1675 1980\ne 1675 2018\ne 1675 2019\ne 1676 1677\ne 1676 1701\ne 1676 1713\ne 1676 1768\ne 1676 1982\ne 1677 1713\ne 1677 1766\ne 1677 1982\ne 1677 1989\ne 1677 1999\ne 1677 2001\ne 1678 1679\ne 1678 1708\ne 1678 1713\ne 1678 1755\ne 1679 1680\ne 1679 1848\ne 1680 1735\ne 1680 2121\ne 1681 1682\ne 1681 1708\ne 1681 1712\ne 1681 1756\ne 1681 1792\ne 1682 1683\ne 1682 1711\ne 1682 1722\ne 1682 1727\ne 1682 1792\ne 1682 1800\ne 1682 1802\ne 1682 1975\ne 1683 1715\ne 1683 1728\ne 1683 1929\ne 1684 1685\ne 1684 1688\ne 1684 1691\ne 1684 1694\ne 1684 1739\ne 1685 1686\ne 1685 1687\ne 1685 1737\ne 1685 1739\ne 1685 1754\ne 1685 1773\ne 1685 1815\ne 1685 2032\ne 1685 2066\ne 1686 1687\ne 1686 1966\ne 1687 1896\ne 1687 1922\ne 1687 1968\ne 1687 1969\ne 1687 2004\ne 1687 2007\ne 1687 2008\ne 1688 1689\ne 1689 1690\ne 1689 1721\ne 1689 2034\ne 1690 1778\ne 1690 1800\ne 1690 2034\ne 1691 1692\ne 1691 2032\ne 1692 1693\ne 1692 1977\ne 1692 2120\ne 1693 1989\ne 1693 2130\ne 1693 2131\ne 1694 1695\ne 1695 1696\ne 1696 1931\ne 1696 1943\ne 1697 1698\ne 1697 1699\ne 1697 1741\ne 1697 1760\ne 1697 1783\ne 1697 1805\ne 1697 1956\ne 1697 2040\ne 1698 1699\ne 1698 1790\ne 1698 1804\ne 1698 2056\ne 1698 2076\ne 1699 1732\ne 1699 1740\ne 1699 1843\ne 1700 1701\ne 1700 1710\ne 1700 1711\ne 1700 1741\ne 1700 1764\ne 1700 1766\ne 1700 1828\ne 1701 1702\ne 1701 1744\ne 1701 1756\ne 1701 1769\ne 1701 1886\ne 1701 2024\ne 1702 1744\ne 1702 1825\ne 1702 1977\ne 1702 2045\ne 1702 2063\ne 1702 2064\ne 1703 1704\ne 1703 1862\ne 1704 1705\ne 1704 1732\ne 1704 1733\ne 1704 1768\ne 1704 1861\ne 1705 1765\ne 1705 1777\ne 1705 1832\ne 1705 1861\ne 1705 2001\ne 1705 2107\ne 1706 1707\ne 1706 1709\ne 1706 1711\ne 1706 1765\ne 1706 1794\ne 1706 1863\ne 1706 1930\ne 1706 1936\ne 1706 1956\ne 1706 1958\ne 1706 1962\ne 1706 2057\ne 1706 2124\ne 1707 1708\ne 1707 1712\ne 1707 1814\ne 1707 1934\ne 1707 1999\ne 1708 1870\ne 1709 1710\ne 1710 1711\ne 1710 1727\ne 1710 1729\ne 1710 1876\ne 1711 1828\ne 1711 1872\ne 1711 1875\ne 1711 2006\ne 1711 2042\ne 1712 1713\ne 1712 1999\ne 1713 1725\ne 1714 1715\ne 1714 1720\ne 1714 1747\ne 1714 1770\ne 1714 2067\ne 1715 1716\ne 1715 1986\ne 1716 1977\ne 1717 1718\ne 1717 1752\ne 1717 1843\ne 1718 1719\ne 1718 1747\ne 1718 1748\ne 1718 1749\ne 1720 1758\ne 1720 1769\ne 1720 1973\ne 1720 2075\ne 1721 1722\ne 1721 1725\ne 1721 1727\ne 1722 1723\ne 1722 1766\ne 1722 1797\ne 1722 1821\ne 1722 2031\ne 1722 2034\ne 1723 1724\ne 1723 1755\ne 1723 1765\ne 1723 1783\ne 1723 1797\ne 1723 1804\ne 1723 1849\ne 1723 2003\ne 1723 2012\ne 1723 2035\ne 1723 2078\ne 1724 1754\ne 1724 1898\ne 1725 1727\ne 1725 1756\ne 1725 1876\ne 1726 1756\ne 1726 1816\ne 1726 1832\ne 1726 1864\ne 1726 1879\ne 1726 1899\ne 1726 1916\ne 1726 1946\ne 1726 1957\ne 1726 1972\ne 1726 1975\ne 1726 2024\ne 1726 2031\ne 1726 2046\ne 1726 2108\ne 1727 1728\ne 1728 1729\ne 1729 1734\ne 1729 1738\ne 1729 1747\ne 1730 1731\ne 1730 1734\ne 1731 1732\ne 1731 1734\ne 1731 2071\ne 1732 1733\ne 1732 1777\ne 1732 1860\ne 1732 2057\ne 1733 1798\ne 1733 1806\ne 1733 1808\ne 1733 2010\ne 1733 2011\ne 1733 2015\ne 1734 1735\ne 1734 1736\ne 1735 1736\ne 1736 1969\ne 1736 2071\ne 1737 1738\ne 1737 1740\ne 1737 1779\ne 1737 1829\ne 1738 1739\ne 1738 1747\ne 1738 1778\ne 1738 1779\ne 1738 1843\ne 1739 1939\ne 1740 1741\ne 1740 1994\ne 1741 1742\ne 1741 1764\ne 1741 1886\ne 1741 2009\ne 1742 1761\ne 1742 1765\ne 1742 1766\ne 1742 1832\ne 1742 1930\ne 1742 1975\ne 1743 1744\ne 1743 1792\ne 1743 1793\ne 1743 2026\ne 1744 1745\ne 1744 2025\ne 1745 1793\ne 1745 1973\ne 1745 1999\ne 1745 2025\ne 1745 2067\ne 1746 1747\ne 1746 1749\ne 1746 1903\ne 1747 1748\ne 1748 1779\ne 1748 1785\ne 1748 1903\ne 1748 2051\ne 1748 2067\ne 1748 2129\ne 1749 1750\ne 1749 1784\ne 1749 1841\ne 1749 1843\ne 1749 1903\ne 1750 1790\ne 1750 1801\ne 1750 1930\ne 1750 1963\ne 1751 1752\ne 1751 1753\ne 1751 2004\ne 1752 1753\ne 1752 1856\ne 1753 1785\ne 1753 1855\ne 1753 1878\ne 1753 1892\ne 1753 1960\ne 1753 2004\ne 1754 1755\ne 1754 1756\ne 1754 1816\ne 1754 2053\ne 1755 1756\ne 1755 1773\ne 1755 1832\ne 1755 1885\ne 1756 1848\ne 1757 1758\ne 1757 2063\ne 1758 1759\ne 1758 1874\ne 1758 1918\ne 1758 2006\ne 1758 2063\ne 1758 2075\ne 1759 1902\ne 1760 1761\ne 1760 1762\ne 1760 2003\ne 1760 2039\ne 1760 2084\ne 1761 1762\ne 1761 1975\ne 1762 1763\ne 1762 1764\ne 1762 1828\ne 1762 1979\ne 1762 2009\ne 1762 2019\ne 1762 2077\ne 1762 2114\ne 1762 2122\ne 1763 1769\ne 1763 1839\ne 1763 1844\ne 1763 1973\ne 1763 1981\ne 1763 2024\ne 1763 2065\ne 1763 2086\ne 1763 2108\ne 1764 1765\ne 1764 1769\ne 1764 1771\ne 1765 1766\ne 1765 2114\ne 1765 2115\ne 1766 1950\ne 1766 2001\ne 1767 1768\ne 1767 2084\ne 1768 1769\ne 1768 1808\ne 1768 1981\ne 1769 1786\ne 1769 1848\ne 1769 1886\ne 1769 2045\ne 1770 1771\ne 1771 1772\ne 1772 1794\ne 1772 1962\ne 1772 2006\ne 1773 1774\ne 1773 1775\ne 1773 1831\ne 1773 1938\ne 1773 2035\ne 1773 2056\ne 1773 2113\ne 1774 1775\ne 1774 1846\ne 1774 1885\ne 1774 2011\ne 1774 2027\ne 1774 2032\ne 1775 1778\ne 1775 1779\ne 1775 1908\ne 1775 1937\ne 1775 2023\ne 1775 2094\ne 1776 1777\ne 1776 2054\ne 1776 2113\ne 1777 1860\ne 1778 1782\ne 1779 1829\ne 1779 1842\ne 1779 1969\ne 1779 1977\ne 1779 2008\ne 1780 1781\ne 1780 1782\ne 1781 1782\ne 1782 1908\ne 1782 1987\ne 1783 1784\ne 1783 1788\ne 1783 1843\ne 1784 1785\ne 1784 1795\ne 1784 1800\ne 1784 2074\ne 1784 2093\ne 1785 2098\ne 1786 1797\ne 1787 1794\ne 1787 1797\ne 1787 1798\ne 1787 2012\ne 1788 1789\ne 1789 1790\ne 1789 2036\ne 1790 1909\ne 1790 1956\ne 1790 2076\ne 1791 1792\ne 1791 1795\ne 1792 1793\ne 1792 1856\ne 1793 1802\ne 1793 1855\ne 1793 1866\ne 1793 1886\ne 1793 2013\ne 1793 2026\ne 1794 1795\ne 1794 2105\ne 1795 1796\ne 1797 1798\ne 1798 1799\ne 1798 1821\ne 1798 1849\ne 1798 2001\ne 1798 2015\ne 1798 2037\ne 1798 2057\ne 1798 2064\ne 1799 1809\ne 1799 1836\ne 1799 1847\ne 1799 1851\ne 1799 1881\ne 1799 1888\ne 1799 1925\ne 1799 1932\ne 1799 1981\ne 1799 1984\ne 1799 2043\ne 1799 2059\ne 1799 2086\ne 1799 2125\ne 1800 1801\ne 1801 1802\ne 1801 1914\ne 1801 1921\ne 1801 2074\ne 1801 2089\ne 1801 2119\ne 1802 1821\ne 1802 1928\ne 1802 1929\ne 1802 1930\ne 1802 2042\ne 1803 1804\ne 1803 1814\ne 1804 1805\ne 1804 1889\ne 1804 1909\ne 1804 1927\ne 1804 2022\ne 1804 2118\ne 1805 1886\ne 1806 1807\ne 1806 1846\ne 1806 1860\ne 1806 1861\ne 1806 1913\ne 1807 1808\ne 1807 1839\ne 1807 1924\ne 1807 1928\ne 1807 1981\ne 1808 1942\ne 1809 1810\ne 1809 1919\ne 1809 1963\ne 1809 2110\ne 1811 1812\ne 1811 1852\ne 1811 1899\ne 1811 1901\ne 1811 1931\ne 1812 1813\ne 1812 1884\ne 1812 1899\ne 1812 1946\ne 1812 1968\ne 1812 1997\ne 1812 2010\ne 1813 1823\ne 1813 1862\ne 1813 1912\ne 1813 1915\ne 1813 1932\ne 1813 1987\ne 1813 2001\ne 1813 2070\ne 1813 2071\ne 1813 2084\ne 1813 2126\ne 1814 1978\ne 1815 1816\ne 1815 1829\ne 1815 1831\ne 1815 1881\ne 1815 1940\ne 1815 1993\ne 1816 1817\ne 1816 1832\ne 1816 1896\ne 1816 1993\ne 1816 2053\ne 1817 1829\ne 1817 1845\ne 1817 1903\ne 1817 1964\ne 1817 1965\ne 1817 1994\ne 1817 2004\ne 1817 2010\ne 1817 2026\ne 1817 2046\ne 1817 2093\ne 1818 1819\ne 1818 1995\ne 1818 1997\ne 1818 2000\ne 1818 2002\ne 1818 2090\ne 1819 1928\ne 1819 1999\ne 1819 2005\ne 1819 2018\ne 1819 2044\ne 1819 2067\ne 1820 1944\ne 1820 2004\ne 1820 2007\ne 1820 2009\ne 1820 2058\ne 1821 1849\ne 1821 1950\ne 1821 2010\ne 1821 2013\ne 1821 2068\ne 1822 1823\ne 1822 1861\ne 1822 1862\ne 1822 2016\ne 1822 2018\ne 1822 2020\ne 1822 2021\ne 1823 1938\ne 1824 1833\ne 1824 1967\ne 1824 2023\ne 1824 2092\ne 1825 1826\ne 1825 1835\ne 1825 1971\ne 1825 2024\ne 1825 2025\ne 1826 1910\ne 1826 1919\ne 1826 1977\ne 1826 2052\ne 1826 2120\ne 1827 1828\ne 1827 1829\ne 1827 2041\ne 1827 2043\ne 1827 2046\ne 1827 2049\ne 1827 2051\ne 1828 1863\ne 1828 1950\ne 1828 1996\ne 1828 2009\ne 1828 2024\ne 1828 2042\ne 1829 1830\ne 1829 1994\ne 1829 2051\ne 1830 1881\ne 1830 1882\ne 1830 1910\ne 1830 2018\ne 1830 2021\ne 1830 2023\ne 1830 2034\ne 1830 2043\ne 1830 2044\ne 1830 2100\ne 1830 2128\ne 1831 1832\ne 1831 2055\ne 1831 2077\ne 1832 1849\ne 1832 2001\ne 1832 2112\ne 1832 2117\ne 1833 1834\ne 1833 1952\ne 1833 2069\ne 1833 2102\ne 1834 2021\ne 1834 2023\ne 1834 2069\ne 1834 2080\ne 1834 2091\ne 1834 2099\ne 1835 1836\ne 1835 1850\ne 1835 1994\ne 1835 2026\ne 1835 2072\ne 1836 1851\ne 1836 1887\ne 1836 2027\ne 1836 2072\ne 1836 2120\ne 1837 1838\ne 1837 1852\ne 1837 2039\ne 1837 2076\ne 1838 1839\ne 1838 1851\ne 1838 1864\ne 1838 1918\ne 1838 2029\ne 1838 2037\ne 1839 1859\ne 1839 1865\ne 1839 2027\ne 1839 2033\ne 1839 2088\ne 1839 2089\ne 1840 1841\ne 1840 1842\ne 1840 1844\ne 1840 1914\ne 1840 1918\ne 1840 1927\ne 1840 2086\ne 1840 2087\ne 1840 2088\ne 1840 2090\ne 1840 2092\ne 1841 1842\ne 1841 1845\ne 1841 1903\ne 1841 1919\ne 1841 2074\ne 1842 1843\ne 1842 2008\ne 1844 1845\ne 1844 1886\ne 1844 1980\ne 1844 2009\ne 1844 2024\ne 1845 1878\ne 1845 1948\ne 1845 2007\ne 1845 2074\ne 1845 2117\ne 1846 2058\ne 1846 2094\ne 1846 2113\ne 1847 1849\ne 1847 1897\ne 1847 1945\ne 1847 2002\ne 1847 2072\ne 1847 2106\ne 1847 2108\ne 1847 2110\ne 1847 2112\ne 1847 2114\ne 1847 2116\ne 1848 2108\ne 1849 1927\ne 1849 2003\ne 1849 2011\ne 1849 2012\ne 1849 2014\ne 1849 2065\ne 1849 2085\ne 1849 2114\ne 1850 1851\ne 1850 1891\ne 1850 2117\ne 1851 1854\ne 1851 2094\ne 1851 2107\ne 1852 2021\ne 1852 2119\ne 1852 2120\ne 1853 1854\ne 1853 1904\ne 1853 2037\ne 1853 2077\ne 1853 2122\ne 1854 1869\ne 1854 1894\ne 1854 1901\ne 1854 1919\ne 1854 1998\ne 1854 2038\ne 1854 2082\ne 1854 2121\ne 1854 2123\ne 1855 1856\ne 1855 1866\ne 1855 1928\ne 1855 2088\ne 1857 1858\ne 1857 1907\ne 1857 1947\ne 1857 2016\ne 1857 2029\ne 1857 2041\ne 1858 1859\ne 1858 1871\ne 1858 1894\ne 1858 1905\ne 1858 1914\ne 1858 1918\ne 1858 1925\ne 1859 1865\ne 1859 2127\ne 1860 1861\ne 1860 1893\ne 1860 1952\ne 1860 2069\ne 1860 2071\ne 1860 2113\ne 1861 1862\ne 1861 1981\ne 1861 2115\ne 1862 1863\ne 1862 1913\ne 1862 1915\ne 1863 1921\ne 1863 1936\ne 1863 1959\ne 1863 1972\ne 1863 1983\ne 1863 1999\ne 1863 2020\ne 1863 2040\ne 1863 2049\ne 1863 2115\ne 1864 1865\ne 1864 1867\ne 1864 1869\ne 1864 1871\ne 1864 1872\ne 1864 1875\ne 1864 1993\ne 1864 1995\ne 1864 2080\ne 1864 2106\ne 1865 1866\ne 1865 1869\ne 1865 1922\ne 1865 1943\ne 1866 1872\ne 1867 1868\ne 1867 1875\ne 1867 1912\ne 1867 1933\ne 1867 2001\ne 1868 1880\ne 1868 1912\ne 1868 1932\ne 1869 1870\ne 1869 1901\ne 1869 1967\ne 1869 1970\ne 1869 1989\ne 1869 1996\ne 1869 2019\ne 1869 2042\ne 1870 1902\ne 1870 2121\ne 1871 1915\ne 1871 1932\ne 1871 1971\ne 1871 1972\ne 1871 1989\ne 1871 2016\ne 1871 2079\ne 1872 1873\ne 1872 1874\ne 1872 1958\ne 1872 1975\ne 1873 1874\ne 1873 1953\ne 1873 1974\ne 1875 1876\ne 1875 2083\ne 1877 1879\ne 1877 1880\ne 1877 1882\ne 1877 1917\ne 1877 2096\ne 1877 2100\ne 1878 1901\ne 1878 1907\ne 1878 2004\ne 1879 1896\ne 1879 2070\ne 1879 2101\ne 1879 2103\ne 1879 2112\ne 1880 1881\ne 1880 1882\ne 1880 1907\ne 1880 1933\ne 1881 2094\ne 1881 2100\ne 1881 2120\ne 1882 1988\ne 1882 1989\ne 1882 2079\ne 1883 1884\ne 1883 1922\ne 1883 1973\ne 1883 2102\ne 1884 1912\ne 1884 1919\ne 1884 1947\ne 1884 2117\ne 1884 2129\ne 1885 1886\ne 1885 2015\ne 1887 2131\ne 1888 1889\ne 1888 1946\ne 1888 2027\ne 1889 1890\ne 1889 2086\ne 1889 2118\ne 1890 2086\ne 1891 1893\ne 1891 1896\ne 1891 1926\ne 1891 2087\ne 1892 2047\ne 1893 1894\ne 1893 1925\ne 1893 1951\ne 1893 1998\ne 1893 2101\ne 1894 1895\ne 1895 1897\ne 1895 1918\ne 1895 2030\ne 1895 2054\ne 1895 2087\ne 1895 2101\ne 1895 2106\ne 1896 1897\ne 1896 2004\ne 1896 2112\ne 1897 1898\ne 1897 1940\ne 1897 2098\ne 1898 2075\ne 1899 1900\ne 1899 1955\ne 1900 1901\ne 1900 1955\ne 1900 1968\ne 1901 1902\ne 1901 1928\ne 1901 1968\ne 1901 2005\ne 1901 2119\ne 1901 2129\ne 1902 2121\ne 1903 1997\ne 1903 2090\ne 1903 2093\ne 1903 2129\ne 1904 1905\ne 1904 1940\ne 1904 1986\ne 1905 1921\ne 1905 1933\ne 1905 1947\ne 1905 1951\ne 1905 1972\ne 1905 1973\ne 1905 1986\ne 1906 1907\ne 1906 1974\ne 1906 1987\ne 1906 2004\ne 1906 2010\ne 1906 2069\ne 1906 2070\ne 1907 1908\ne 1907 2094\ne 1907 2112\ne 1907 2117\ne 1908 1912\ne 1908 1987\ne 1908 2082\ne 1909 1927\ne 1909 2012\ne 1909 2061\ne 1910 1911\ne 1910 2017\ne 1910 2055\ne 1910 2110\ne 1910 2111\ne 1910 2119\ne 1911 2041\ne 1911 2043\ne 1911 2119\ne 1912 2080\ne 1912 2082\ne 1912 2085\ne 1913 1914\ne 1913 1921\ne 1913 1944\ne 1913 2126\ne 1914 1915\ne 1914 2089\ne 1914 2126\ne 1915 2079\ne 1915 2102\ne 1916 1919\ne 1916 1957\ne 1916 2047\ne 1916 2048\ne 1916 2083\ne 1916 2095\ne 1917 1918\ne 1917 1950\ne 1917 1989\ne 1918 1973\ne 1918 2063\ne 1919 1920\ne 1919 1997\ne 1919 2082\ne 1919 2094\ne 1919 2110\ne 1920 1921\ne 1920 1947\ne 1920 2039\ne 1920 2040\ne 1920 2074\ne 1921 1951\ne 1921 1983\ne 1922 1924\ne 1922 1967\ne 1922 2032\ne 1923 1938\ne 1924 1929\ne 1924 2073\ne 1924 2084\ne 1924 2126\ne 1925 1959\ne 1925 1982\ne 1925 1983\ne 1925 2063\ne 1925 2127\ne 1926 1927\ne 1926 1971\ne 1926 1990\ne 1926 2049\ne 1927 1991\ne 1927 2017\ne 1927 2022\ne 1927 2050\ne 1927 2065\ne 1927 2076\ne 1927 2087\ne 1928 1929\ne 1928 1934\ne 1928 1960\ne 1928 2055\ne 1928 2088\ne 1928 2103\ne 1928 2119\ne 1929 1947\ne 1929 1986\ne 1929 2041\ne 1929 2068\ne 1929 2073\ne 1929 2093\ne 1929 2129\ne 1930 1949\ne 1930 1956\ne 1930 1960\ne 1931 1932\ne 1931 1980\ne 1931 2112\ne 1932 1981\ne 1932 2001\ne 1932 2112\ne 1933 1934\ne 1933 1937\ne 1933 1952\ne 1933 1978\ne 1933 2000\ne 1933 2007\ne 1933 2065\ne 1933 2117\ne 1934 1935\ne 1934 1936\ne 1934 1978\ne 1934 1999\ne 1935 1936\ne 1935 1972\ne 1935 2103\ne 1936 1945\ne 1936 1952\ne 1936 2042\ne 1936 2062\ne 1936 2114\ne 1937 1938\ne 1937 1939\ne 1937 1985\ne 1937 2008\ne 1937 2130\ne 1938 1939\ne 1940 2051\ne 1940 2052\ne 1941 1985\ne 1941 2126\ne 1942 2124\ne 1943 1982\ne 1943 1995\ne 1943 1996\ne 1943 2024\ne 1943 2027\ne 1943 2127\ne 1944 1945\ne 1944 2129\ne 1945 2041\ne 1945 2049\ne 1945 2055\ne 1945 2076\ne 1946 1947\ne 1946 1964\ne 1946 2025\ne 1946 2026\ne 1946 2084\ne 1947 1948\ne 1947 2085\ne 1948 1964\ne 1948 2007\ne 1949 1957\ne 1949 1969\ne 1949 1970\ne 1949 1976\ne 1950 1951\ne 1950 2114\ne 1951 1952\ne 1951 2104\ne 1952 2057\ne 1954 1955\ne 1954 2081\ne 1955 1975\ne 1955 2095\ne 1956 2040\ne 1957 1964\ne 1958 2105\ne 1958 2124\ne 1959 1961\ne 1960 1961\ne 1960 1962\ne 1960 1963\ne 1960 1966\ne 1961 1962\ne 1961 2006\ne 1963 2119\ne 1964 1965\ne 1965 2000\ne 1965 2007\ne 1965 2025\ne 1965 2052\ne 1965 2066\ne 1966 2020\ne 1966 2124\ne 1967 1969\ne 1967 2018\ne 1969 1970\ne 1969 2018\ne 1970 2019\ne 1971 1972\ne 1972 1999\ne 1972 2024\ne 1973 2065\ne 1973 2067\ne 1973 2104\ne 1974 2101\ne 1975 2031\ne 1976 1977\ne 1978 1979\ne 1979 2114\ne 1979 2116\ne 1980 1981\ne 1980 1982\ne 1980 2039\ne 1981 1982\ne 1981 2084\ne 1981 2115\ne 1982 1983\ne 1982 2024\ne 1983 2027\ne 1983 2039\ne 1984 2043\ne 1984 2045\ne 1984 2116\ne 1985 2015\ne 1986 2027\ne 1986 2052\ne 1986 2067\ne 1986 2089\ne 1987 2071\ne 1987 2130\ne 1988 2097\ne 1989 1999\ne 1989 2055\ne 1990 1991\ne 1990 1992\ne 1990 2049\ne 1991 1992\ne 1991 2050\ne 1992 2056\ne 1992 2097\ne 1992 2130\ne 1993 1994\ne 1993 2029\ne 1993 2054\ne 1994 2009\ne 1994 2092\ne 1995 1996\ne 1995 2001\ne 1995 2037\ne 1995 2107\ne 1996 1998\ne 1996 2005\ne 1997 1998\ne 1997 2071\ne 1997 2128\ne 1998 2005\ne 1998 2071\ne 1999 2000\ne 2000 2001\ne 2000 2010\ne 2000 2066\ne 2002 2003\ne 2002 2049\ne 2002 2107\ne 2002 2115\ne 2002 2128\ne 2003 2022\ne 2003 2066\ne 2003 2084\ne 2003 2115\ne 2004 2005\ne 2004 2098\ne 2005 2044\ne 2007 2008\ne 2007 2065\ne 2009 2040\ne 2009 2092\ne 2010 2011\ne 2010 2013\ne 2010 2117\ne 2011 2014\ne 2012 2061\ne 2013 2014\ne 2013 2015\ne 2014 2015\ne 2015 2125\ne 2016 2017\ne 2016 2055\ne 2016 2062\ne 2016 2103\ne 2016 2112\ne 2017 2021\ne 2018 2019\ne 2018 2049\ne 2018 2055\ne 2020 2062\ne 2020 2115\ne 2021 2022\ne 2021 2079\ne 2022 2066\ne 2022 2090\ne 2022 2091\ne 2023 2092\ne 2023 2094\ne 2023 2111\ne 2023 2128\ne 2024 2025\ne 2025 2026\ne 2025 2052\ne 2026 2027\ne 2026 2073\ne 2027 2032\ne 2027 2037\ne 2027 2067\ne 2027 2073\ne 2027 2127\ne 2028 2029\ne 2028 2031\ne 2028 2032\ne 2028 2034\ne 2028 2037\ne 2028 2099\ne 2028 2100\ne 2029 2030\ne 2029 2033\ne 2029 2092\ne 2029 2094\ne 2030 2031\ne 2030 2087\ne 2031 2091\ne 2031 2101\ne 2032 2033\ne 2032 2052\ne 2032 2066\ne 2032 2120\ne 2033 2065\ne 2034 2035\ne 2035 2036\ne 2035 2078\ne 2035 2116\ne 2036 2116\ne 2037 2038\ne 2037 2063\ne 2037 2122\ne 2038 2123\ne 2039 2040\ne 2039 2073\ne 2040 2076\ne 2041 2042\ne 2041 2072\ne 2041 2092\ne 2043 2044\ne 2044 2051\ne 2044 2098\ne 2044 2119\ne 2045 2059\ne 2045 2060\ne 2045 2075\ne 2045 2131\ne 2046 2047\ne 2047 2048\ne 2049 2050\ne 2051 2052\ne 2051 2098\ne 2053 2095\ne 2054 2106\ne 2054 2113\ne 2056 2057\ne 2058 2059\ne 2058 2063\ne 2058 2065\ne 2059 2060\ne 2059 2063\ne 2060 2064\ne 2060 2065\ne 2061 2065\ne 2062 2114\ne 2063 2064\ne 2063 2090\ne 2063 2101\ne 2064 2065\ne 2065 2066\ne 2067 2088\ne 2067 2090\ne 2068 2093\ne 2068 2096\ne 2069 2070\ne 2069 2071\ne 2070 2101\ne 2070 2102\ne 2071 2082\ne 2072 2073\ne 2072 2111\ne 2074 2093\ne 2076 2092\ne 2076 2128\ne 2077 2079\ne 2077 2082\ne 2077 2084\ne 2079 2080\ne 2080 2081\ne 2080 2096\ne 2082 2083\ne 2084 2085\ne 2086 2109\ne 2086 2125\ne 2086 2131\ne 2087 2091\ne 2087 2109\ne 2088 2089\ne 2090 2091\ne 2091 2099\ne 2091 2101\ne 2092 2093\ne 2093 2098\ne 2094 2113\ne 2094 2120\ne 2096 2098\ne 2096 2099\ne 2097 2116\ne 2098 2099\ne 2099 2100\ne 2100 2101\ne 2101 2104\ne 2101 2107\ne 2101 2113\ne 2102 2103\ne 2102 2104\ne 2103 2104\ne 2104 2105\ne 2106 2107\ne 2106 2108\ne 2107 2113\ne 2108 2109\ne 2110 2111\ne 2110 2128\ne 2112 2113\ne 2114 2115\ne 2122 2123\ne 2122 2124\ne 2123 2124\ne 2125 2127\ne 2126 2129\n"
VTXG3 = "[[-1.0, 1.0408340855860843e-16], [-0.3765101981412665, -0.7818314824680297], [0.41239324334649896, -0.16731430613464227], [0.9123932433464991, 0.6987110976497966], [0.33967068575833653, -0.121038176231964], [0.500000000000002, 0.8660254037844397], [-0.13974330102097354, -0.5098611692036452], [0.7952434915444768, -0.1551786383000625], [1.2952434915444773, 0.7108467654843759], [0.7225209339563146, -0.10890250839738473], [-0.04442719421385943, -0.2947551744109042], [0.26122556688144516, 0.6573878766243993], [-0.3622642349772881, -0.12444360584363032], [-0.500000000000002, 0.8660254037844397], [-0.8396706857583365, 0.9870635800164027], [-0.3396706857583362, 1.8530889838008409], [0.5612981821440828, 2.2869727229183994], [-0.17175368968574334, 1.6067999851474801], [-1.137735765022712, 0.9904690096280693], [-0.6377357650227119, 1.8564944134125079], [0.05076724427057089, 2.581727897329304], [-1.3653410243663948, 0.9308737486442044], [-0.38793457358534633, 1.1422422183164662], [-1.3602566989790266, 1.3758865729880843], [-1.5, 0.8660254037844387], [-0.7110965585122344, 1.480542580117826], [0.28890344148776537, 1.4805425801178262], [-1.6234898018587332, 0.7818314824680299], [-0.7632331028797068, 0.27197031326438426], [-0.26323310287970647, 1.1379957170488229], [-0.1885030092932825, 0.14079191986764267], [-1.7330518718298262, 0.6801727377709195], [-0.8727951728507999, 0.17031156856727397], [0.12720482714920012, 0.17031156856727386], [-0.4962849747095332, 0.9521430510353037], [-1.774657478323787, 0.6323810491128204], [-1.3510950612024157, -0.27348587201980457], [-1.488830826225128, 0.716983137608264], [-0.4888308262251281, 0.7169831376082637], [-1.8262387743159947, 0.5633200580636221], [-1.326238774315995, 1.4293454618480608], [-0.8262387743159948, 0.563320058063622], [-5.551115123125783e-17, 2.7755575615628914e-17], [-1.900968867902419, 0.433883739117558], [-0.9659820753369686, 0.7885662700211411], [0.011424375444079815, 0.9999347396934029], [-1.4009688679024193, 1.2999091429019964], [-0.42356241712137077, 1.5112776125742586], [-0.43498679256545064, 0.511342872880856], [-1.97232212539368, 0.23364435467161776], [-1.7498011914373657, 1.2085722668534413], [-1.3262387743159947, 0.3027053457208162], [-0.3262387743159949, 0.30270534572081664], [-2.3376631497600746, 1.164518103315822], [-1.477406450781048, 0.6546569341121767], [-0.4774064507810482, 0.6546569341121768], [-1.9888308262251284, 0.14904226617617466], [-1.0114243754440801, 0.3604107358484366], [-0.5114243754440801, 1.226436139632875], [-0.28890344148776576, 0.25150822745105145], [-1.9009688679024195, -0.43388373911755773], [-1.4774064507810483, -1.339750660250183], [-0.9774064507810484, -0.4737252564657444], [-1.826238774315995, -0.5633200580636221], [-0.9659820753369682, -1.0731812272672676], [-0.8056527610953048, -0.0861176472508646], [-1.7330518718298265, -0.6801727377709192], [-1.0612981821440828, -1.4209473191339608], [-0.16032931424166352, -0.9870635800164026], [-1.572722557588163, -0.8197492738817607], [-0.6377357650227122, -0.46506674297817735], [-0.13773576502271212, 0.4009586608062611], [-0.3602566989790263, 1.3758865729880845], [-1.4123932433464992, 0.16731430613464254], [-0.9123932433464992, 1.033339709919081], [-1.4979924640017386, 0.17745452329941977], [-0.5205860132206903, 0.38882299297168155], [-1.7952434915444768, 0.15517863830006318], [-0.8602566989790262, 0.5098611692036462], [-1.365341024366395, -0.9308737486442041], [-1.6491601404667924, 0.028004089944210164], [-0.6491601404667923, 0.028004089944210206], [-0.4888308262251283, 1.0150676699606132], [-1.0114243754440797, -0.9999347396934024], [-0.05585156965793914, -1.2946899141043064], [0.9215548811231093, -1.0833214444320447], [0.6377357650227125, -0.12444360584363029], [-0.5114243754440798, -0.13390933590896398], [0.4659820753369688, 0.07745913376329817], [-1.7444762472739057, -0.31976200192248294], [-1.2444762472739062, 0.5462634018619557], [-0.925269906413576, -0.9972037971811801], [-0.06501320743454925, -1.5070649663848257], [-0.6885030092932829, -0.7252334839167959], [-0.7774790660436861, -0.9749279121818236], [-0.27747906604368583, -0.1089025083973848], [0.6999273847373626, 0.102465961274877], [-0.6346589756336057, -0.9308737486442045], [0.30032781693184485, -0.5761912177406217], [-0.13465897563360546, -0.06484834485976568], [0.48883082622512825, 0.7169831376082639], [-0.5764375828786295, -0.905866921132625], [0.21246585860913658, -0.2913497447992376], [0.7124658586091364, 0.5746756589852008], [-0.651167676465053, 0.09133687604855498], [-0.5, -0.8660254037844386], [0.3716810744231065, -0.3759518332946817], [0.7952434915444775, -1.2818187544273068], [0.9555728057861408, -0.294755174410904], [1.455572805786141, 0.5712702293735346], [0.6234898018587334, -0.7818314824680297], [-0.1511676764650539, -0.1494504333552093], [0.21417334790134157, 0.7814233152889947], [-0.21618088389960288, 0.2052320975483732], [0.7838191161003969, 0.2052320975483733], [1.772649942325526, 0.056189831372198684], [0.7838191161003976, -0.09285243480397559], [-1.1511676764650534, -0.14945043335520924], [-0.7858266520986583, 0.7814233152889951], [-0.9235624171213702, 1.7718923249170642], [0.0764375828786295, 1.7718923249170642], [-0.32824631031425633, -0.7407745813630414], [0.5727225575881629, -0.30689084224548346], [0.4349867925654509, 0.6835781673825857], [0.5953161068071144, 1.6706417473989887], [0.36025669897902635, -0.015541097446245433], [1.093308570808853, -0.695713835217165], [1.5933085708088526, 0.17031156856727378], [-0.2535162167278322, 0.2564292158181386], [0.7464837832721678, 0.25642921581813877], [1.0521365443674724, 1.2085722668534418], [0.2774790660436852, 1.8409533159662623], [-0.40297640390068035, 0.2564292158181386], [0.49799246400173847, -0.1774545232994193], [-0.0747300935864243, -0.9972037971811801], [-0.4659820753369682, 0.24969442826502775], [0.5303028993725654, 0.16357678101416318], [-0.042419658215597456, -0.6561724928675973], [-0.6120654264146533, 0.2181032572253728], [-0.11206542641465322, 1.0841286610098113], [-0.8451172982444796, 0.40395592323889207], [-0.40096886790241915, 1.2999091429019969], [-0.26694812817017377, -0.6801727377709192], [0.7330518718298262, -0.6801727377709192], [-0.12720482714920012, -0.17031156856727353], [0.3727951728508001, 0.6957138352171646], [0.7104583226108747, -0.4688042680986573], [1.3989613319041574, 0.2564292158181387], [1.8989613319041574, 1.1224546196025773], [1.326238774315995, 0.30270534572081653], [0.4215548811231089, 0.04506074614587685], [-0.5050843253873684, 0.4210125794405588], [0.4949156746126313, 0.42101257944055864], [-0.5507672442705709, 0.27870510081749506], [-0.050767244270570666, 1.1447305046019336], [0.17175368968574356, 2.119658416783757], [-0.1737612256840051, -0.5633200580636221], [0.8262387743159949, -0.5633200580636224], [0.6885030092932831, 0.4271489515644469], [0.06501320743454958, -0.35468253090358276], [0.7612255668814454, -0.20863752716003892], [1.2612255668814456, 0.6573878766243992], [2.1214822658604717, 0.14752670742075402], [1.1234898018587338, 0.0841939213164091], [0.326238774315995, 0.3027053457208164], [-0.6717536896857434, 0.23937255961647197], [-1.2952434915444768, 1.021204042084502], [0.8602566989790266, -0.5098611692036455], [0.9349867925654507, -1.5070649663848257], [0.03401792466303144, -1.0731812272672676], [0.19434723890469496, -0.0861176472508646], [0.8376631497600747, -0.29849269953138335], [0.21417334790134163, 0.4833387829366464], [0.9888308262251284, -0.14904226617617378], [0.36025669897902657, 0.35616423458079316], [-0.4659820753369681, -0.20715582348282907], [0.4349867925654513, -0.6410395626003871], [0.16590946007433094, 0.44228188183165795], [-0.406813097513832, -0.3774673920501024], [0.2649405921719119, -1.1182419734131441], [0.02058601322068998, 0.4772024108127572], [1.02058601322069, 0.47720241081275716], [1.095316106807114, -0.520001386368423], [1.400968867902419, 0.43214166466688086], [-0.09903113209758085, -0.4338837391175581], [0.9009688679024193, -0.43388373911755784], [-0.07135325749126087, -0.20023938444594025], [0.6171497518020221, 0.5249940994708557], [-0.3716810744231066, 0.37595183329468146], [0.7726499423255255, 0.056189831372198684], [1.2726499423255255, -0.8098355724122399], [0.2838191161003971, -0.9588778385884145], [0.40096886790241926, 0.43214166466688053], [0.0953161068071145, -0.5200013863684229], [-0.1885030092932825, 0.4388764522199912], [-0.3262387743159946, 1.4293454618480608], [0.3451172982444797, 0.46206948054554675], [-0.3879345735853468, -0.2181032572253725], [0.11206542641465372, 0.647922146559066], [0.932979256567189, -0.08338670473864229], [1.7218826980549546, 0.5311304715947451], [1.710458322610875, -0.4688042680986573], [0.7444762472739062, 0.3197620019224833], [1.744476247273906, 0.31976200192248305], [0.8178370407634286, 0.6957138352171652], [0.45557280578614073, 0.5712702293735343], [0.3178370407634288, 1.5617392390016034], [-0.3056527610953047, 0.7799077565335738], [-0.1821629592365714, 0.6957138352171652], [-1.137735765022712, 0.4009586608062609], [-0.011169173774871477, -0.1490422661761745], [0.6773338355184111, 0.5761912177406217], [0.05384403365967794, -0.2056402647274082], [0.9888308262251286, 0.14904226617617455], [1.4888308262251284, 1.015067669960613], [-0.0037150252904665226, -0.08611764725086457], [0.49628497470953326, -0.9521430510353031], [0.21246585860913614, 0.006734787553111188], [0.07473009358642424, 0.9972037971811805], [0.5747300935864244, 1.8632292009656188], [0.9962849747095334, -0.08611764725086468], [0.06964576819905577, 0.28983418604381717], [0.4349867925654507, 1.2207079346880212], [0.9774064507810483, 0.21136846967226194], [1.837663149760075, -0.29849269953138363], [0.8488323235349462, -0.14945043335520936], [-0.12348980185873365, 0.08419392131640882], [0.8262387743159947, 0.5633200580636217], [0.2535162167278321, -0.25642921581813855], [-0.24648378327216797, 0.6095961879663001], [0.42526990641357554, -0.13117839339674142], [-0.9774064507810483, -0.21136846967226244], [-0.6120654264146541, -1.1422422183164669], [-0.11206542641465389, -0.2762168145320285], [0.13773576502271223, -0.990469009628069], [0.9555728057861401, 0.2947551744109042], [1.4555728057861406, 1.160780578195343], [-0.011169173774871477, 0.1490422661761745], [-0.5838917313630343, -0.670707007705586], [-1.3585492096868212, -0.03832595859276547], [-0.35854920968682125, -0.03832595859276561], [-0.02259354921895146, 0.2113684696722622], [0.9123932433464989, 0.5660510005758448], [0.8376631497600747, 1.563254797757025], [-0.13465897563360546, 1.7968991524286428], [0.8653410243663949, 1.7968991524286428], [-0.04442719421385921, 0.2947551744109042], [-0.6171497518020221, -0.5249940994708562], [-0.11714975180202214, 0.34103130431358264], [0.043179562439641606, 1.3280948843299853], [-0.4009688679024195, 0.43214166466688053], [-0.06501320743454941, 0.354682530903583], [-1.02058601322069, 0.6494377053144871], [-0.1489049387975837, 1.1395112758042438], [0.8510950612024164, 1.1395112758042438], [-0.09903113209758085, 0.433883739117558], [0.9009688679024193, 0.43388373911755773], [0.1263113895786323, 1.0662647882303784], [1.0612981821440826, 1.420947319133961], [0.1346589756336044, 1.7968991524286428], [-0.5431795624396419, -0.4620694805455466], [-0.3828502481979778, 0.5249940994708562], [0.1171497518020218, 1.3910195032552948], [-0.45557280578614073, 0.5712702293735346], [-0.12831892557689362, 0.49007357048975686], [0.8716810744231063, 0.4900735704897571], [0.6491601404667919, 1.4650014826715805], [-0.27747906604368566, 1.8409533159662625], [0.7225209339563148, 1.8409533159662623], [-0.1737612256840051, 0.5633200580636218], [-0.7464837832721678, -0.2564292158181386], [1.2498011914373655, -0.34254686306900334], [0.2774790660436859, -0.10890250839738491], [-0.21109655851223452, 0.6145171763333875], [-1.1666693642983752, 0.31976200192248344], [-0.9441484303420609, 1.294689914104307], [-0.7838191161003975, 2.28175349412071], [-0.26694812817017366, 0.6801727377709194], [0.5933085708088528, 0.1703115685672738], [1.0933085708088526, 1.0363369723517124], [0.137735765022712, 1.3310921467626167], [1.0727225575881627, 1.6857746776661993], [-0.3114969907067173, 0.7252334839167962], [0.47740645078104804, 1.3397506602501836], [0.402676357194623, 0.3425468630690035], [-0.42356241712137177, 0.9058669211326256], [0.11206542641465378, -0.18063343721582928], [-0.6625920519091332, 0.45174761189699153], [0.2723947406563174, 0.8064301428005746], [-0.3765101981412665, 0.7818314824680299], [0.6234898018587338, 0.7818314824680297], [0.3396706857583362, 1.740709321056444], [-0.025670338608059062, 0.8098355724122399], [0.42663920651047776, 0.49007357048975675], [0.9266392065104776, 1.3560989742741956], [0.19358733468065137, 0.6759262365032759], [0.8653410243663953, -0.06484834485976558], [0.4888308262251284, 1.0150676699606132], [0.9123932433464996, 0.1092007488279878], [1.4123932433464996, -0.7568246549564508], [0.47740645078104854, 1.0773938734567006], [0.11206542641465389, 2.008267622100905], [0.9723221253936802, 1.4984064528972594], [-1.1102230246251565e-16, 1.7320508075688776], [0.45557280578614057, 1.160780578195343], [0.011424375444079593, 0.26482735853223816], [-0.5612981821440829, -0.5549219153495224], [0.23305187182982612, 1.546198141555358], [0.0727225575881627, 0.5591345615389549], [1.072722557588163, 0.5591345615389549], [0.18850300929328268, 1.5912588877012346], [1.1773338355184113, 1.4422166215250598], [1.1659094600743316, 0.4422818818316576], [-1.3716810744231065, 0.37595183329468174], [-0.41610826863696604, 0.6707070077055863], [0.5727225575881627, 0.8197492738817607], [-1.6773338355184109, -0.5761912177406214], [-0.634658975633605, 0.9308737486442042], [-0.6460833510776844, -0.06906099104919824], [-0.1460833510776846, 0.7969644127352402], [0.8313230997033638, 1.0083328824075022], [-0.6943472389046955, 0.9521430510353035], [-0.19434723890469552, 0.08611764725086496], [-0.034017924663031884, 1.073181227267268], [0.4659820753369681, 1.9392066310517064], [1.326238774315995, 1.4293454618480608], [0.30565276109530476, 0.9521430510353034], [-0.52058601322069, 1.5154631090989255], [0.21246585860913592, 2.1956358468698447], [-0.7774790660436857, 0.9749279121818235], [-0.22363503238400761, 0.14230747862306364], [-1.2124658586091361, 0.29134974479923825], [-0.7889034414877658, -0.614517176333387], [0.22252093395631428, 0.9749279121818236], [1.2188059086658476, 0.8888102649309589], [0.6460833510776848, 0.06906099104919844], [1.4723221253936798, 0.6323810491128203], [0.14916014046679216, 0.8380213138402284], [0.5727225575881629, -0.06784560729239662], [0.7952434915444779, 0.907082304889427], [0.1603293142416634, 0.9870635800164026], [-0.5727225575881629, 0.30689084224548346], [-0.7952434915444769, 1.2818187544273072], [0.137735765022712, 1.198432049688665], [-0.5953161068071143, 0.5182593119177455], [0.36025669897902635, 0.8130144863286499], [0.06129818214408289, 1.4209473191339608], [0.9215548811231091, 0.9110861499303154], [0.2330518718298265, 0.18585266601351919], [0.03201038866477002, 1.4771371505061597], [1.0320103886647702, 1.4771371505061597], [1.3376631497600748, 2.429280201541463], [0.5114243754440798, 1.865960143477841], [-0.05076724427057083, 1.6015807563497906], [0.8209138301525354, 2.0916543268395476], [0.8094894547084561, 1.091719587146145], [0.171753689685743, 0.6266528441679668], [1.1491601404667917, 0.8380213138402286], [0.025670338608057897, 0.9222152351566368], [0.997992464001738, 0.6885708804850192], [-0.5340179246630319, 1.9392066310517064], [-1.434986792565451, 2.3730903701692645], [-0.4349867925654509, 2.3730903701692645], [0.42526990641357554, 1.8632292009656188], [-0.9774064507810484, 1.9775325896444724], [-0.5538440336596776, 1.071665668511847], [-1.1773338355184109, 0.2898341860438171], [-0.925269906413576, 0.9972037971811802], [0.05213654436747228, 1.2085722668534422], [1.0501290083692107, 1.271905052957787], [0.12348980185873348, 1.647856886252469], [1.123489801858734, 1.6478568862524683], [-0.42526990641357587, 1.8632292009656188], [0.5521365443674728, 2.074597670637881], [0.7746574783237867, 1.0996697584560573], [-0.7649405921719126, 1.984267377197583], [-0.09318690248616879, 1.2434927958345414], [0.7330518718298262, 0.6801727377709192], [1.2330518718298265, 1.5461981415553576], [-1.0747300935864241, 0.9972037971811801], [-0.09732364280537575, 1.2085722668534422], [0.40267635719462436, 2.0745976706378806], [0.9026763571946244, 1.2085722668534422], [-0.31149699070671666, 0.4271489515644471], [-0.1511676764650533, 1.4142125315808498], [-0.774657478323787, 0.6323810491128203], [-0.14145079031317864, 0.9043513623772046], [0.8585492096868212, 0.9043513623772044], [-0.09702359609931932, 0.6095961879663001], [0.5747300935864245, -0.13117839339674128], [-0.13773576502271195, 0.9904690096280693], [-0.3602566989790262, 0.015541097446245516], [0.31149699070671766, -0.725233483916796], [-0.16032931424166375, 1.2018374793003312], [-1.1491601404667926, 1.3508797454765058], [-0.22252093395631428, 0.974927912181824], [-0.4046838931928858, 1.6706417473989887], [-0.688503009293283, 2.6295195859874028], [-0.4659820753369688, 1.6545916738055797], [-0.44923275572942933, 1.7157024935448653], [-1.0727225575881625, 0.9338710110768353], [-0.07473009358642402, 0.9972037971811804], [-1.5933085708088526, 1.5617392390016036], [-0.8602566989790262, 0.8815665012306841], [-0.7141733479013409, 0.08460208849544386], [0.2858266520986591, 0.08460208849544351], [0.651167676465054, 1.0154758371396482], [-0.12348980185873315, 1.6478568862524685], [-1.222520933956314, 0.9749279121818235], [-0.36226423497728755, 0.4650667429781781], [0.637735765022712, 0.46506674297817807], [-0.3178370407634287, 0.17031156856727397], [-0.45557280578614046, 1.1607805781953429], [-0.3508398595332076, 1.4650014826715805], [-1.083891731363034, 0.7848287449006612], [-0.08389173136303418, 0.7848287449006615], [-1.2838191161003976, 0.9588778385884146], [-0.42356241712137066, 0.4490166693847688], [-0.3488323235349464, -0.5481871277964114], [0.47740645078104826, -1.1115071858600332], [-0.2949882898752685, 0.8098355724122399], [0.20501171012473118, 1.6758609761966783], [-0.7838191161003969, 1.8249032423728528], [-0.28753414139086375, 0.8727601913375498], [0.7124658586091364, 0.8727601913375497], [-0.21417334790134135, 1.2487120246322316], [-0.2838191161003974, 0.9588778385884142], [0.6717536896857436, 1.2536330129993187], [0.44923275572942956, 0.2787051008174948], [1.4266392065104776, 0.49007357048975664], [-0.30641266531934885, 1.1702463082606762], [0.6935873346806509, 1.1702463082606762], [1.6491601404667917, 1.4650014826715805], [-0.3282463103142565, 1.253633012999319], [1.5727225575881625, 0.8197492738817604], [1.0000000000000002, 5.551115123125783e-17], [-0.4575803417844021, 1.5221978966520362], [-1.030302899372565, 0.7024486227702758], [-0.03401792466303144, 0.6163309755194112], [-1.4215548811231093, 1.9493468482164835], [-0.4215548811231091, 1.9493468482164833], [0.4046838931928857, 1.3860267901528616], [-0.6935873346806509, 0.19009916728116277], [-1.665909460074331, 0.42374352195278064], [-0.6659094600743309, 0.42374352195278053], [0.16666936429837537, -0.31976200192248283], [0.47232212539367985, 0.6323810491128207], [-0.8653410243663948, 1.7968991524286428], [-0.19358733468065098, 1.0561245710656013], [0.1717536896857434, 0.12525082242139707], [-0.8282463103142561, 0.12525082242139712], [-0.7535162167278322, 1.1224546196025773], [0.24648378327216736, 1.122454619602577], [-0.5650132074345493, 1.2207079346880216], [-0.19967218306815515, 0.28983418604381717], [0.8003278169318454, 0.28983418604381694], [-0.022593549218951958, 2.2057760640346222], [0.977406450781048, 2.2057760640346222], [-0.8114969907067172, 1.5912588877012348], [-0.13974330102097332, 0.8504843063381932], [0.8602566989790263, 0.8504843063381932], [-1.5114243754440795, -0.13390933590896384], [-2.2444762472739055, 0.5462634018619559], [-1.4555728057861403, 1.160780578195343], [-0.6272048271491997, 0.6957138352171655], [0.2444762472739066, 1.1857874057069224], [1.2444762472739066, 1.1857874057069218], [1.2330518718298267, 0.18585266601351913], [-0.6234898018587333, 0.7818314824680299], [0.24819127256437312, 1.2719050529577869], [0.23676689712029325, 0.27197031326438403], [-0.9349867925654506, 1.5070649663848263], [-0.5114243754440795, 0.6011980452522012], [0.34883232353494664, 0.09133687604855514], [-1.1234898018587332, 1.6478568862524687], [-0.34883232353494575, 1.0154758371396482], [-1.3376631497600746, 1.164518103315823], [-2.6214822658604717, 0.7184986963636855], [-1.7612255668814452, 0.20863752716003972], [-1.2612255668814454, 1.0746629309444782], [-0.7367668971202928, 0.594055090520055], [-0.7481912725643722, -0.4058796491733475], [0.1234898018587336, 0.08419392131640932], [-1.2330518718298262, 1.546198141555358], [-0.37279517285079944, 1.0363369723517124], [-0.9962849747095328, 1.8181684548197425], [-0.744476247273906, -0.31976200192248305], [0.21109655851223474, -0.6145171763333873], [-1.309489454708455, -0.22569418336170557], [-1.3209138301525345, -1.225628923055108], [-0.4492327557294286, -0.7355553525653513], [-1.274657478323787, -0.2336443546716181], [-1.052136544367473, -1.2085722668534418], [-0.5521365443674728, -0.3425468630690034], [0.09702359609931932, 1.1224546196025775], [-0.5747300935864244, 1.8632292009656184], [-1.699927384737363, 1.6295848462940006], [-1.0281736950516192, 0.8888102649309592], [-0.7225209339563148, 1.8409533159662628], [-1.9123932433464987, 1.6228500587408896], [-0.912393243346499, 1.6228500587408892], [0.08389173136303424, 1.536732411490025], [-1.552136544367473, -0.34254686306900317], [-1.2464837832721685, 0.6095961879663003], [-0.5747300935864247, -0.1311783933967412], [-0.8488323235349464, 0.7746885277358839], [-1.4215548811231091, -0.045060746145876365], [-0.4252699064135755, -0.13117839339674114], [-1.4026763571946237, -0.342546863069003], [-0.9026763571946238, 0.5234785407154355], [-0.9009688679024193, 0.43388373911755845], [-0.5953161068071142, 1.3860267901528613], [0.40096886790241903, 1.2999091429019967], [1.4009688679024188, 1.2999091429019969], [-0.9235624171213708, 0.6452522087898201], [0.07643758287862912, 0.6452522087898199], [-0.7498011914373657, 1.208572266853442], [-1.5953161068071147, 1.3860267901528616], [-1.5205860132206905, 0.38882299297168155], [-0.6603293142416642, -0.121038176231964], [-1.9266392065104776, 0.3759518332946819], [-1.4266392065104776, -0.49007357048975675], [-0.44923275572942933, -0.2787051008174948], [0.05076724427057089, 0.5873203029669437], [-0.23305187182982634, 1.546198141555358], [-0.9266392065104776, 0.3759518332946818], [-1.2104583226108747, 1.3348296718830959], [-0.22162749638574636, 1.1857874057069218], [-0.2330518718298265, 0.18585266601351913], [-0.9492327557294293, 0.5873203029669438], [-0.7330518718298262, 0.6801727377709197], [-1.0256703386080586, 0.8098355724122399], [-0.5256703386080586, -0.05618983137219871], [-0.3653410243663947, 0.930873748644204], [-2.210458322610875, 1.3348296718830963], [-2.221882698054954, 0.3348949321896937], [-1.4329792565671888, 0.9494121085230813], [-1.5612981821440834, -0.5549219153495226], [-0.626311389578633, -0.20023938444593964], [-1.4009688679024195, 0.43214166466688075], [-1.9555728057861406, 0.29475517441090404], [-1.7952434915444773, 1.281818754427307], [-1.2952434915444768, 2.147844158211745], [-0.8716810744231062, 1.2419772370791207], [-1.47232212539368, -0.6323810491128208], [-1.2498011914373657, 0.3425468630690028], [-0.8262387743159945, -0.5633200580636222], [-1.1120654264146534, -0.2762168145320277], [-0.4885756245559201, 0.5056146679360022], [0.5114243754440799, 0.5056146679360023], [1.4888308262251286, 0.7169831376082643], [-0.9723221253936799, 0.2336443546716181], [-1.337663149760075, -0.6972293939725863], [-1.4123932433464992, 0.299974403208594], [-0.9949156746126316, 0.4450128243438798], [0.005084325387368249, 0.4450128243438797], [-0.9215548811231091, 0.8209646576385619], [-0.05992888204718094, 0.7996953552474628], [0.6285741272461018, 1.524928839164259], [-1.146083351077685, 0.7969644127352395], [-1.7188059086658476, -0.02278486114652012], [-1.4962849747095337, 0.9521430510353031], [-1.3488323235349466, 1.0154758371396477], [-0.6603293142416637, 1.7407093210564437], [-0.16032931424166358, 2.6067347248408828], [-0.9888308262251289, 0.1490422661761744], [-0.011424375444080148, 0.3604107358484364], [0.6120654264146532, 1.1422422183164664], [-0.7050117101247315, -0.8098355724122399], [-1.199927384737363, 0.7635594425095622], [-0.699927384737363, 1.6295848462940006], [-0.27636496761599183, 0.7237179251613756], [-1.4888308262251286, 1.015067669960613], [-0.9349867925654504, 0.18244723640185304], [-0.7124658586091361, 1.1573751485836765], [-1.9888308262251286, -0.1490422661761746], [-1.128574127246102, -0.6589034353798201], [-0.4400711179528194, 0.06633004853697605], [-0.3653410243663954, -0.930873748644204], [0.13465897563360496, -0.06484834485976551], [-0.9888308262251282, -0.1490422661761747], [-0.3003278169318457, 0.5761912177406215], [-1.2726499423255258, 0.8098355724122394], [-1.4123932433464992, 0.7568246549564503], [-1.1171497518020221, 0.34103130431358225], [-0.42864674250873924, 1.0662647882303784], [-1.300327816931846, 0.5761912177406215], [-1.8003278169318457, 1.4422166215250602], [-0.8003278169318457, 1.44221662152506], [-1.7726499423255255, 1.675860976196678], [-0.9979924640017385, 1.0434799270838582], [-2.2726499423255255, 0.8098355724122397], [-1.2838191161003971, 0.6607933062360654], [-0.2838191161003971, 0.6607933062360654], [-1.9555728057861408, -0.29475517441090404], [-1.0953161068071142, -0.8046163436145497], [-0.7225209339563141, -0.10890250839738451], [-1.4555728057861408, 0.5712702293735347], [-0.6666693642983751, 1.1857874057069218], [-0.44414843034206086, 2.1607153178887453], [-0.9046838931928859, -0.5200013863684223], [-0.0784451188768911, -1.0833214444320443], [-1.188503009293283, 0.43887645221999205], [-0.9009688679024193, -0.43388373911755795], [-0.4774064507810485, -1.3397506602501832], [0.423562417121371, -0.9058669211326249], [-1.8376631497600744, -1.5632547977570246], [-1.5320103886647698, -0.6111117467217211], [-0.5320103886647699, -0.6111117467217211], [-0.7330518718298266, -0.6801727377709195], [-1.5933085708088528, -0.17031156856727347], [-1.0933085708088528, 0.6957138352171652], [-1.2330518718298262, 0.18585266601351924], [-0.7889034414877657, 1.0818058856766242], [0.1885030092932829, 1.2931743553488855], [-1.572722557588163, 0.3068908422454836], [-1.149160140466792, -0.5989760788871417], [-0.1935873346806512, -0.3042209044762374], [-1.6234898018587332, -0.78183148246803], [-0.6234898018587336, -0.7818314824680298], [-1.550129008369211, -0.4058796491733483], [-0.7612255668814454, 0.20863752716003942], [-0.6254973378569948, -0.718498696363685], [-0.5507672442705707, -1.7157024935448653], [-0.050767244270570555, -0.8496770897604266], [-0.6885030092932827, -0.42714895156444704], [0.2889034414877658, -0.21578048189218502], [0.15116767646505347, 0.7746885277358841], [-1.3178370407634288, 0.17031156856727347], [-0.8178370407634288, -0.6957138352171651], [-0.9555728057861406, 0.2947551744109043], [-0.7124658586091362, -1.3296104430854059], [0.02058601322069009, -0.6494377053144864], [-0.9349867925654506, -0.35468253090358237], [-0.6717536896857437, -1.2536330129993185], [-0.5970235960993195, -0.25642921581813843], [0.07473009358642435, -0.9972037971811798], [-0.7838191161003973, -0.2052320975483729], [0.2050117101247313, -0.05618983137219846], [0.7050117101247313, 0.80983557241224], [-1.072722557588163, 0.04627612990267821], [-1.8989613319041574, 0.6095961879663001], [-1.0727225575881625, 1.1729162460299218], [-2.072722557588163, 0.046276129902678265], [-1.1717536896857434, -0.38760760921487986], [-0.1717536896857435, -0.38760760921488], [-2.3989613319041574, -0.2564292158181384], [-2.093308570808853, 0.6957138352171652], [-0.43035423180094456, -0.5761912177406212], [-1.0538440336596775, 0.20564026472740876], [-0.07643758287862934, 0.4170087343996709], [-0.8653410243663954, -0.06484834485976533], [-0.7050117101247316, 0.9222152351566375], [0.2838191161003968, 1.071257501332812], [-0.7141733479013416, 0.3826866208477929], [-1.2225209339563141, -0.9749279121818237], [-0.22252093395631434, -0.9749279121818235], [0.6491601404667919, -0.4848543416920664], [-0.33967068575833625, -0.3358120755158922], [-1.360256698979026, 0.015541097446245572], [0.3282463103142565, 0.7407745813630415], [-2.1491601404667917, -0.5989760788871418], [-1.193587334680651, -0.3042209044762376], [-0.21618088389960255, -0.09285243480397559], [-1.074730093586425, -0.99720379718118], [-0.28582665209865954, -0.3826866208477926], [-0.5696457681990562, 0.5761912177406217], [0.36534102436639415, 0.9308737486442046], [-1.212465858609137, -0.0067347875531108825], [-0.7124658586091368, 0.8592906162313276], [-0.2124658586091368, -0.006734787553111021], [-0.3396706857583358, -1.740709321056444], [-0.8396706857583365, -0.8746839172720056], [-1.1234898018587332, 0.08419392131640888], [-0.03401792466303122, -0.7885662700211404], [0.18850300929328268, -1.763494182202964], [-0.09531610680711461, -0.8046163436145499], [-0.076437582878629, -0.6452522087898195], [-0.9026763571946238, -1.2085722668534415], [-0.4026763571946238, -0.34254686306900334], [-0.27837250361425325, -0.31976200192248316], [-1.2050117101247309, 0.05618983137219871], [-0.2050117101247314, 0.05618983137219885], [0.22162749638574653, 0.5462634018619553], [-0.35109506120241574, -0.27348587201980507], [-0.7889034414877653, -0.025006827511578927], [-1.2889034414877651, 0.8410185762728597], [-0.28890344148776537, 0.8410185762728599], [-1.295243491544477, -0.04105690110498799], [-0.3064126653193485, -0.19009916728116263], [-0.6717536896857433, 0.7407745813630418], [-0.23676689712029342, -0.27197031326438403], [0.263233102879707, 0.5940550905200546], [0.12549733785699502, 1.5845241001481236], [0.22252093395631378, -0.9749279121818235], [0.5281736950516185, -0.02278486114652012], [1.1999273847373622, -0.7635594425095615], [1.2746574783237867, 0.23364435467161834], [0.19992738473736216, -0.7635594425095615], [-0.08389173136303474, 0.19531839607885293], [-0.9555728057861412, -0.29475517441090393], [0.01142437544407926, -0.36041073584843614], [-0.7216274963857465, 0.31976200192248344], [0.2746574783237869, 0.23364435467161865], [-1.331323099703364, -0.14230747862306378], [-0.3539166489223155, 0.06906099104919843], [0.14608335107768444, 0.935086394833637], [0.36534102436639454, -0.9308737486442042], [1.300327816931845, -0.5761912177406218], [0.525670338608058, 0.05618983137219857], [0.5538440336596774, 0.6603851390570306], [-0.06964576819905577, 1.4422166215250602], [-0.4743296613919418, 0.05618983137219856], [0.16032931424166275, -0.8746839172720055], [-0.7723947406563177, 0.059595260983864716], [0.16259205190913284, 0.41427779188744773], [-0.6120654264146538, 1.0466588410002682], [0.28381911610039756, -1.4157280903362703], [0.4441484303420607, -0.42866451031986774], [0.6666693642983751, 0.5462634018619559], [0.4123932433464996, -0.7568246549564508], [0.28890344148776587, 0.8910322312960177], [0.2952434915444775, -0.41579335064286815], [1.2952434915444773, -0.41579335064286815], [0.5205860132206902, 0.216587698469952], [0.249801191437366, -0.3425468630690033], [-0.5764375828786288, 0.22077319499461875], [0.42356241712137094, 0.22077319499461845], [-0.07643758287862923, -0.03984151734818661], [0.7498011914373661, 0.5234785407154352], [0.326238774315995, 1.4293454618480606], [-0.2110965585122342, 0.025006827511579177], [0.7889034414877657, 0.025006827511579004], [1.0114243754440801, 0.9999347396934024], [-0.41610826863696543, 0.08119665888377782], [0.5838917313630345, 0.08119665888377747], [-0.14916014046679194, -0.5989760788871418], [-0.860256698979026, 0.05301091745578923], [0.0953161068071145, 0.3477660918666935], [-0.6377357650227118, -0.3324066459042258], [1.365341024366395, 0.9308737486442042], [0.4387018178559172, 1.3068255819388859], [-0.5612981821440828, 1.306825581938886], [0.23676689712029264, 1.6323157888062223], [-0.7632331028797072, 1.6323157888062227], [0.7774790660436857, 0.9749279121818235], [-0.19992738473736282, 0.7635594425095614], [0.7889034414877656, 0.6145171763333871], [-1.7612255668814454, -0.3808728216617685], [0.3765101981412662, -0.7818314824680296], [-0.2952434915444775, -0.04105690110498805], [-0.011424375444079926, -0.9999347396934024], [0.4999999999999999, -0.8660254037844385], [-0.4266392065104778, -0.49007357048975686], [0.6346589756336046, -0.930873748644204], [0.7774790660436853, -0.9749279121818235], [0.839670685758336, -0.9870635800164028], [1.4123932433464985, -0.16731430613464207], [0.9252699064135756, -0.9972037971811801], [1.0747300935864244, -0.99720379718118], [0.30007261526263695, -0.3648227480683601], [1.2838191161003976, -0.9588778385884142], [0.328246310314257, -1.2536330129993183], [0.6603293142416637, -1.7407093210564437], [0.6717536896857434, -0.7407745813630412], [1.3653410243663948, -0.9308737486442044], [0.693587334680651, -0.19009916728116283], [1.8262387743159945, -0.5633200580636224], [2.398961331904157, 0.2564292158181384], [1.900968867902419, -0.433883739117558], [1.912393243346499, 0.5660510005758443], [1.9555728057861406, -0.2947551744109044], [2.2612255668814454, 0.657387876624399], [1.4349867925654505, 1.2207079346880212], [1.9962849747095335, -0.08611764725086496], [1.221627496385746, 0.546263401861955], [1.0612981821440826, -0.44080017815444755], [0.06129818214408267, -0.44080017815444733], [1.2632331028797068, -0.7662903850217839], [0.4885756245559194, -0.13390933590896392], [1.9888308262251284, 0.14904226617617422], [1.8716810744231063, 0.49007357048975697], [0.1999273847373626, 1.230848151852799], [1.6491601404667915, -0.4848543416920663], [1.826238774315995, 0.5633200580636218], [1.837663149760075, 1.5632547977570244], [1.7330518718298267, 0.6801727377709189], [1.6234898018587343, 0.7818314824680292], [0.8488323235349469, 1.4142125315808494], [0.6460833510776847, 1.1957011071764425], [-0.35391664892231534, 1.1957011071764427], [1.5, 0.8660254037844386], [0.6397433010209735, 1.3758865729880843], [1.212465858609136, 2.195635846869845], [0.43035423180094434, 0.5761912177406215], [1.205011710124731, -0.05618983137219864], [0.21618088389960244, -0.20523209754837296], [-0.5727225575881634, -0.8197492738817602], [1.3056527610953046, 0.9521430510353034], [0.4794139867793099, 1.5154631090989257], [-0.3094894547084559, 0.9009459327655384], [0.3333306357016248, 1.1857874057069218], [0.5558515696579394, 2.1607153178887453], [0.4046838931928858, 0.5182593119177452], [-0.09531610680711422, 1.3842847157021838], [0.4794139867793098, 0.38882299297168166], [0.516749319607539, 0.33762587470191624], [0.09318690248616801, 1.2434927958345416], [1.0894718771957015, 1.1573751485836765], [0.6171497518020219, 0.2269095671185075], [-0.3828502481979781, 0.2269095671185078], [1.2225209339563143, 0.9749279121818235], [0.7110965585122342, 0.8410185762728597], [0.5507672442705707, 1.715702493544865], [0.25019880856263454, 1.2085722668534418], [0.3879345735853462, 0.21810325722537272], [0.5340179246630317, 0.2496944282650275], [1.0747300935864244, 0.9972037971811801], [0.30007261526263695, 1.6295848462940001], [0.6285741272461021, 0.16458336362242032], [0.2632331028797066, -0.7662903850217838], [1.4123932433464992, 1.432076404360283], [0.03401792466303166, 0.7885662700211403], [1.030302899372565, 0.7024486227702753], [1.3359556604678697, 1.6545916738055788], [0.9252699064135758, 0.9972037971811802], [0.7090890225139733, 0.9043513623772046], [0.06501320743454952, 1.507064966384826], [0.5650132074345495, 2.3730903701692645], [1.188503009293283, 1.5912588877012346], [-0.0727225575881627, 0.9338710110768353], [0.8828502481979779, 0.639115836665931], [-0.11714975180202214, 0.6391158366659312], [0.10572537635794199, 1.715702493544865], [-0.8209138301525356, 2.0916543268395467], [-0.14916014046679194, 1.3508797454765054], [-0.6491601404667919, 2.2169051492609437], [0.7216274963857461, -0.3197620019224832], [0.6346589756336051, 0.9308737486442042], [-0.33766314976007483, 1.1645181033158225], [0.5764375828786292, 0.9058669211326252], [1.0764375828786292, 1.7718923249170637], [0.38793457358534633, 1.0466588410002677], [0.5953161068071144, 1.2137914956511322], [1.1717536896857441, 0.12525082242139743], [1.7952434915444784, 0.9070823048894266], [1.2889034414877656, 1.480542580117826], [0.3765101981412665, 0.7818314824680297], [1.3727951728507999, 0.6957138352171649], [1.212465858609136, -0.2913497447992379], [-0.44641116800953107, 1.3731556304758614], [0.26694812817017377, 0.6801727377709195], [1.263233102879707, 0.5940550905200546], [0.22534252167621271, 0.6323810491128201], [0.44786345563252716, -0.3425468630690035], [-0.4123932433464992, 0.1673143061346422], [0.1737612256840052, 0.5633200580636222], [-0.48656808855765865, 0.442281881831658], [1.1717536896857437, 0.6266528441679664], [0.13974330102097354, 0.5098611692036457], [0.09903113209758097, 0.4338837391175583], [1.0953161068071144, 0.3477660918666934], [0.07336079348952229, 0.3759518332946816], [0.8064126653193484, -0.3042209044762379], [-1.084146933032243, -0.9536586097907247], [0.04442719421385932, 0.29475517441090426], [0.05585156965793936, 1.2946899141043067], [0.027677874606320207, 0.2336443546716182], [-0.8989613319041576, 0.6095961879662999], [0.002007535998261867, 1.0434799270838573], [-0.07272255758816248, 0.0462761299026771], [0.6004004321944825, 1.053393628553379], [0.4400711179528186, 0.0663300485369761], [-0.3345863603709689, 0.6987110976497962], [0.011169173774871477, 0.1490422661761744], [0.4492327557294291, 2.4676061601342294], [0.26494059217191235, 0.7435055238752639], [-0.02058601322068987, 0.6494377053144871], [0.011169173774871366, -0.14904226617617433], [0.8828502481979776, 0.34103130431358286], [0.022593549218951625, -0.21136846967226214], [0.5225935492189516, 0.6546569341121764], [-0.4123932433464992, 0.2999744032085939], [-1.507709350153613, -0.04779168865809913], [0.04442719421385943, -0.2947551744109042], [0.48857562455591974, 0.6011980452522006], [0.8539166489223146, -0.3296757033920037], [-0.8791352229075113, 1.47713715050616], [0.27239474065631675, 1.3959404916223823], [-0.6831780651298238, 1.101185317211478], [-0.21246585860913658, -0.4635850393009673], [-0.6067404822511944, 0.3860920504594589], [-0.38421954829488003, 0.7715096138194739], [-0.795243491544477, 0.15517863830006307], [-0.8616259990759287, 0.021269302391099143], [-1.7218826980549546, 0.5311304715947449], [-0.8502016236318488, 1.021204042084502], [-0.12857412724610207, 0.701442040162018], [-0.9123932433464992, -0.5660510005758446], [-0.837663149760075, -1.5632547977570248], [-0.7238902340532167, -0.96919970723697], [0.09903113209758097, -0.43388373911755834], [-1.1265665912478409, 1.1395112758042434], [-0.8359556604678695, -0.7885662700211409], [-0.5303028993725647, 0.16357678101416262], [0.1737612256840051, -0.5633200580636218], [-0.4979924640017388, 0.1774545232994197], [0.28582665209865854, 0.38268662084779287], [-0.21246585860913658, 0.2913497447992379], [-0.46065713117350926, 0.1262574050296542], [-1.4329792565671893, -1.0449954858392791], [-0.6999273847373626, -0.36482274806836024], [-0.7612255668814456, -0.38087282166176895], [0.22760525934368292, -0.529915087837943], [0.26694812817017344, -0.680172737770919], [0.27837250361425325, 0.31976200192248344], [-0.5933085708088528, -0.17031156856727336], [0.36226423497728766, -0.4650667429781777], [0.32292136615079714, 0.2747012557766064], [1.2238902340532163, 1.8352251110214084], [-0.053844033659677715, 0.20564026472740826], [-0.11206542641465367, 0.1806334372158292], [0.03401792466303177, 1.1157198320494666], [-0.4215548811231091, -0.04506074614587652], [-0.5894718771957019, -0.2913497447992377], [0.40681309751383155, -0.3774673920501026], [-0.016749319607539226, 0.5283995290825226], [0.682162959236571, 0.17031156856727392], [0.9659820753369686, -0.7885662700211403], [0.8056527610953046, 0.08611764725086497], [-0.09531610680711444, -0.3477660918666934], [-0.2746574783237874, -0.23364435467161848], [0.34883232353494664, 0.5481871277964108], [-1.3989613319041574, -0.2564292158181387], [-0.47232212539368, -0.6323810491128203], [-0.1666693642983752, 0.3197620019224833], [1.371681074423106, -0.3759518332946814], [1.1491601404667913, -1.350879745476505], [0.21109655851223375, -0.025006827511578844], [-0.6491601404667928, 0.4848543416920668], [-0.7255977233454218, 0.4450128243438803], [-0.3376631497600755, -0.697229393972586], [0.23505940782808699, 0.12251987990917482], [1.1885030092932825, -1.7634941822029642], [0.35391664892231445, -0.06906099104919822], [1.2255977233454207, 0.4210125794405589], [1.1491601404667917, 0.3811710620923724], [1.3502016236318477, -0.1551786383000627], [2.221882698054954, 0.3348949321896944], [1.3616259990759274, 0.8447561013933401], [1.339670685758336, -0.12103817623196422], [2.072722557588163, 0.5591345615389546], [-0.061298182144083, -1.420947319133961], [0.2443545789512218, -0.4688042680986576], [0.05076724427057033, -1.6015807563497901], [1.5727225575881627, -0.3068908422454839], [0.5953161068071144, -0.5182593119177461], [1.497992464001738, -0.1774545232994193], [0.40297640390068046, -0.2564292158181387], [0.09732364280537587, -1.2085722668534422], [0.9235624171213709, -0.6452522087898205], [0.9349867925654507, 0.354682530903582], [1.137735765022712, -0.9904690096280693], [1.637735765022712, -0.12444360584363057], [0.9659820753369683, 0.616330975519411], [0.4659820753369682, -0.2496944282650277], [0.3114969907067172, -0.4271489515644471], [0.8114969907067172, 0.4388764522199915], [0.4492327557294291, -1.7157024935448653], [1.3209138301525354, -1.225628923055108], [0.3942746236420577, -0.8496770897604264], [1.7612255668814463, -0.20863752716003997], [0.9865680885576589, 0.42374352195278003], [1.2225209339563143, -0.9749279121818235], [0.28753414139086375, -1.329610443085406], [1.7838191161003976, -0.09285243480397565], [0.8282463103142569, -0.38760760921487986], [1.2952434915444775, 0.04105690110498822], [0.30641266531934885, 0.19009916728116263], [0.9464111680095313, -0.5071302266914232], [0.6120654264146539, -0.21810325722537272], [-0.16259205190913373, 0.4142777918874474], [-1.1349141773028135, 0.6479221465590657], [0.38285024819797864, -0.524994099470856], [1.3791352229075113, -0.611111746721721], [0.29498828987526904, -0.8098355724122398], [-0.6773338355184109, -0.5761912177406214], [-0.7952434915444773, 0.8249685026794502], [1.553844033659678, 0.6603851390570303], [0.4387018178559171, -0.5549219153495226], [0.8094894547084555, -0.034920528981099166], [-0.1460833510776851, -0.32967570339200336], [1.6234898018587336, -0.7818314824680297], [1.6349141773028135, 0.21810325722537272], [0.6625920519091336, 0.4517476118969909], [0.1511676764650539, 0.31783827598802716], [0.5507672442705712, -0.7355553525653515], [0.7632331028797072, -0.2719703132643841], [-0.20908902251397277, -0.038325958592765774], [0.6511676764650538, -0.5481871277964115], [0.15116767646505358, -1.4142125315808503], [0.5696457681990559, -0.5761912177406214], [1.7330518718298262, -0.6801727377709196], [1.4026763571946237, 0.3425468630690028], [1.477406450781048, 1.339750660250183], [0.714173347901341, -0.38268662084779326], [2.7726499423255255, 0.05618983137219913], [1.795243491544477, -0.15517863830006298], [1.4441484303420604, -0.4286645103198682], [-0.04241965821559779, -0.3580879605152489], [1.0205860132206899, -0.6494377053144871], [1.9888308262251286, -0.1490422661761741], [1.6265665912478406, -0.27348587201980457], [1.4349867925654505, 0.6835781673825858], [2.2612255668814454, 1.2468982254462078], [1.2612255668814456, 1.246898225446208], [1.2141733479013412, 0.48333878293664595], [0.5424196582155975, 1.2241133642996873], [1.0114243754440801, -0.3604107358484362], [1.5114243754440801, 0.5056146679360024], [2.007709350153613, 0.9138170924425375], [1.9215548811231091, 0.9110861499303151], [1.0696457681990557, 0.2898341860438167], [0.44615596634032206, 1.0716656685118466], [1.4962849747095333, -0.9521430510353035], [1.4215548811231091, 0.045060746145876684], [1.921554881123109, -1.0833214444320451], [0.9441484303420606, -1.2946899141043073], [1.2141733479013412, 0.7814233152889942], [0.17707863384920197, 0.5913241480078317], [1.9555728057861406, 0.29475517441090454], [0.960657131173509, 0.7397679987547845], [1.1666693642983748, -0.3197620019224827], [2.0933085708088526, -0.6957138352171647], [1.3602566989790263, -0.015541097446245239], [0.6885030092932826, 0.7252334839167962], [1.9009688679024195, 0.43388373911755734], [0.9743296613919418, 0.809835572412239], [1.1120654264146537, -0.1806334372158299], [0.6120654264146539, 0.685391966568609], [2.177333835518411, 1.4422166215250605], [1.6773338355184113, 0.5761912177406214], [1.1999273847373626, 1.2308481518527987], [1.932979256567189, 1.9110208896237175], [1.0970235960993189, 1.122454619602577], [1.997992464001738, 0.6885708804850192], [0.9161082686369656, 0.7848287449006615], [1.0538440336596775, -0.20564026472740773], [0.9161082686369656, 0.19531839607885296], [0.4161082686369655, -0.6707070077055857], [1.1831780651298234, -0.23515991342703885], [0.8539166489223151, 0.79696441273524], [0.2556454210487782, 1.3348296718830954], [0.806412665319349, 1.0561245710656006], [1.2889034414877663, -0.21578048189218588], [0.3488323235349471, 2.280237935365288], [0.8345863603709684, 0.16731430613464196], [0.059928882047181053, 0.799695355247462], [-0.10040043219448291, -0.18736822476894088], [1.5841469330322422, 1.819684013575163], [0.9492327557294289, 1.6015807563497906], [0.7124658586091359, 1.3296104430854063], [0.5970235960993191, 0.256429215818139], [0.5838917313630339, 0.6707070077055864], [0.8842195482948796, 0.09451578996496474], [1.106740482251194, 0.4799333533249797], [-0.07594388448119127, -0.3822568447683964], [0.3168447780834016, 0.5373719208542967], [0.3350820323021908, 1.5372056083036445], [0.2308092051387587, 0.5426568777150699], [-0.10676274578120926, 1.4839566057252764], [-0.00043991124290687633, -0.029658539477047496], [0.6459635903293279, 0.7333372096298221], [0.6642008445481182, 1.7331708970791695], [0.5599280173846859, 0.7386221664905949], [-0.021176059724390184, 0.20470391775276092], [-0.21468416624935482, 1.1858025940892052], [-0.3817352039833296, 0.19985434453854972], [-0.9817627457812128, 0.9998336874493474], [-1.3375719509199695, 0.9412997280102047], [-1.3193346967011794, 1.9411334154595516], [-0.7410399992606262, 2.756961364792093], [-1.0531731763703398, 1.806923007804652], [-1.6000275417978813, 0.799979342910798], [-1.5817902875790917, 1.7998130303601454], [-1.3304523051126862, 2.7677124148881638], [-1.7703307120606795, 0.6376445671815032], [-1.0174283879965142, 1.2957768414385697], [-1.981322834538304, 1.029492226926395], [-1.8567627457812106, 0.5157107691734204], [-1.4639740832166177, 1.4353395347961135], [-0.5889740832166179, 1.9194624530720406], [-1.9240561155188085, 0.38225684476839666], [-0.9244960267617153, 0.3525983052913488], [-0.9062587725429255, 1.3524319927406963], [-0.35810072824761585, 0.5160572211972563], [-1.9707075985924825, 0.24026393411166813], [-0.9711475098353892, 0.21060539463462036], [-0.09614750983538911, 0.6947283129105475], [-1.0202036253541977, 1.076985157678944], [-1.9839754524922046, 0.17830397890333544], [-1.1748074000826492, -0.40927330363890774], [-1.7748349418805305, 0.39070603927188946], [-0.8998349418805304, 0.8748289575478163], [-1.9956750779596208, 0.09290392419108484], [-1.9774378237408317, 1.0927376116404321], [-1.1206750779596208, 0.5770268424670117], [-0.125, 0.4841229182759271], [-1.99840082138868, -0.056531405876814156], [-1.351997319816445, 0.7064643432300559], [-0.59909499575228, 1.3645966174871222], [-1.9801635671698907, 0.943302281572533], [-1.2272612431057255, 1.6014345558296], [-0.7531662473534457, 0.7209608566184047], [-1.9638944465417894, -0.2662846145121748], [-2.2411735752841357, 0.6945047925713366], [-1.4320055228745803, 0.10692751002909329], [-0.5570055228745807, 0.5910504283050206], [-2.7342251586024684, 0.3713599526693283], [-1.7346650698453754, 0.34170141319228087], [-0.8596650698453756, 0.825824331468208], [-1.9373817497946546, -0.3483036823691525], [-1.1844794257304896, 0.30982859188791406], [-1.1662421715117, 1.3096622793372612], [-0.4995514083458037, 0.5643278401025814], [-1.578294697440554, -0.8158279493325407], [-0.7691266450309985, -1.403405231874784], [-0.7508893908122092, -0.4035715444254367], [-1.4502427770933701, -0.8929061774202539], [-0.45068268833627667, -0.9225647168973016], [-0.7882546392562463, 0.018735011112903308], [-1.3121331771097138, -0.9500383569874409], [-0.36572274652058423, -1.2730047590668185], [0.2125719509199695, -0.45717680973427743], [-1.1042728271634323, -0.9945487305885747], [-0.4578693255911974, -0.23155298148170442], [-0.4396320713724078, 0.7682807059676429], [-1.1063228345383038, 1.5136151452023225], [-1.4418447780834018, -0.05324900257836951], [-1.4236075238646122, 0.9465846848709777], [-1.5216532076824998, -0.08581685706494896], [-0.7687508836183348, 0.5723154171921175], [-1.7709635903293282, -0.24921429135389445], [-1.1245600887570935, 0.5137814577529756], [-0.8690160805805116, -0.9913844929458544], [-1.581572544655896, -0.2897697229300104], [-0.7065725446558959, 0.19435319534591683], [-1.044144495575865, 1.135652923356122], [-0.5259050042477202, -0.8804736992111949], [0.4529189360278896, -0.6757697814584337], [1.2058212600920548, -0.017637507201367142], [0.49326479601667095, 0.6839772628144771], [-0.5076677500289308, 0.11935998823815233], [0.2452345740352344, 0.7774922624952193], [-1.4966126028402025, -0.6402097650995265], [-1.4783753486214133, 0.35962392234982055], [-0.4518419557046903, -0.8363747715434399], [0.547718133052403, -0.8660333110204874], [-0.3763379824664057, -0.48377646625209114], [-0.3333092368341041, -0.7453344392346801], [-0.31507198261531455, 0.25449924821466746], [0.4378303414488507, 0.9126315224717341], [-0.22966928793932095, -0.6376445671815035], [0.4167342136329142, 0.12535118192536632], [-0.2114320337205312, 0.3621891202678441], [-0.044380995986556204, 1.3481373698184997], [-0.1908319475904452, -0.5875772825422435], [0.20195671497414813, 0.33205148308044996], [0.22019396919293754, 1.3318851705297972], [-0.7389899918857545, 0.24879748900119653], [-0.1432372542187894, -0.5157107691734202], [0.3822278387860243, 0.3351043905607271], [1.1913958911955795, -0.25247289198151635], [0.85382394027561, 0.6888268360286884], [0.8720611944943997, 1.6886605234780356], [0.7990561155188085, 0.10186607350753074], [-0.18491933697339635, 0.2801700524108659], [-0.3159032563928841, 1.2715545453567203], [-0.41351583540116077, 0.5590428832418086], [0.46148416459883923, 1.0431658015177356], [1.398865914393494, 1.3914694838868884], [0.6057937182941737, 0.7823418357094307], [-1.059919336973396, -0.2039528658650609], [-1.1909032563928843, 0.7874316270807937], [-1.7909307981907654, 1.5874109699915913], [-0.9159307981907656, 2.0715338882675183], [-0.05358956941087056, -0.32296640207937743], [0.5247051280296833, 0.4928615472531632], [-0.07532241376819793, 1.2928408901639612], [-0.41289436468816765, 2.234140618174166], [0.19774841305553514, 0.6449329824486408], [1.1684560116480176, 0.4046690483369727], [1.1866932658668068, 1.4045027357863202], [-0.4709699499299379, 0.5857654714442478], [0.40403005007006176, 1.0698883897201752], [0.21052194354509757, 2.050987066056619], [-0.7734535089471074, 2.2292910449599543], [-0.6017476137061801, 0.5134083694640623], [0.39665320768249934, 0.5699397753408764], [0.29238038051906745, -0.42460895524769837], [-0.6536171112087517, 0.47701294083142043], [0.25982376836423593, 0.8839843888777517], [0.15555094120080337, -0.11056434171082297], [-0.7661460334862543, 0.3786483679364666], [-0.7479087792674648, 1.378482055385814], [-1.0600419563771784, 0.42844369839837315], [-1.1051635671698907, 1.4274251998484604], [-0.029292401407517654, -0.24026393411166802], [0.8457075985924822, 0.2438589841642591], [-0.153852490164611, 0.27351752364130716], [-0.13561523594582114, 1.2733512110906542], [0.7236099226566475, 0.41786834014539886], [0.974947905123053, 1.385767724673417], [0.9931851593418424, 2.385601412122764], [0.8889123321784103, 1.39105268153419], [0.22204558105888772, 0.7276354504163504], [-0.7707706233035863, 0.6079860277044553], [0.10422937669641352, 1.092108945980382], [-0.7418488654829017, 0.46135083590417614], [-0.7236116112641121, 1.4611845233535234], [-1.0008907400064584, 2.4219739304370345], [-0.004324922040378931, -0.09290392419108473], [0.8706750779596213, 0.3912189940848422], [0.27064753616174, 1.1911983369956403], [0.10359649842776486, 0.20525008744498457], [0.6420785795318555, 0.6700918249157853], [0.6603158337506452, 1.6699255123651322], [1.6598759225077386, 1.6402669728880848], [0.8172933697375984, 1.1016997609568786], [0.01391233217841048, 0.9069297632582625], [-0.8286702205917302, 0.3683625513270564], [-1.752726336110539, 0.7506193960954533], [0.8745600887570932, 0.45446437879887946], [1.4227181330524035, -0.3819103927445604], [0.4243173116637231, -0.43844179862137456], [0.08674536074375339, 0.5028579293888303], [0.752462412821258, 0.628473734780019], [-0.17159370269755025, 1.0107305795484156], [0.8123817497946539, 0.8324266006450805], [0.017797342975882646, 0.9701751479722998], [-0.43244543411748704, 0.07726897055204576], [0.5659553872711931, 0.1338003764288601], [-0.19394801776787152, 0.9514401368593963], [-0.29822084493130396, -0.043108593729178135], [0.6481895856578259, -0.36607499580855574], [-0.33801186216287604, 0.911641188533157], [0.536988137837124, 1.395764106809084], [1.0851461821324335, 0.5593893352656442], [0.8916380756074689, 1.540488011602089], [-0.0015991786113199735, 0.0565314058768141], [0.8734008213886799, 0.5406543241527415], [-0.09049362515310944, 0.27436970964056656], [0.16084435731329627, 1.242269094168585], [-0.6322278387860243, 0.633141445991127], [0.5238659143934938, 0.9073465656109612], [1.3806286601747044, 0.391635796437541], [0.5875564640753841, -0.21749185173991714], [0.01663807560746955, 1.0563650933261615], [0.21014618213243397, 0.07526641698971731], [-0.5024102819429498, 0.7768811870055613], [-1.1024378237408312, 1.5768605299163594], [-0.046720789404032126, 1.0555129073269014], [-0.3588539665137459, 0.10547455033946071], [-0.34061671229495616, 1.1053082377888082], [0.7317262643397751, 0.8628361920098276], [1.124514926904368, 1.7824649576325209], [1.5986099226566477, 0.901991258421326], [0.3716126028402025, 1.1243326833754541], [1.2466126028402025, 1.608455601651381], [0.2537963984777285, 1.4888061789394862], [-0.002938805505600439, 1.2045376052021082], [-0.6029663473034818, 2.004516948112906], [-0.7700173850374568, 1.0185686985622504], [-0.6212036015222716, 1.004683260663559], [-1.3146320713724076, 0.28415778769171557], [-0.0626182502053454, 0.3483036823691527], [0.18871973226105987, 1.3162030668971711], [0.021668694527085197, 0.3302548173465155], [0.6680721960993203, 1.0932505664533854], [0.6863094503181097, 2.0930842539027323], [-0.08655912042701286, 0.4069714480463311], [0.7702036253541975, -0.10873932112708906], [0.05764716127881364, 0.5928754488887549], [-0.5423803805190678, 1.3928547917995528], [-0.524143126300278, 2.3926884792489], [0.7884408795729869, 0.8910943663222579], [-0.20437532478948683, 0.7714449436103628], [-0.3353592442089752, 1.762829436556217], [0.627902324064165, 1.1422551925329938], [1.6274624128212585, 1.1125966530559461], [0.6900806630266039, 0.764292970686793], [-0.2738137835151856, 0.49800835617461875], [0.32524277709337013, 1.3770290956961806], [0.2209699499299378, 0.3824803651076063], [-0.6357927958512728, 0.8981911342810266], [0.31061763473785664, 0.5752247322016489], [-0.8779023240641648, -0.17400935598113992], [-0.10757161200348575, -0.8116539231626435], [-0.089334357784696, 0.18817976428670377], [0.4750275417978813, -0.3158564246348705], [0.5684284698501358, 1.2046483912477703], [0.5866657240689257, 2.2044820786971178], [-0.20692780390067966, 0.6091276481774581], [-0.311200631064112, -0.3854210824111165], [-1.2951760835563166, -0.207117103507781], [-0.42017608355631664, 0.277005814768146], [-0.24709767593583487, 0.658132274257067], [0.39930582563639994, 1.4211280233639365], [-0.14885221865891007, 2.257502794907376], [-1.1127466652006992, 1.9912181803952014], [-0.23774666520069876, 2.475341098671129], [-0.3065715301498637, 0.7205254729718433], [-0.41084435731329616, -0.27402325761673113], [-0.39260710309450686, 0.7258104298326165], [-0.730179054014476, 1.6671101578428211], [-0.6850574432217644, 0.6681286563927336], [-0.3535964984277651, 0.76299574910687], [-1.332420438703375, 0.558291831354109], [-0.8069553456985612, 1.4091069910882559], [0.0680446543014388, 1.8932299093641831], [-0.4217053025594465, 0.8158279493325407], [0.45329469744055384, 1.2999508676084677], [-0.5306807550516509, 1.4782548465118033], [0.11572274652058434, 2.2412505956186726], [-0.8770934578418905, 2.121601172906777], [-0.3765836917667349, -0.18315355211754678], [-0.7141556426867038, 0.7581461758926582], [-0.6959183884679147, 1.7579798633420054], [-0.800191215631347, 0.7634311327534312], [-0.4745349069951863, 0.8508151597341471], [0.4004650930048135, 1.3349380780100744], [-0.2662256701610827, 2.080272517244754], [-1.259041874523557, 1.960623094532859], [-0.3840418745235564, 2.4447460128087863], [-0.5497572229066297, 0.8929061774202537], [-0.654030050070062, -0.10164255316832073], [1.134410829502925, 0.789451813153937], [0.1705163829611358, 0.5231671986417633], [-0.6072113374354073, 0.9196287656226934], [-1.3006398072855436, 0.19910329265085], [-1.5779189360278898, 1.1598926997343613], [-1.9154908869478593, 2.101192427744566], [-0.6878668228902864, 0.9500383569874409], [0.311693265866807, 0.920379817510393], [0.329930520085596, 1.9202135049597404], [-0.6488934201900136, 1.7155095872069794], [-0.002489918617778253, 2.478505336313849], [-0.7486620175335947, 0.9678993845280186], [-0.3558733549690021, 1.8875281501507115], [0.06150702555006471, 0.9787962766270857], [-0.9341680524095561, 1.0717002008181704], [0.060506034875960735, 0.3803221019857749], [-0.923469417616244, 0.5586260808891106], [-0.277065916044009, 1.3216218299959805], [-0.8329489622660251, 0.9859482495506557], [0.04205103773397534, 1.470071167826583], [-0.6705054263414091, 2.171685937842427], [-0.5395215069219211, 1.1803014448965723], [0.01105345858126383, 1.1194831101612428], [0.0292907128000528, 2.1193167976105904], [-0.28284246430966054, 1.169278440623149], [0.6635679662794691, 0.8463120385437717], [-0.18869054968189047, 1.6089613356268053], [0.6204775027276652, 1.021384053084562], [1.4772402485088758, 0.5056732839111419], [-0.22886042171704535, 1.657965961706414], [-0.9991911337777248, 2.2956105288879174], [0.0003689549793683966, 2.2659519894108695], [-0.9635254915624214, 1.9996673748986948], [-0.2883342759310744, 1.7203591604211907], [-0.2432126651383626, 0.7213776589711032], [-0.3474854923017947, -0.27317107161747123], [-0.669629568671497, 1.949872044436788], [-0.33205761775152753, 1.0085723164265832], [0.5429423822484727, 1.4926952347025102], [-0.7304247633148052, 1.9677330719773656], [0.20695698647984972, 2.316036754346518], [0.6810519822321293, 1.4355630551353236], [-1.5072278387860245, 0.14901852771520013], [-0.8137993689358884, 0.8695440006870436], [-0.020727172836567886, 1.4786716488645018], [-1.3137197322610596, -0.8320801486212437], [-1.1309839194194884, 0.9913844929458544], [-0.6568889236672084, 0.11091079373465972], [-0.638651669448419, 1.1107444811840068], [0.11425065461574624, 1.7688767554410736], [-1.1935081065249646, 0.9810986763364441], [-0.33674536074375416, 0.46538790716302403], [-0.6743173116637237, 1.406687635173229], [-0.656080057444934, 2.4065213226225763], [0.3434800313121593, 2.3768627831455285], [-0.3185081065249643, 1.4652215946123714], [-1.3141831844845853, 1.558125518803456], [-1.002050007374872, 2.5081638757908964], [-1.2772791287423462, 0.9607894070835113], [-0.3895749651794934, 0.5003751175646306], [-1.326956714974148, 0.152071435195478], [-0.5177886625645933, -0.4355058473467658], [-0.40227912874234617, 1.4449123253594385], [0.5111617508306407, 1.8518837734057694], [0.4068889236672084, 0.8573350428171949], [0.8571317007605783, 1.7502412202374487], [-0.4001902011253149, 1.2896034103793577], [0.40897785128424013, 0.7020261278371143], [0.13169872254189463, 1.6628155349206262], [-0.4625719509199695, 1.4254226462861315], [-0.7747051280296832, 0.47538428929869103], [-1.4413958911955793, 1.2207187285333712], [-0.5846696268558044, 1.5994320022672717], [-0.896802803965518, 0.6493936452798308], [-0.20337433411538175, 1.3699191182516741], [-0.7592772534794158, 1.7571276773427456], [0.24028283527767735, 1.727469137865698], [-0.01105514718872791, 0.7595697533376793], [-0.8121068579151557, 1.792114887744352], [0.06289314208484464, 2.276237806020279], [-0.13061496444012022, 3.257336482356723], [-0.5808577415334902, 2.3644303049364694], [-0.9447832883553768, 1.8609284936328983], [-0.41931819535056336, 2.7117436533670456], [0.05477680040171684, 1.8312699541558506], [-0.2780925251894809, 1.1155940543982181], [0.47480979887468466, 1.7737263286552847], [-0.5490039846405008, 1.3034888482780485], [0.41489046190128853, 1.5697734627902236], [-1.5310800574449341, 1.9223984043466489], [-2.529480878833614, 1.8658669984698348], [-1.6544808788336138, 2.349989916745762], [-0.6549207900765208, 2.320331377268714], [-1.9375994927178508, 1.741279070921003], [-1.1284314403082956, 1.1537017883787595], [-1.2954824780422702, 0.16775353882810362], [-1.4173803805190677, 0.9087318735236254], [-0.6644780564549027, 1.566864147780692], [0.17810449631523784, 2.1054313597118988], [-0.814711708047236, 1.9857819370000034], [0.06028829195276497, 2.4699048552759306], [-1.3991431263002783, 1.9085655609729728], [-0.646240802236113, 2.56669783523004], [0.020449960929783018, 1.8213633959953597], [-1.7549523314390372, 1.85003160153383], [-0.8085419008499075, 1.5270651994544526], [0.18713317710971378, 1.434161275263368], [0.20537043132850363, 2.433994962712715], [-1.5481580442953098, 0.8363747715434398], [-0.7952557202311448, 1.4945070458005065], [-0.7770184660123551, 2.494340733249854], [0.07974427976885545, 1.9786299640764338], [-0.6043524638382598, 0.7070754187197132], [-0.9419244147582294, 1.648375146729918], [-1.1089754524922046, 0.6624268971792624], [-0.6865866622248942, 1.2069507909571289], [0.18841333777510605, 1.6910737092330557], [-0.5050151320750302, 0.9705482362612123], [0.4413952985140994, 0.647581834181835], [-0.7250275417978813, 1.284102261186725], [-0.4477484130555348, 0.32331285410321364], [0.49866201753359474, 0.00034645202383609686], [-0.8471252177337163, 1.4581116171678645], [-1.7845069675283711, 1.1098079347987118], [-0.7916907631658965, 1.2294573575106074], [-1.2878943646881678, 1.7500176998982389], [-2.000450828763552, 2.451632469914083], [-1.3337600655976558, 1.706298030679403], [-1.3486895593314758, 1.7678787274388161], [-1.5157405970654505, 0.7819304778881605], [-0.6731580442953098, 1.3204976898193672], [-2.275218757429228, 1.0792875573883018], [-1.3045111588367455, 0.8390236232766338], [-0.7908594893883258, 0.21240206036855439], [0.08414051061167438, 0.6965249786444813], [-0.046843408807814235, 1.687909471590336], [-1.0308188613000189, 1.8662134504936714], [-1.6666907631658958, 0.74533443923468], [-0.6671306744088028, 0.7156758997576323], [0.20786932559119697, 1.1997988180335593], [-0.48555914425893953, 0.47927334506171587], [-1.0855866860568204, 1.2792526879725137], [-1.1412256701610821, 1.5961495989688272], [-1.4533588472707961, 0.6461112419813864], [-0.5783588472707962, 1.1302341602573138], [-1.7125564640753845, 0.701614770015844], [-0.7129963753182907, 0.6719562305387963], [-0.16483833102298084, -0.16441854100464354], [0.8308367469366398, -0.2573224651957281], [-0.7751747142807294, 1.049918452384997], [-0.7569374600619401, 2.049752139834344], [-1.6943192098565945, 1.7014484574651914], [-0.799115584502397, 1.1085862180621753], [0.07588441549760327, 1.5927091363381023], [-0.916931788864871, 1.4730597136262071], [-0.8375564640753842, 1.1857376882917712], [-0.14412799422524802, 1.9062631612636145], [0.1331511345170988, 0.9454737541801033], [0.8860534585812636, 1.6036060284371698], [-0.9596541400112191, 1.3597470442729107], [-0.08465414001121918, 1.8438699625488377], [0.6087743298389174, 2.564395435520681], [-1.019127994225248, 1.4221402429876877], [0.8542728271634323, 1.9627945671404285], [0.75, 0.9682458365518543], [-1.2623136869820137, 1.5945209474360977], [-1.366586514145446, 0.5999722168475234], [-0.45314563457245893, 1.0069436648938543], [-2.3125840058732656, 1.5015941129266417], [-1.4375840058732652, 1.9857170312025687], [-0.44190892791364433, 1.8928131070114844], [-0.8239202814715497, 0.31467816510212565], [-1.7878147280133392, 0.048393550589950984], [-0.9128147280133391, 0.532516468865878], [0.17563980728554363, 0.2850196256250778], [-0.017868299239421237, 1.2661183019615219], [-1.75209345784189, 1.6374782546308504], [-0.8056830272527602, 1.314511852551473], [-0.03535231519208126, 0.6768672853699697], [-0.9103523151920808, 0.19274436709404283], [-1.3277326957111486, 1.1014762406176681], [-0.4527326957111488, 1.585599158893595], [-1.2103592442089757, 1.2787065182802901], [-0.4400285321482965, 0.6410619510987867], [0.43497146785170404, 1.125184869374714], [-1.2126361007502127, 2.4032389193241315], [-0.33763610075021266, 2.887361837600059], [-1.605424763314805, 1.4836101537014388], [-0.6590143327256752, 1.1606437516220613], [0.21598566727432456, 1.6447666698979884], [-1.3826677500289306, -0.3647629300377745], [-2.353375348621413, -0.12449899592610603], [-1.9605866860568204, 0.7951297696965868], [-1.0106152359458214, 0.789228292814728], [-0.4851501429410078, 1.6400434525488752], [0.3898498570589928, 2.1241663708248018], [0.8639448528112723, 1.2436926716136065], [-1.0490561155188085, 0.8663797630443237], [-0.5235910225139949, 1.7171949227784709], [-0.04949602676171527, 0.8367212235672757], [-1.6727181330524035, 1.3501562292964153], [-0.8635500806428481, 0.762578946754172], [0.13601000811424524, 0.7329204072771237], [-1.905818861300019, 1.3820905322177441], [-0.921843408807814, 1.203786553314409], [-1.859225158602469, 0.8554828709452564], [-2.7666386682889494, -0.15631036716280944], [-1.767078579531856, -0.18596890663985743], [-1.7488413253130668, 0.8138647808094897], [-1.0572667190194953, 0.6472353821579995], [-0.5831717232672152, -0.23323831705319542], [-0.05770663026240197, 0.6175768426809514], [-1.952470344373693, 1.2400976215610156], [-0.9529102556165994, 1.2104390820839677], [-1.876966371135408, 1.5926959268523646], [-0.6216126028402027, -0.1560868468235997], [0.3572113374354071, 0.04861707092916123], [-1.161539546182927, -0.347313348430575], [-0.687444550430647, -1.2277870476417698], [-0.16197945742583375, -0.37697188790762304], [-1.127212706710994, -0.3374067902700848], [-0.4605219435450981, -1.0827412295047651], [-0.4422846893263084, -0.0829075420554177], [-0.5835103594873912, 1.51324205691341], [-1.5299207900765204, 1.8362084589927867], [-2.4013558330112725, 1.0870358524269605], [-1.4549454024221427, 0.7640694503475833], [-1.6484535089471075, 1.7451681266840273], [-2.5840029942900857, 0.9782833218141334], [-1.7090029942900857, 1.46240624009006], [-0.7955621147170988, 1.8693776881363913], [-1.3172846893263088, -0.5670304603313447], [-1.5107927958512732, 0.41406821600509947], [-0.5643823652621436, 0.09110181392572195], [-1.2427727538954558, 0.7510361984481514], [-1.3470455810588882, -0.24351253214042295], [-0.4336047014859006, 0.16345891590590805], [-1.1865070255500656, -0.49467335835115844], [-1.1682697713312762, 0.5051603290981888], [-1.1234008213886804, 0.42759151239911325], [-1.3169089279136443, 1.4086901887355572], [-0.4034680483406572, 1.815661636781888], [0.4715319516593426, 2.2997845550578155], [-1.245498497324515, 0.6016008683802526], [-0.3704984973245149, 1.0857237866561795], [-1.366173575284136, 1.1786277108472643], [-2.1919089279136448, 0.9245672704596299], [-1.643750883618335, 0.08819249891619024], [-0.6441907948612418, 0.05853395943914243], [-1.9928162043624742, -0.11964942271189521], [-1.1360534585812636, -0.6353601918853156], [-0.3831511345170986, 0.022772082371751157], [-0.36491388029830885, 1.0226057698210984], [-1.077470344373693, 1.7242205398369426], [-1.1178162043624742, 0.3644734955640318], [-1.830372668437858, 1.0660882655798756], [-0.8929909186432037, 1.4143919479490288], [-0.41889592289092425, 0.5339182487378334], [-1.2399138802983092, 0.5384828515451713], [-1.0957075985924825, 0.7243868523875955], [-1.4145215069219206, 0.6961785266206454], [-0.5577587611407101, 0.18046775744722518], [-0.8953307120606793, 1.12176748545743], [-2.7053726684378585, 0.5819653473039489], [-2.231277672685578, -0.298508351907246], [-1.8384890101209856, 0.6211204137154474], [-1.2224854923017952, -0.7572939898933988], [-0.5760819907295602, 0.005701759213471047], [-1.5600574432217644, 0.18400573811680662], [-1.9788239402756098, -0.20470391775276103], [-2.3163958911955795, 0.7365958102574436], [-2.2981586369767895, 1.736429497706791], [-1.4889905845672349, 1.148852215164548], [-1.1071317007605788, -0.7819953836855948], [-1.384410829502925, 0.1787940233979164], [-0.5752427770933697, -0.4087832591443266], [-0.964334357784696, -0.29594315398922244], [-0.7972833200507213, 0.6900050955614333], [0.07771667994927878, 1.1741280138373607], [0.8306190040134438, 1.8322602880944272], [-1.0888944465417894, 0.2178383037637527], [-0.9579105271223011, -0.773546189182102], [-1.506068571417611, 0.06282858236133801], [-1.2109921224776243, 0.391847659744892], [-0.33599212247762444, 0.8759705780208189], [-1.3288083268400985, 0.756321155308924], [-0.5645886209053892, 1.154843408851762], [-0.31325063843898393, 2.1227427933797807], [-1.5136516694484192, 0.6266215629080789], [-1.6179244966118516, -0.36792716768049477], [-1.895203625354198, 0.592862239403016], [-1.7968434088078147, 0.7196636350384811], [-1.5455054263414092, 1.6875630195664992], [-1.5272681721226196, 2.6873967070158473], [-1.0623817497946548, 0.13581923590677417], [-0.3094794257304895, 0.793951510163841], [-0.1424283879965147, 1.779899759714497], [-0.34982528571927063, -0.5657955341090698], [-1.544593087230062, 0.5713250832535407], [-1.5263558330112723, 1.5711587707028878], [-0.7171877806017171, 0.9835814881606446], [-1.9191444955758652, 0.6515300050801944], [-1.031440332013012, 0.1911157155613138], [-1.3087194607553583, 1.151905122644825], [-1.7930721960993203, -0.6091276481774582], [-0.7935121073422271, -0.638786187654506], [-0.5421741248758216, 0.3291131968735125], [0.005983919419487949, -0.5072615746699273], [0.024221173638277804, 0.4925721127794201], [-0.91807219609932, -0.12500472990153094], [-0.6667342136329149, 0.8428946546264874], [-1.6306286601747042, 0.5766100401143127], [-1.7272402485088751, 0.4625725526407124], [-1.2676071030945066, 0.24168751155668894], [-1.0162691206281012, 1.2095868960847076], [-1.541734213632915, 0.35877173635056014], [-2.3984969594141257, 0.8744825055239804], [-1.5234969594141252, 1.3586054237999077], [-2.487391405955915, 1.0923208092877332], [-1.5034159534637102, 0.9140168303843983], [-2.505628660174704, 0.09248712183838592], [-1.5682469103800498, 0.4407908042075387], [-0.6932469103800498, 0.9249137224834658], [-1.6934284698501365, -0.7205254729718433], [-0.693868381093043, -0.7501840124488911], [-0.7044836170388644, 0.03904428036583647], [-1.675191215631347, 0.2793082144775042], [-1.282402553066754, 1.1989369801001972], [-1.5596816818091002, 2.1597263871837087], [-0.6648538178675667, -0.4088565012862095], [0.33082126009205415, -0.501760425477294], [-1.3774102819429506, 0.29275826872963473], [-0.7032946974405536, -0.3317050310566136], [0.10587335496900141, -0.919282313598857], [0.684168052409555, -0.10345436426631605], [-0.97614778134109, -1.7733798766314486], [-1.1696558878660548, -0.7922812002950046], [-0.2946558878660548, -0.30815828201907747], [-0.4371331771097138, -0.4659154387115139], [-1.436693265866807, -0.43625689923446564], [-1.4184560116480176, 0.5635767882148818], [-1.2938959228909241, 0.04979533046190654], [-1.3390175336836363, 1.048776831911994], [-0.5861152096194707, 1.7069091061690602], [-1.6497051280296833, -0.00873862897723604], [-0.840537075620128, -0.5963159115194795], [-0.14710860576999163, 0.124209561452364], [-1.1670510377339745, -0.9859482495506557], [-0.2920510377339749, -0.5018253312747287], [-1.2848672420964486, -0.6214747539866241], [-0.892078579531856, 0.29815401163606936], [-0.32446848496383396, -0.44738103761944914], [0.22368955933147583, -1.283755809162889], [0.24192681355026524, -0.2839221217135416], [-0.5206475361617396, -0.2229525004437858], [0.2322547879024257, 0.43517977381328093], [-0.36777275389545605, 1.2351591167240785], [-1.3605591442589393, -0.004849573214211672], [-0.5037963984777287, -0.5205603423876318], [-1.1038239402756098, 0.2794190005231663], [-0.1047127384063391, -1.0242072700656222], [0.20742043870337445, -0.07416891307818113], [-0.7714035015722353, -0.27887283083094233], [-0.10587200577475231, -0.9380173247117605], [-0.5232523862938199, -0.029285451188134998], [0.4231580442953098, -0.3522518532675123], [-0.7114841645988397, -0.07491996496588124], [0.08158803150048088, 0.5342076832115769], [0.0998252857192703, 1.534041370660924], [-1.0860355729446431, 0.005284956860772828], [-2.0817106509042635, 0.09818888105185758], [-1.6314678738108932, 0.9910950584721111], [-1.9610355729446431, -0.4788379614151543], [-0.9626347515559628, -0.42230655553833996], [-0.08763475155596256, 0.06181636273758695], [-2.099947905123053, -0.9016448063974897], [-2.293456011648018, 0.07945386993895454], [-0.2226125790082768, -0.22838874383898444], [-1.1466686945270852, 0.15386810092941236], [-0.3937663704629204, 0.812000375186479], [-0.8507788263617226, 0.008449194503492963], [-1.188350777281692, 0.9497489225136978], [-0.3952785811823717, 1.5588765706911558], [-0.9351690430836606, 0.4732260261768595], [-0.7227208712576535, -0.9607894070835115], [0.15227912874234617, -0.47666648880758417], [0.6777442217471596, 0.37414867092656307], [-0.2596375280474945, 0.025844988557410298], [-1.3227484130555347, -0.16081006417271335], [-0.19641043058912933, 1.291212238631232], [-1.7155370756201278, -1.0804388297954066], [-1.0221086057699913, -0.3599133568235632], [-0.2692062817058263, 0.29821891743350354], [-0.5826196194809332, -0.9087318735236257], [-0.18983095691634044, 0.010896892099067357], [-0.9023874209917241, 0.7125116621149117], [-0.2559839194194893, 1.4755074112217814], [-1.1826471612788145, -0.10875253061282796], [-1.164409907060025, 0.8910811568365193], [-0.3076471612788142, 0.37537038766309916], [0.42050542634140964, -1.2034401012905722], [-0.4362573194398015, -0.6877293321171526], [-1.1488137835151853, 0.013885437898691858], [0.22699731981644478, -0.222341424954128], [0.8936880829823406, -0.9676758641888081], [0.18113161890695673, -0.2660610941729644], [0.1204984973245149, -0.11747795010432482], [-0.3297442797688551, -1.0103841275245788], [-0.3115070255500656, -0.010550440075231704], [-0.21377182713800646, 0.06956465777624604], [-1.2065880315004807, -0.050084764935649256], [-0.3315880315004811, 0.43403815334027773], [-0.19553457291921705, 1.069398345225593], [-0.29980740008264906, 0.07484961463701892], [-0.8031841329900662, 0.08031570787231652], [-1.6599468787712768, 0.5960264770457369], [-0.7849468787712771, 1.080149395321664], [-1.2384614683231046, -0.17885892919535074], [-0.30107971852844984, 0.16944475317380184], [-1.0714104305891294, 0.8070893203553051], [-0.20050397323828495, 0.13152461298457843], [-0.1822667190194951, 1.131358300433926], [-0.7822942608173764, 1.9313376433447238], [0.5416907631658958, -0.2612115209587529], [0.3481826566409312, 0.7198871553776912], [1.2945930872300608, 0.3969207532983138], [0.8772127067109938, 1.305652626821939], [0.41959308723006083, -0.08720216497761346], [-0.29296337684532314, 0.6144126050382309], [-0.818428469850137, -0.23640255469591626], [0.05947942573048848, 0.174294326388013], [-0.9112281728619935, 0.4145582604996815], [0.002212706710993828, 0.8215297085460124], [-1.2210134003969568, -0.2849201497157994], [-0.4681110763327917, 0.37321212454126756], [-0.4498738221140024, 1.373045811990615], [0.6453307120606793, -0.15352164890557596], [1.2917342136329144, 0.6094741002012934], [0.30775876114070955, 0.7877780791046286], [0.039905948745874165, 1.3300885047958628], [-0.8841501667729342, 1.712345349564259], [-0.5672412388592901, 0.3036551608287016], [0.43874268056019794, -0.2036064138412257], [-0.829696829737202, 0.1623347757292943], [-0.18329332816496724, 0.9253305248361643], [-1.1672687806571715, 1.1036345037395], [0.8087281411666485, -0.6172358220192908], [0.47115619024667876, 0.3240639059909136], [0.19387706150433281, 1.284853313074425], [0.6022402485088758, 0.02155036563521462], [-0.30357861279114373, 1.4036408978529584], [0.33463314541436895, 0.26323787719190395], [1.2096331454143692, 0.747360795467831], [0.2256576929221641, 0.925664774371166], [0.2594108295029256, 0.3053288948780105], [-0.7362642484566954, 0.39823281906909513], [0.13873575154330475, 0.8823557373450219], [-0.17259469337165545, 0.41225640490710386], [0.277648083721715, 1.3051625823273576], [-0.5315199686878403, 1.8927398648696012], [-0.32181586700993337, 0.4038072104036111], [0.5531841329900669, 0.8879301286795381], [0.2759050042477207, 1.8487195357630493], [-0.5284038985104142, 0.35372244546796155], [0.34659610148958575, 0.8378453637438884], [0.034462924379872195, -0.11219299324355247], [-0.9033884116658287, 0.11403748747360085], [-0.20995994181569277, 0.8345629604454441], [-0.5220931189254062, -0.11547539654199662], [0.6190160805805118, 1.9596303294977087], [-0.3738001237819627, 1.839980906785813], [-1.2488001237819626, 1.3558579885098863], [-0.7080704482444843, 2.027023514666384], [-1.5830704482444844, 1.5429005963904574], [0.08330923683410374, 1.713580275786534], [-0.6695930872300614, 1.0554480015294674], [0.2677886625645931, 1.4037516838986202], [-1.4816831091063825, -0.7017904618589397], [0.5829489622660247, -0.017702412998801487], [-0.3634614683231049, 0.3052639890805761], [0.34909499575227954, -0.3963507809352678], [0.7317627457812104, -0.03158785089749294], [-0.2610534585812637, -0.15123727360938855], [0.880983919419488, -0.023138656394000146], [1.0272791287423457, 0.007456429468342685], [1.087571950919969, 0.02694610854164925], [1.1918447780834005, 1.0214948391302239], [1.1673803805190675, 0.059513963028228756], [1.29815804429531, 0.13187106500841472], [0.31418259180310515, 0.3101750439117493], [1.4625564640753845, 0.2666310665360103], [0.7691279942252482, -0.4538944064358328], [1.2955054263414088, -0.7193171830146451], [0.8214104305891292, 0.1611565161965498], [1.5203307120606797, 0.33060126937035117], [0.57392028147155, 0.6535676714497287], [1.7456750779596204, 0.8753419123607691], [1.8499479051230527, 1.8698906429493436], [1.7484008213886795, 1.0247772424286683], [1.2743058256364002, 1.9052509416398635], [1.72882394027561, 1.172949754304615], [1.5353158337506452, 2.154048430641059], [0.5396407557910246, 2.246952354832144], [1.6634408795729878, 1.3752172845981852], [0.6794654270807827, 1.5535212635015196], [1.0170373780007522, 0.6122215354913153], [0.142037378000752, 0.1280986172153884], [1.3513077024632736, 0.4251789276107444], [0.3673322499710685, 0.6034829065140791], [1.5430721960993203, 1.577373484729312], [1.2754650930048137, 1.8190609962860016], [-0.5459453375843163, 1.6579044800894522], [1.552744221747159, 0.85827158920249], [1.20024277709337, 1.861152013972108], [0.7261477813410906, 2.7416257131833035], [1.0621331771097142, 1.9182841935392947], [0.9170510377339758, 1.9541940861025098], [-0.06692441475822897, 2.1324980650058447], [-0.13854337719904242, 1.8431451444285334], [-1.0135433771990425, 1.3590222261526064], [0.7682372542187894, 1.9680795240012015], [-0.23132283453830393, 1.9977380634782493], [-0.12705000737487238, 2.992286794066824], [-0.027387420991723532, 1.196634580390839], [0.9565880315004809, 1.0183306014875038], [0.16351583540116033, 0.40920295331004575], [-0.229272827163433, -0.5104258123126475], [0.5564918934750356, 1.949344512888298], [-0.4391831844845855, 2.042248437079383], [-0.8319718470491786, 1.12261967145669], [-0.407402553066754, 1.6830598983761242], [-0.6846816818090999, 2.6438493054596357], [-0.021802803965517703, 1.1335165635557576], [-0.8785655497467283, 1.649227332729178], [0.10624911638166523, 1.0564383354680449], [0.16370323091044225, 1.0297157472656053], [-0.6454648214991129, 1.6172930298078492], [0.2679760580738746, 2.02426447785418], [0.3051539110086301, 0.9814451283602802], [-0.5698460889913701, 0.4973222100843533], [0.47272087125765383, 1.9290352436353655], [0.09005312122872255, 1.564272313597591], [-0.4736895593314757, 2.252001645714743], [-0.49117357528413563, 1.6627506291231915], [0.10885396651374557, 0.8627712862123935], [0.22138288879124857, 0.9611358591073472], [0.33261961948093277, 1.8769777100754799], [-0.6513558330112722, 2.0552816889788144], [0.3453237830437852, 0.9324405022806718], [0.47630770246327314, -0.058943990665182744], [0.4175430798551898, 2.420961710813284], [-0.47699731981644455, 1.1905872615059825], [0.4364435597565426, 1.597558709552313], [0.2429354532315784, 2.5786573858887576], [0.2018419557046902, 1.804620608095294], [0.057635673998863934, 1.6187166072528705], [-0.7977181330524031, 1.8342791475723421], [-0.7794808788336136, 2.8341128350216893], [0.14457523668519512, 2.451855990253293], [-0.6407405970654505, 1.2660533961640876], [0.33808334321015954, 1.4707573139168484], [-0.5369166567898408, 0.9866343956409214], [-0.8631011937550258, 2.0365466778659114], [-1.8559173981175, 1.9168972551540162], [-0.9095069675283703, 1.5939308530746388], [-1.7662697133095808, 2.1096416222480587], [0.6612281728619929, 0.5536875760521727], [-0.020330712060679712, 1.6058904037333572], [-0.9842251586024691, 1.339605789221183], [-0.05916805240955503, 1.5558231190940979], [-0.040930798190765616, 2.5556568065434453], [-0.2922687806571711, 1.5877574220154267], [-0.19172268759690336, 1.8343966478947915], [0.8396476848079191, 1.1609902036458974], [1.0066987225418949, 2.1469384531965527], [0.2860259167833825, 2.403585371347968], [-0.17405611551880862, 1.3505026813202505], [0.7393847640541784, 1.7574741293665814], [1.0769567149741475, 0.8161744013563763], [-1.1803858830813343, 1.4695162175345666], [-0.22070759859248246, 1.2085097706635224], [0.6927332809805049, 1.615481218709853], [-0.23397545249220464, 1.1465498154551894], [0.4327153106736916, 0.4012153762205093], [-0.5668447780834016, 0.4308739156975573], [-0.24567507795962107, 1.061149760742939], [-0.7648658728208624, 0.6355608019061545], [0.5969074748105201, 1.5997169726741451], [-0.24956008875709323, 0.9979043760289021], [-0.24840082138868003, 0.9117144306750404], [0.6650400581843074, 1.3186858787213713], [-0.2428162043624742, 0.8485964138399587], [0.7278913942300083, 0.6083324797282905], [-0.6119405771923634, -0.8751887423504227], [-0.22882394027560993, 0.7635419187990933], [-0.7029189360278894, 1.6440156180102883], [-0.21389444654178935, 0.70196122203968], [-1.2067106509042635, 0.5823117993277844], [-0.6284159534637095, 1.3981397486603249], [-0.21103557294464204, 0.4894078751366992], [-0.10962161937835779, 1.6965099526282543], [0.2279503315416116, 0.7552102246180495], [-0.7560251209505935, 0.9335142035213844], [-0.18738174979465438, 0.6199421541827015], [-0.9265460341365874, 2.8607621810822454], [-0.2531250458223415, 1.2629540643188015], [-0.4574204387033747, 1.0424147496300362], [-0.04307219609932056, 0.35911818837439624], [0.4823928969054929, 1.2099333481085437], [-0.0029023240641649073, 0.3101135622947875], [0.015334930154624615, 1.309947249744135], [-0.631068571417611, 0.546951500637265], [-1.421108629601918, -0.28761145980817837], [0.056571530149863714, 0.24772036358001093], [0.011449919357151561, 1.2467018650300983], [0.7817806314178308, 0.6090572978485951], [-1.609359268040902, 1.351008415295675], [-0.562461386469483, 1.837443385215062], [-1.255889856319619, 1.1169179122432187], [-0.08647548418754991, -0.02437358261627498], [-0.8428139321613648, 0.5282164895243621], [-0.8346975904782373, 0.9731843413887911], [-0.8959635903293285, 0.23490862692203252], [-0.8892197059347098, 0.08560066473308897], [-1.8887797946918026, 0.11525920421013724], [-1.3633147016869895, 0.9660743639442844], [-0.5770865288249956, 1.0356390217205296], [-0.6493058256364004, -0.4528821868120822], [-0.10114778134109059, -1.289256958355522], [-0.28919216413682847, -0.7143786781777088], [0.1717053025594466, 0.15241788721931337], [-1.662409291592536, 0.9357985788175719], [-0.4746981990127883, -0.6105778618875563], [-0.6682063055377527, 0.37052081444888807], [0.29975722290662965, 0.07533965913160051], [-0.6466532076825, 0.398306061210978], [-0.060169043083660445, 0.9573489444527866], [-0.45195671497414813, 0.6361943534714046], [-0.5891990931537225, 0.3715834730085387], [-0.8729505853066085, -1.1239862313516182], [-0.5608174081968944, -0.17394787436417766], [-0.6066831091063826, -0.2176675435830131], [0.3306986406882719, 0.13063613878614], [0.4378668228902858, 0.018207479564413487], [-0.03622817286199376, 0.8986811787756086], [-0.5616932658668071, 0.047866019041461566], [0.4171306744088026, 0.2525699367942223], [0.02456702178131498, 0.8808201512350309], [0.057429418355617656, 2.68245820217891], [-0.2716686945270851, 0.637991019205339], [-0.3105060348759606, 0.5879237345660794], [-0.6353798569899621, 1.476846628280768], [-0.47204558105888805, 0.24061038613550406], [-0.4997388038550856, -0.056184953852978003], [0.41370207571790174, 0.35078649419335284], [-0.3954659766916533, 0.9383637767355963], [0.38944085574106024, 0.9633962633376428], [1.1019973198164448, 0.261781493321799], [0.538254639256246, 0.9495108254389512], [-0.04004005818430734, 0.13368287610641022], [-0.25221270671099427, 0.1467161280058418], [-0.0851616689770186, 1.1326643775564973], [-1.2249479051230527, -0.41752188812156277], [-0.23213170076057899, -0.29787246540966744], [-0.4256398072855435, 0.6832262109267769], [1.257227838786024, 0.8192273088366542], [1.5345069675283698, -0.1415620982468575], [0.07181586700993292, 0.5644386261482432], [-0.9277442217471605, 0.594097165625291], [-0.975338915118816, 0.5222306522564679], [-0.08291052712230162, -0.28942327090617476], [0.021362300041130267, 0.7051254596823999], [1.768688082982341, -0.48355294591288134], [0.2181110763327907, 0.5950337120105863], [0.743576169337604, 1.4458488717447335], [0.6959814759659488, 1.3739823583759105], [1.1315519559057776, 1.0020051600569175], [1.6570170489105913, 1.8528203197910647], [0.6574569601534979, 1.8824788592681125], [1.1058092051387587, 1.0267797959909966], [1.417942382248473, 1.9768181529784372], [0.5092772534794157, -0.7888818407908914], [0.31576914695445124, 0.1922168355455527], [0.6947832883553762, -0.8926826570810439], [1.3997051280296833, 0.9769844655290901], [0.6468028039655183, 0.31885219127202313], [1.2716532076824993, 1.0540626936168032], [0.35174761370618013, 0.4548374670877919], [0.5452557202311448, -0.5262612092486523], [0.9954984973245151, 0.3666449681716013], [0.5214035015722356, 1.2471186673827963], [1.3500275417978815, 0.1682664936410564], [1.3682647960166707, 1.1681001810904037], [0.42185436542754084, 1.4910665831697814], [0.40361711120875166, 0.4912328957204339], [0.35435246383826025, 0.2611704178321412], [0.37258971805704966, 1.2610041052814884], [1.0986895593314756, -0.799632890886962], [1.6241546523362889, 0.05118226884718524], [0.6313384479738151, -0.06846715386471014], [1.5170785795318569, 1.1542147431917118], [0.5331031270396522, 1.3325187220950467], [1.4166907631658963, 0.22291139731717446], [0.7702872615936611, -0.5400843517896953], [1.4807937182941742, 1.2664647539853577], [0.7873652484440377, 0.5459392810135145], [0.9884614683231048, 1.1471047657472053], [0.05107971852844995, 0.7988010833780526], [0.9486231373001242, 0.49856330646663494], [0.5161460334862547, 0.589597468615388], [-0.4678294190059502, 0.7679014475187227], [-1.4317238655477396, 0.5016168330065485], [0.4641556426867042, 0.2100996606591965], [1.377596522259691, 0.617071108705527], [0.5251747142807299, -0.08167261583314223], [-0.43871973226105965, -0.34795723034531645], [-1.2202242141043151, 0.8209747582539961], [0.9149059487458746, 1.8142114230717898], [0.5275145076982053, 0.21095184665845568], [0.6002091012679673, 0.8454598525445117], [-0.09321936858216884, 0.12493437957266845], [1.674056115518809, 0.5859889917834579], [1.1999611197665292, 1.466462690994653], [0.23606667322473962, 1.2001780764824785], [-0.1466010768041912, 0.8354151464447039], [0.7130205425741665, 0.10715103036830398], [0.6744960267617155, 0.6156475312605059], [-0.2893984197800741, 0.34936291674833153], [0.7101616689770194, 0.3197043772712836], [0.6919244147582297, -0.680129310178064], [0.6523874209917238, 0.2557341744369427], [1.7207075985924827, 0.7279819024401859], [0.9365070255500658, 1.4629191949030127], [0.5191266450309988, 2.3716510684266385], [0.6851690430836603, 0.4950198103749942], [2.2738659143934936, 1.8755924021628159], [1.520963590329329, 1.217460127905749], [1.3461561902466785, 0.8081868242668402], [0.011241387505469325, 0.15025962409748178], [1.082420438703375, 0.4099540051977453], [1.6873817497946542, 1.3165495189210072], [1.4306465458113249, 1.0322809451836297], [0.7996775862318015, 1.7769638084398882], [1.2499203633251716, 2.6698699858601422], [0.37492036332517165, 2.1857470675842148], [0.7034062973024497, 1.4948534978243422], [-0.2430041332866799, 1.8178198999037196], [0.9344794257304896, 0.6584172446639405], [0.952716679949279, 1.658250932113288], [1.1893458838207076, 2.25569098380938], [1.1152828352776778, 2.2115920561416247], [0.6706246752105134, 1.2555678618862895], [-0.2534314403082958, 1.637824706654686], [1.6452036253541977, 0.3753835971488375], [1.0970455810588882, 1.2117583686922775], [2.080821260092055, 0.4664854110745593], [1.3279189360278898, -0.19164686318250757], [0.5590967436071159, 1.7556774636326469], [-0.25632976756252623, 1.08725937276617], [1.443428469850136, 1.6887713095236978], [0.35743634737251173, 1.596496050992663], [1.0506398072855427, 0.7691425439010047], [2.0434560116480176, 0.8887919666128999], [1.072748413055535, 1.1290559007245682], [0.12633798246640526, 1.4520223028039454], [1.3282946974405538, 1.7840737858843945], [0.33547849307807986, 1.664424363172499], [0.935506034875961, 0.8644450202617014], [0.07874328909475037, 1.380155789435122], [1.0819569864798488, 2.8001596726224456], [1.06371973226106, 1.8003259851730982], [0.3290546624156838, 2.1420273983653795], [0.641187839525398, 3.0920657553528197], [0.2914896405126086, 1.9973649751893363], [1.2898904619012885, 2.0538963810661506], [0.29664115272920366, 1.6143570785332406], [0.8966686945270848, 0.8143777356224429], [0.5820366231546772, 1.0985355233141583], [0.5637993689358878, 0.09870183586481085], [1.0241271105384078, 0.851161611757983], [0.23634833055158078, 1.5948673994599336], [-0.5475318927356614, 1.775862688455648], [0.0693169727472398, 1.7986347708273995], [1.1072547879024266, 0.9193026920892076], [-0.9236871605394392, 2.6482088341792647], [0.5242623751693827, 1.0345653204798166], [-0.4597130773228222, 1.2128692993831516], [-0.12214112640285302, 0.27156957137294646], [0.3801778314111519, 2.8432682663516236], [-0.06978328835537706, 2.3450514119088255], [-0.14528726159366145, 1.9924531066174764], [0.2732523862938192, 0.9975312877399896], [0.06120063106411133, 1.3536669189629709], [0.6029348446970264, 0.9948951826124105], [0.6110511863801538, 1.4398630344768393]]"
open('/tmp/g3.edge','w').write(EDGEG3)
open('/tmp/g3.vtx.json','w').write(VTXG3)
import sys; sys.path.insert(0,'/tmp')
print('archivos ok')


## G3 k5 (esperado 0)


In [ ]:
import sys
sys.argv=["nn_siren.py","--vtx","/tmp/g3.vtx.json","--edges","/tmp/g3.edge","--k","5","--restarts","16","--steps","8000","--kicks","100","--kick-size","15","--tabu-iters","100000","--slim-rounds","10","--slim-radius","2","--seed","7"]
import nn_siren; _ = nn_siren.main()


## G3 k4 (pregunta abierta)


In [ ]:
import sys
sys.argv=["nn_siren.py","--vtx","/tmp/g3.vtx.json","--edges","/tmp/g3.edge","--k","4","--restarts","16","--steps","8000","--kicks","100","--kick-size","15","--tabu-iters","100000","--slim-rounds","10","--slim-radius","2","--seed","7"]
import nn_siren; _ = nn_siren.main()
